In [ ]:
# ============================================================
# ZİRAAT KATILIM
# BİREYSEL FİNANSMAN ÜRÜN KEŞFİ V1
# Google Colab
# ============================================================


# ============================================================
# KURULUM
# ============================================================

!pip -q install -U selenium beautifulsoup4

!wget -q -O /tmp/google-chrome.deb \
https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb

!apt-get -qq update
!apt-get -qq install -y /tmp/google-chrome.deb


# ============================================================
# IMPORT
# ============================================================

import json
import re
import time
import shutil

from collections import deque
from datetime import datetime
from zoneinfo import ZoneInfo
from urllib.parse import (
    urljoin,
    urlparse,
    urldefrag,
)

from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.options import Options

from google.colab import files


# ============================================================
# AYARLAR
# ============================================================

BASE_URL = (
    "https://www.ziraatkatilim.com.tr"
)

SEED_URLS = [
    BASE_URL + "/",
    BASE_URL + "/bireysel",
]

OUTPUT_PATH = (
    "/content/"
    "ziraat_katilim_finansman_discovery.json"
)

ISTANBUL = ZoneInfo(
    "Europe/Istanbul"
)

MAX_PAGES = 100
MAX_DEPTH = 2

PAGE_WAIT = 2.0


# ============================================================
# NORMALIZATION
# ============================================================

def clean_text(value):

    value = str(
        value or ""
    )

    value = (
        value
        .replace("\xa0", " ")
        .replace("’", "'")
        .replace("‘", "'")
        .replace("–", "-")
        .replace("—", "-")
    )

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value.strip()


def normalize(value):

    value = clean_text(
        value
    )

    value = (
        value
        .replace("İ", "i")
        .replace("I", "ı")
        .casefold()
    )

    return value


# ============================================================
# URL
# ============================================================

def normalize_url(url):

    if not url:
        return ""

    url = urljoin(
        BASE_URL,
        url
    )

    url, _ = urldefrag(
        url
    )

    parsed = urlparse(
        url
    )

    if (
        parsed.scheme
        not in (
            "http",
            "https",
        )
    ):
        return ""

    if (
        parsed.netloc
        != "www.ziraatkatilim.com.tr"
    ):
        return ""

    # Query parametrelerini keşif için at.
    result = (
        f"{parsed.scheme}://"
        f"{parsed.netloc}"
        f"{parsed.path}"
    )

    return result.rstrip("/")


def is_html_candidate(url):

    normalized = normalize(
        url
    )

    bad_extensions = (
        ".pdf",
        ".jpg",
        ".jpeg",
        ".png",
        ".webp",
        ".svg",
        ".zip",
        ".doc",
        ".docx",
        ".xls",
        ".xlsx",
    )

    if normalized.endswith(
        bad_extensions
    ):
        return False

    bad_paths = (
        "/sites/default/files/",
        "/internet-subesi",
        "/subeler-ve-atmler",
        "/iletisim",
        "/gizlilik",
        "/kvkk",
        "/cerez",
        "/kariyer",
    )

    if any(
        bad in normalized
        for bad in bad_paths
    ):
        return False

    return True


# ============================================================
# FİNANSMAN HEURISTIC
# ============================================================

FINANCE_TERMS = (
    "finansman",
    "konut",
    "taşıt",
    "tasit",
    "ihtiyaç",
    "ihtiyac",
    "kolay fon",
    "kff",
    "togg",
    "yeşil ev",
    "yesil ev",
    "yeşil taşıt",
    "yesil tasit",
    "enerji verimlili",
    "ısı yalıtım",
    "isi yalitim",
)


NEGATIVE_TERMS = (
    "finansman sözleş",
    "finansman sozles",
    "sözleşme",
    "sozlesme",
    "bilgi formu",
    "hesaplama aracı",
    "hesaplama araci",
    "ürün ve hizmet ücret",
    "urun ve hizmet ucret",
    "kampanya",
    "basın",
    "basin",
    "haber",
)


def finance_score(
    url,
    anchor_text="",
    title=""
):

    combined = normalize(
        " ".join(
            [
                url,
                anchor_text,
                title,
            ]
        )
    )

    score = 0

    for term in FINANCE_TERMS:

        if term in combined:
            score += 2

    if (
        "/bireysel/"
        in normalize(url)
    ):
        score += 2

    if (
        "finansman"
        in normalize(url)
    ):
        score += 4

    for term in NEGATIVE_TERMS:

        if term in combined:
            score -= 5

    return score


# ============================================================
# CHROME
# ============================================================

def create_driver():

    chrome_binary = (
        shutil.which(
            "google-chrome"
        )
        or
        shutil.which(
            "google-chrome-stable"
        )
        or
        "/usr/bin/google-chrome"
    )

    print(
        "Chrome binary:",
        chrome_binary
    )

    options = Options()

    options.binary_location = (
        chrome_binary
    )

    options.add_argument(
        "--headless"
    )

    options.add_argument(
        "--no-sandbox"
    )

    options.add_argument(
        "--disable-dev-shm-usage"
    )

    options.add_argument(
        "--disable-gpu"
    )

    options.add_argument(
        "--window-size=1920,1080"
    )

    options.add_argument(
        "--disable-notifications"
    )

    options.add_argument(
        "--disable-extensions"
    )

    options.add_argument(
        "--lang=tr-TR"
    )

    options.add_argument(
        "--remote-debugging-port=9222"
    )

    options.add_argument(
        (
            "--user-agent="
            "Mozilla/5.0 "
            "(Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 "
            "(KHTML, like Gecko) "
            "Chrome/151.0.0.0 "
            "Safari/537.36"
        )
    )

    driver = webdriver.Chrome(
        options=options
    )

    driver.set_page_load_timeout(
        40
    )

    return driver


# ============================================================
# PAGE READ
# ============================================================

def read_page(
    driver,
    url
):

    try:

        driver.get(
            url
        )

        time.sleep(
            PAGE_WAIT
        )

        # Lazy içerik için biraz scroll.
        driver.execute_script(
            (
                "window.scrollTo("
                "0, document.body.scrollHeight);"
            )
        )

        time.sleep(
            0.6
        )

        driver.execute_script(
            "window.scrollTo(0, 0);"
        )

        html = driver.page_source

        soup = BeautifulSoup(
            html,
            "html.parser"
        )

        body_text = clean_text(
            soup.get_text(
                " ",
                strip=True
            )
        )

        title = ""

        if soup.title:

            title = clean_text(
                soup.title.get_text(
                    " ",
                    strip=True
                )
            )

        h1 = soup.find(
            "h1"
        )

        h1_text = (
            clean_text(
                h1.get_text(
                    " ",
                    strip=True
                )
            )
            if h1
            else ""
        )

        # WAF / support page kontrolü
        normalized_body = normalize(
            body_text
        )

        waf_blocked = (
            (
                "support id"
                in normalized_body
            )
            and
            (
                "enable javascript"
                in normalized_body
            )
        )

        anchors = []

        for a in soup.find_all(
            "a",
            href=True
        ):

            href = normalize_url(
                a.get(
                    "href"
                )
            )

            if not href:
                continue

            text = clean_text(
                a.get_text(
                    " ",
                    strip=True
                )
            )

            anchors.append(
                {
                    "url": href,
                    "text": text,
                }
            )

        return {
            "ok": True,
            "url": normalize_url(
                driver.current_url
            ),
            "title": title,
            "h1": h1_text,
            "body_text": body_text,
            "anchors": anchors,
            "waf_blocked": waf_blocked,
        }

    except Exception as error:

        return {
            "ok": False,
            "url": url,
            "error": (
                f"{type(error).__name__}: "
                f"{error}"
            ),
            "title": "",
            "h1": "",
            "body_text": "",
            "anchors": [],
            "waf_blocked": False,
        }


# ============================================================
# DISCOVERY
# ============================================================

print(
    "=" * 115
)

print(
    "ZİRAAT KATILIM - "
    "FİNANSMAN DISCOVERY V1"
)

print(
    "=" * 115
)

driver = create_driver()


visited = set()

queue = deque(
    (
        normalize_url(url),
        0,
    )
    for url in SEED_URLS
)


candidate_map = {}

page_logs = []


try:

    while (
        queue
        and
        len(visited)
        < MAX_PAGES
    ):

        url, depth = (
            queue.popleft()
        )

        if not url:
            continue

        if url in visited:
            continue

        visited.add(
            url
        )

        print()
        print(
            f"[{len(visited):03d}] "
            f"Depth={depth}"
        )

        print(
            url
        )

        page = read_page(
            driver,
            url
        )

        page_logs.append(
            {
                "url": url,
                "ok": page.get(
                    "ok"
                ),
                "title": page.get(
                    "title",
                    ""
                ),
                "h1": page.get(
                    "h1",
                    ""
                ),
                "waf_blocked": page.get(
                    "waf_blocked",
                    False
                ),
                "error": page.get(
                    "error",
                    ""
                ),
            }
        )

        if not page.get(
            "ok"
        ):

            print(
                "  ERROR:",
                page.get(
                    "error"
                )
            )

            continue

        if page.get(
            "waf_blocked"
        ):

            print(
                "  ⚠️ WAF/SUPPORT PAGE"
            )

            continue

        print(
            "  H1:",
            page.get(
                "h1"
            )
            or "-"
        )

        print(
            "  Link:",
            len(
                page.get(
                    "anchors",
                    []
                )
            )
        )


        # ====================================================
        # PAGE ITSELF CANDIDATE?
        # ====================================================

        page_score = finance_score(
            url,
            "",
            (
                page.get(
                    "h1"
                )
                or
                page.get(
                    "title"
                )
            )
        )

        if (
            page_score >= 6
            and
            "/bireysel"
            in normalize(url)
        ):

            candidate_map[
                url
            ] = {
                "url": url,
                "anchor_text": "",
                "title": (
                    page.get(
                        "h1"
                    )
                    or
                    page.get(
                        "title"
                    )
                ),
                "score": page_score,
                "source_page": url,
            }


        # ====================================================
        # ANCHORS
        # ====================================================

        for anchor in page.get(
            "anchors",
            []
        ):

            href = anchor[
                "url"
            ]

            text = anchor[
                "text"
            ]

            if not is_html_candidate(
                href
            ):
                continue

            parsed = urlparse(
                href
            )

            path_n = normalize(
                parsed.path
            )

            # -----------------------------------------------
            # Candidate
            # -----------------------------------------------

            score = finance_score(
                href,
                text,
                ""
            )

            if (
                score >= 6
                and
                "/bireysel"
                in path_n
            ):

                old = candidate_map.get(
                    href
                )

                candidate = {
                    "url": href,
                    "anchor_text": text,
                    "title": "",
                    "score": score,
                    "source_page": url,
                }

                if (
                    old is None
                    or
                    candidate["score"]
                    > old["score"]
                ):

                    candidate_map[
                        href
                    ] = candidate


            # -----------------------------------------------
            # Crawl yalnızca bireysel sayfalarda
            # -----------------------------------------------

            if depth >= MAX_DEPTH:
                continue

            if (
                "/bireysel"
                not in path_n
            ):
                continue

            # Gereksiz bireysel alanları azalt.
            excluded = (
                "/hesaplar/",
                "/kartlar/",
                "/sigorta",
                "/emeklilik",
                "/odeme",
                "/dijital",
                "/para-transfer",
                "/yatirim",
            )

            if any(
                x in path_n
                for x in excluded
            ):
                continue

            if href not in visited:

                queue.append(
                    (
                        href,
                        depth + 1,
                    )
                )


finally:

    driver.quit()


# ============================================================
# CANDIDATE CLEAN
# ============================================================

candidates = sorted(
    candidate_map.values(),
    key=lambda item: (
        -item[
            "score"
        ],
        item[
            "url"
        ],
    )
)


# ============================================================
# OUTPUT
# ============================================================

output = {
    "banka":
        "Ziraat Katılım Bankası A.Ş.",

    "discovery_zamani":
        datetime.now(
            ISTANBUL
        ).isoformat(
            timespec="seconds"
        ),

    "ziyaret_edilen_sayfa_sayisi":
        len(
            visited
        ),

    "aday_finansman_linki_sayisi":
        len(
            candidates
        ),

    "aday_finansman_linkleri":
        candidates,

    "page_logs":
        page_logs,
}


with open(
    OUTPUT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# PRINT
# ============================================================

print()
print(
    "=" * 115
)

print(
    "DISCOVERY SONUCU"
)

print(
    "=" * 115
)

print(
    "Ziyaret edilen sayfa:",
    len(
        visited
    )
)

print(
    "Finansman aday linki:",
    len(
        candidates
    )
)

print(
    "Dosya:",
    OUTPUT_PATH
)

print()


for index, item in enumerate(
    candidates,
    start=1
):

    print(
        f"[{index:02d}] "
        f"score={item['score']}"
    )

    print(
        "     Text :",
        item[
            "anchor_text"
        ]
        or "-"
    )

    print(
        "     Title:",
        item[
            "title"
        ]
        or "-"
    )

    print(
        "     URL  :",
        item[
            "url"
        ]
    )


print()
print(
    "=" * 115
)


# ============================================================
# DOWNLOAD
# ============================================================

files.download(
    OUTPUT_PATH
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.8/511.8 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 5.5 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libatk1.0-data.
(Reading database ... 118422 files and directories currently installed.)
Preparing to unpack .../00-libatk1.0-data_2.36.0-3build1_all.deb ...
Unpacking libatk1.0-data (2.36.0-3build1) ...
Selecting previously unselected package libatk1.0-0:amd64.
Preparing to unpack .../01-libatk1.0-0_2.36.0-3build1_amd64.deb ...
Unpacking libatk1.0-0:amd64 (2.36.0-3build1) ...
Selecting previously unselected package libatspi2.0-0:amd64.
Preparing to

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# ZİRAAT KATILIM
# FİNANSMAN DISCOVERY V2
#
# WAF nedeniyle ziraatkatilim.com.tr doğrudan crawl edilmiyor.
# Arama motoru indeksinden gerçek ürün URL'leri keşfediliyor.
# ============================================================


!pip -q install -U requests beautifulsoup4


import json
import re
import time
import urllib.parse
import xml.etree.ElementTree as ET

import requests

from datetime import datetime
from zoneinfo import ZoneInfo
from urllib.parse import urlparse

from google.colab import files


# ============================================================
# AYARLAR
# ============================================================

DOMAIN = "www.ziraatkatilim.com.tr"

OUTPUT_FILE = (
    "/content/"
    "ziraat_katilim_finansman_discovery_v2.json"
)

ISTANBUL = ZoneInfo(
    "Europe/Istanbul"
)


# ============================================================
# ARAMA SORGULARI
#
# Bunlar "ürün budur" varsayımı değil.
# Search-engine discovery için aday sorgular.
# ============================================================

SEARCH_QUERIES = [

    # Genel
    (
        'site:ziraatkatilim.com.tr/bireysel '
        '"finansman" "Ziraat Katılım"'
    ),

    # Temel bireysel finansman
    (
        'site:ziraatkatilim.com.tr/bireysel '
        '"Konut Finansmanı"'
    ),

    (
        'site:ziraatkatilim.com.tr/bireysel '
        '"Taşıt Finansmanı"'
    ),

    (
        'site:ziraatkatilim.com.tr/bireysel '
        '"Bireysel Finansman"'
    ),

    # Dijital / yeni ürünler
    (
        'site:ziraatkatilim.com.tr '
        '"Anında Finansman"'
    ),

    (
        'site:ziraatkatilim.com.tr '
        '"Kolay Fon Finansmanı"'
    ),

    (
        'site:ziraatkatilim.com.tr '
        '"Dijital Taşıt Finansmanı"'
    ),

    (
        'site:ziraatkatilim.com.tr '
        '"Togg Taşıt Finansmanı"'
    ),

    # Konut varyantları
    (
        'site:ziraatkatilim.com.tr '
        '"İlk Evim Konut Finansmanı"'
    ),

    (
        'site:ziraatkatilim.com.tr '
        '"Genişletilmiş Konut Finansmanı"'
    ),

    # Çevresel / sürdürülebilir bireysel ürünler
    (
        'site:ziraatkatilim.com.tr '
        '"Yeşil Ev Konut Finansmanı"'
    ),

    (
        'site:ziraatkatilim.com.tr '
        '"Yeşil Taşıt Finansmanı"'
    ),

    (
        'site:ziraatkatilim.com.tr '
        '"Bireysel Enerji Verimliliği Finansmanı"'
    ),

    (
        'site:ziraatkatilim.com.tr '
        '"Konutlarda Isı Yalıtım Finansmanı"'
    ),
]


# ============================================================
# SESSION
# ============================================================

session = requests.Session()

session.headers.update(
    {
        "User-Agent": (
            "Mozilla/5.0 "
            "(Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 "
            "(KHTML, like Gecko) "
            "Chrome/151.0.0.0 "
            "Safari/537.36"
        ),
        "Accept-Language": (
            "tr-TR,tr;q=0.9,en-US;q=0.8,en;q=0.7"
        ),
    }
)


# ============================================================
# NORMALIZATION
# ============================================================

def clean_text(value):

    value = str(
        value or ""
    )

    value = (
        value
        .replace("\xa0", " ")
        .replace("’", "'")
        .replace("‘", "'")
        .replace("–", "-")
        .replace("—", "-")
    )

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value.strip()


def normalize(value):

    value = clean_text(
        value
    )

    value = (
        value
        .replace("İ", "i")
        .replace("I", "ı")
        .casefold()
    )

    return value


# ============================================================
# URL CLEANING
# ============================================================

def normalize_ziraat_url(url):

    url = clean_text(
        url
    )

    if not url:
        return ""

    try:

        parsed = urlparse(
            url
        )

    except Exception:

        return ""

    host = (
        parsed.netloc
        .lower()
        .replace(":443", "")
    )

    if host not in {
        "ziraatkatilim.com.tr",
        "www.ziraatkatilim.com.tr",
    }:

        return ""

    path = parsed.path

    if not path:
        return ""

    # WWW standardı
    final = (
        "https://www.ziraatkatilim.com.tr"
        + path
    )

    return final.rstrip("/")


# ============================================================
# İSTENMEYEN DOSYA / SAYFA
# ============================================================

BAD_EXTENSIONS = (
    ".pdf",
    ".jpg",
    ".jpeg",
    ".png",
    ".webp",
    ".svg",
    ".zip",
    ".doc",
    ".docx",
    ".xls",
    ".xlsx",
)


BAD_PATH_MARKERS = (
    "/sites/default/files/",
    "/basin-odasi/",
    "/basin-bultenleri/",
    "/rapor",
    "/yatirimci",
    "/sozlesme",
    "/form",
    "/kvkk",
    "/cerez",
    "/urun-ve-hizmet-ucret",
)


def valid_candidate_url(url):

    normalized = normalize(
        url
    )

    if normalized.endswith(
        BAD_EXTENSIONS
    ):
        return False

    if any(
        marker in normalized
        for marker in BAD_PATH_MARKERS
    ):
        return False

    # Öncelikle bireysel ürün sayfaları.
    if "/bireysel/" not in normalized:

        return False

    return True


# ============================================================
# FİNANSMAN SCORE
# ============================================================

FINANCE_TERMS = (
    "finansman",
    "konut",
    "taşıt",
    "tasit",
    "togg",
    "kolay fon",
    "anında finansman",
    "aninda finansman",
    "yeşil ev",
    "yesil ev",
    "yeşil taşıt",
    "yesil tasit",
    "enerji verimlili",
    "ısı yalıtım",
    "isi yalitim",
)


def score_candidate(
    url,
    title,
    description
):

    combined = normalize(
        " ".join(
            [
                url,
                title,
                description,
            ]
        )
    )

    score = 0

    if "/bireysel/" in combined:
        score += 4

    if "finansman" in combined:
        score += 5

    for term in FINANCE_TERMS:

        if term in combined:
            score += 2

    if (
        "/hesaplar/"
        in combined
        or
        "/kartlar/"
        in combined
        or
        "/sigorta/"
        in combined
    ):

        score -= 10

    return score


# ============================================================
# BING RSS SEARCH
# ============================================================

def bing_rss_search(query):

    encoded = urllib.parse.quote_plus(
        query
    )

    url = (
        "https://www.bing.com/"
        "search"
        f"?q={encoded}"
        "&format=rss"
        "&count=50"
    )

    response = session.get(
        url,
        timeout=30
    )

    response.raise_for_status()

    root = ET.fromstring(
        response.text
    )

    results = []

    for item in root.findall(
        ".//item"
    ):

        title = clean_text(
            item.findtext(
                "title"
            )
        )

        link = clean_text(
            item.findtext(
                "link"
            )
        )

        description = clean_text(
            item.findtext(
                "description"
            )
        )

        results.append(
            {
                "title": title,
                "url": link,
                "description": description,
            }
        )

    return results


# ============================================================
# DISCOVERY
# ============================================================

print(
    "=" * 115
)

print(
    "ZİRAAT KATILIM - "
    "FİNANSMAN DISCOVERY V2"
)

print(
    "=" * 115
)

print(
    "Yöntem: Bing RSS search index"
)

print(
    "Doğrudan site crawl: KAPALI (WAF)"
)

print()


candidate_map = {}

query_logs = []


for query_index, query in enumerate(
    SEARCH_QUERIES,
    start=1
):

    print(
        "=" * 115
    )

    print(
        f"QUERY {query_index:02d}/"
        f"{len(SEARCH_QUERIES):02d}"
    )

    print(
        query
    )

    try:

        results = bing_rss_search(
            query
        )

        print(
            "Sonuç:",
            len(results)
        )

        query_logs.append(
            {
                "query": query,
                "result_count": len(results),
                "error": "",
            }
        )

    except Exception as error:

        print(
            "ERROR:",
            type(error).__name__,
            error
        )

        query_logs.append(
            {
                "query": query,
                "result_count": 0,
                "error": (
                    f"{type(error).__name__}: "
                    f"{error}"
                ),
            }
        )

        continue


    for result in results:

        url = normalize_ziraat_url(
            result[
                "url"
            ]
        )

        if not url:
            continue

        if not valid_candidate_url(
            url
        ):
            continue

        score = score_candidate(
            url,
            result[
                "title"
            ],
            result[
                "description"
            ],
        )

        if score < 7:
            continue

        candidate = {
            "url":
                url,

            "title":
                result[
                    "title"
                ],

            "description":
                result[
                    "description"
                ],

            "score":
                score,

            "found_by_query":
                query,
        }

        old = candidate_map.get(
            url
        )

        if (
            old is None
            or
            candidate[
                "score"
            ]
            > old[
                "score"
            ]
        ):

            candidate_map[
                url
            ] = candidate


    time.sleep(
        0.7
    )


# ============================================================
# SORT
# ============================================================

candidates = sorted(
    candidate_map.values(),
    key=lambda item: (
        -item[
            "score"
        ],
        item[
            "url"
        ],
    )
)


# ============================================================
# BASİT URL DEDUPE
# ============================================================

final_candidates = []

seen_paths = set()


for item in candidates:

    parsed = urlparse(
        item[
            "url"
        ]
    )

    path = (
        parsed.path
        .rstrip("/")
        .casefold()
    )

    if path in seen_paths:
        continue

    seen_paths.add(
        path
    )

    final_candidates.append(
        item
    )


# ============================================================
# SAVE
# ============================================================

output = {

    "banka":
        "Ziraat Katılım Bankası A.Ş.",

    "discovery_zamani":
        datetime.now(
            ISTANBUL
        ).isoformat(
            timespec="seconds"
        ),

    "discovery_yontemi":
        (
            "Bing RSS search index - "
            "Ziraat Katılım WAF nedeniyle "
            "doğrudan crawl yapılmadı"
        ),

    "arama_sorgusu_sayisi":
        len(
            SEARCH_QUERIES
        ),

    "aday_finansman_linki_sayisi":
        len(
            final_candidates
        ),

    "aday_finansman_linkleri":
        final_candidates,

    "query_logs":
        query_logs,
}


with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# PRINT
# ============================================================

print()
print(
    "=" * 115
)

print(
    "DISCOVERY V2 SONUCU"
)

print(
    "=" * 115
)

print(
    "Arama sorgusu         :",
    len(
        SEARCH_QUERIES
    )
)

print(
    "Finansman aday linki  :",
    len(
        final_candidates
    )
)

print(
    "Dosya                 :",
    OUTPUT_FILE
)

print()


if not final_candidates:

    print(
        "⚠️ ADAY URL BULUNAMADI"
    )

    print(
        "Bing de Ziraat'ın ürün "
        "sayfalarını indekslememiş olabilir."
    )

else:

    for index, item in enumerate(
        final_candidates,
        start=1
    ):

        print(
            "-" * 115
        )

        print(
            f"[{index:02d}] "
            f"score={item['score']}"
        )

        print(
            "Başlık :",
            item[
                "title"
            ]
        )

        print(
            "URL    :",
            item[
                "url"
            ]
        )

        print(
            "Açıklama:",
            item[
                "description"
            ][:300]
        )


print()
print(
    "=" * 115
)


files.download(
    OUTPUT_FILE
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
ZİRAAT KATILIM - FİNANSMAN DISCOVERY V2
Yöntem: Bing RSS search index
Doğrudan site crawl: KAPALI (WAF)

QUERY 01/14
site:ziraatkatilim.com.tr/bireysel "finansman" "Ziraat Katılım"
Sonuç: 10
QUERY 02/14
site:ziraatkatilim.com.tr/bireysel "Konut Finansmanı"
Sonuç: 10
QUERY 03/14
site:ziraatkatilim.com.tr/bireysel "Taşıt Finansmanı"
Sonuç: 10
QUERY 04/14
site:ziraatkatilim.com.tr/bireysel "Bireysel Finansman"
Sonuç: 10
QUERY 05/14
site:ziraatkatilim.com.tr "Anında Finansman"
Sonuç: 10
QUERY 06/14
site:ziraatkatilim.com.tr "Kolay Fon Finansmanı"
Sonuç: 10
QUERY 07/14
site:ziraatkatilim.com.tr "Dijital Taşıt Finansmanı"
Sonuç: 10
QUERY 08/14
sit

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# ZİRAAT KATILIM
# BİREYSEL FİNANSMAN RAW SCRAPER V1
# Google Colab
# ============================================================


# Colab ile uyumlu requests sürümünü koruyoruz.
!pip -q install requests==2.32.4 beautifulsoup4


import json
import re
import time

import requests

from bs4 import BeautifulSoup
from datetime import datetime
from zoneinfo import ZoneInfo

from google.colab import files


# ============================================================
# AYARLAR
# ============================================================

BANK_NAME = (
    "Ziraat Katılım Bankası A.Ş."
)

OUTPUT_FILE = (
    "/content/"
    "ziraat_katilim_finansmanlar.json"
)

ISTANBUL = ZoneInfo(
    "Europe/Istanbul"
)

EXPECTED_COUNT = 18


# ============================================================
# DOĞRULANMIŞ GÜNCEL ÜRÜN ENVANTERİ
# ============================================================

PRODUCTS = [

    # ========================================================
    # KONUT / GAYRİMENKUL
    # ========================================================

    {
        "urun_adi":
            "Konut Finansmanı",

        "liste_kategorisi":
            "Konut-Gayrimenkul Finansmanı",

        "kaynak_url":
            (
                "https://www.ziraatkatilim.com.tr/"
                "bireysel/finansman-urunleri/"
                "konut-gayrimenkul-finansmani/"
                "konut-finansmani"
            ),
    },

    {
        "urun_adi":
            "Kentsel Dönüşüm Finansmanı",

        "liste_kategorisi":
            "Konut-Gayrimenkul Finansmanı",

        "kaynak_url":
            (
                "https://www.ziraatkatilim.com.tr/"
                "bireysel/finansman-urunleri/"
                "konut-finansmani/"
                "kentsel-donusum-finansmani"
            ),
    },

    {
        "urun_adi":
            "Bireysel Arsa Finansmanı",

        "liste_kategorisi":
            "Konut-Gayrimenkul Finansmanı",

        "kaynak_url":
            (
                "https://www.ziraatkatilim.com.tr/"
                "bireysel/finansman-urunleri/"
                "konut-gayrimenkul-finansmani/"
                "bireysel-arsa-finansmani"
            ),
    },

    {
        "urun_adi":
            "Bireysel İş Yeri Finansmanı",

        "liste_kategorisi":
            "Konut-Gayrimenkul Finansmanı",

        "kaynak_url":
            (
                "https://www.ziraatkatilim.com.tr/"
                "bireysel/finansman-urunleri/"
                "konut-gayrimenkul-finansmani/"
                "bireysel-is-yeri-finansmani"
            ),
    },


    # ========================================================
    # TAŞIT
    # ========================================================

    {
        "urun_adi":
            "Taşıt Finansmanı",

        "liste_kategorisi":
            "Taşıt Finansmanı",

        "kaynak_url":
            (
                "https://www.ziraatkatilim.com.tr/"
                "bireysel/finansman-urunleri/"
                "tasit-finansmani/"
                "tasit-finansmani"
            ),
    },

    {
        "urun_adi":
            "TOGG Finansmanı",

        "liste_kategorisi":
            "Taşıt Finansmanı",

        "kaynak_url":
            (
                "https://www.ziraatkatilim.com.tr/"
                "bireysel/finansman-urunleri/"
                "tasit-finansmani/"
                "togg-finansmani"
            ),
    },


    # ========================================================
    # İHTİYAÇ
    # ========================================================

    {
        "urun_adi":
            "Eğitim Finansmanı",

        "liste_kategorisi":
            "İhtiyaç Finansmanı",

        "kaynak_url":
            (
                "https://www.ziraatkatilim.com.tr/"
                "bireysel/finansman-urunleri/"
                "ihtiyac-finansmani/"
                "egitim-finansmani"
            ),
    },

    {
        "urun_adi":
            "Doğalgaz Dönüşüm Finansmanı",

        "liste_kategorisi":
            "İhtiyaç Finansmanı",

        "kaynak_url":
            (
                "https://www.ziraatkatilim.com.tr/"
                "bireysel/finansman-urunleri/"
                "ihtiyac-finansmani/"
                "dogal-gaz-donusum-finansmani"
            ),
    },

    {
        "urun_adi":
            "Hac ve Umre Finansmanı",

        "liste_kategorisi":
            "İhtiyaç Finansmanı",

        "kaynak_url":
            (
                "https://www.ziraatkatilim.com.tr/"
                "bireysel/finansman-urunleri/"
                "ihtiyac-finansmani/"
                "hac-ve-umre-finansmani"
            ),
    },

    {
        "urun_adi":
            "İpotekli Bireysel Finansman",

        "liste_kategorisi":
            "İhtiyaç Finansmanı",

        "kaynak_url":
            (
                "https://www.ziraatkatilim.com.tr/"
                "bireysel/finansman-urunleri/"
                "ihtiyac-finansmani/"
                "ipotekli-bireysel-finansman"
            ),
    },

    {
        "urun_adi":
            "Yasa Kapsamında İpotekli Bireysel Finansman",

        "liste_kategorisi":
            "İhtiyaç Finansmanı",

        "kaynak_url":
            (
                "https://www.ziraatkatilim.com.tr/"
                "bireysel/finansman-urunleri/"
                "ihtiyac-finansmani/"
                "yasa-kapsaminda-ipotekli-"
                "bireysel-finansman"
            ),
    },

    {
        "urun_adi":
            "Dayanıklı Tüketim Finansmanı",

        "liste_kategorisi":
            "İhtiyaç Finansmanı",

        "kaynak_url":
            (
                "https://www.ziraatkatilim.com.tr/"
                "bireysel/finansman-urunleri/"
                "ihtiyac-finansmani/"
                "dayanikli-tuketim-finansmani"
            ),
    },

    {
        "urun_adi":
            "Kolay Fon Finansmanı",

        "liste_kategorisi":
            "İhtiyaç Finansmanı",

        "kaynak_url":
            (
                "https://www.ziraatkatilim.com.tr/"
                "bireysel/finansman-urunleri/"
                "ihtiyac-finansmani/"
                "kolayfon-finansmani"
            ),
    },

    {
        "urun_adi":
            "Anında Finansman",

        "liste_kategorisi":
            "İhtiyaç Finansmanı",

        "kaynak_url":
            (
                "https://www.ziraatkatilim.com.tr/"
                "bireysel/finansman-urunleri/"
                "ihtiyac-finansmani/"
                "aninda-finansman"
            ),
    },


    # ========================================================
    # SÜRDÜRÜLEBİLİRLİK TEMALI
    # ========================================================

    {
        "urun_adi":
            "Yeşil Ev Konut Finansmanı",

        "liste_kategorisi":
            "Sürdürülebilirlik Temalı Bireysel Ürünler",

        "kaynak_url":
            (
                "https://www.ziraatkatilim.com.tr/"
                "bireysel/finansman-urunleri/"
                "surdurulebilirlik-temali-"
                "bireysel-urunler/"
                "yesil-ev-konut-finansmani"
            ),
    },

    {
        "urun_adi":
            "Yeşil Taşıt Finansmanı",

        "liste_kategorisi":
            "Sürdürülebilirlik Temalı Bireysel Ürünler",

        "kaynak_url":
            (
                "https://www.ziraatkatilim.com.tr/"
                "bireysel/finansman-urunleri/"
                "surdurulebilirlik-temali-"
                "bireysel-urunler/"
                "yesil-tasit-finansmani"
            ),
    },

    {
        "urun_adi":
            "Bireysel Enerji Verimliliği Finansmanı",

        "liste_kategorisi":
            "Sürdürülebilirlik Temalı Bireysel Ürünler",

        "kaynak_url":
            (
                "https://www.ziraatkatilim.com.tr/"
                "bireysel/finansman-urunleri/"
                "surdurulebilirlik-temali-"
                "bireysel-urunler/"
                "bireysel-enerji-verimliligi-"
                "finansmani"
            ),
    },

    {
        "urun_adi":
            "Enerji Verimliliği Yönetim Finansmanı",

        "liste_kategorisi":
            "Sürdürülebilirlik Temalı Bireysel Ürünler",

        "kaynak_url":
            (
                "https://www.ziraatkatilim.com.tr/"
                "bireysel/finansman-urunleri/"
                "surdurulebilirlik-temali-"
                "bireysel-urunler/"
                "enerji-verimliligi-yonetim-"
                "finansmani"
            ),
    },
]


# ============================================================
# SESSION
# ============================================================

session = requests.Session()

session.headers.update(
    {
        "User-Agent": (
            "Mozilla/5.0 "
            "(Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 "
            "(KHTML, like Gecko) "
            "Chrome/151.0.0.0 "
            "Safari/537.36"
        ),

        "Accept-Language":
            "tr-TR,tr;q=0.9,en;q=0.8",
    }
)


# ============================================================
# TEXT CLEAN
# ============================================================

def clean_text(value):

    value = str(
        value or ""
    )

    value = (
        value
        .replace("\xa0", " ")
        .replace("’", "'")
        .replace("‘", "'")
        .replace("–", "-")
        .replace("—", "-")
    )

    value = re.sub(
        r"[ \t]+",
        " ",
        value
    )

    value = re.sub(
        r"\n[ \t]+",
        "\n",
        value
    )

    value = re.sub(
        r"\n{3,}",
        "\n\n",
        value
    )

    return value.strip()


# ============================================================
# WAF DETECTION
# ============================================================

def is_blocked(text):

    normalized = (
        str(text or "")
        .casefold()
    )

    markers = (
        "please enable javascript",
        "your support id is",
        "request rejected",
        "the requested url was rejected",
    )

    return any(
        marker in normalized
        for marker
        in markers
    )


# ============================================================
# HTML -> TEXT
# ============================================================

def html_to_text(html):

    soup = BeautifulSoup(
        html,
        "html.parser"
    )

    for tag in soup(
        [
            "script",
            "style",
            "noscript",
            "svg",
        ]
    ):

        tag.decompose()

    text = soup.get_text(
        "\n",
        strip=True
    )

    return clean_text(
        text
    )


# ============================================================
# JINA / TEXT READER CLEAN
# ============================================================

def reader_to_text(text):

    text = clean_text(
        text
    )

    # Jina başındaki metadata alanlarını kaldır.
    lines = text.splitlines()

    result = []

    for line in lines:

        stripped = line.strip()

        if stripped.startswith(
            "URL Source:"
        ):
            continue

        if stripped.startswith(
            "Published Time:"
        ):
            continue

        if stripped.startswith(
            "Markdown Content"
        ):
            continue

        result.append(
            line
        )

    return clean_text(
        "\n".join(
            result
        )
    )


# ============================================================
# DIRECT REQUEST
# ============================================================

def try_direct(url):

    try:

        response = session.get(
            url,
            timeout=30,
            allow_redirects=True
        )

        text = response.text

        parsed_text = html_to_text(
            text
        )

        if (
            response.status_code == 200
            and
            len(parsed_text) >= 300
            and
            not is_blocked(
                parsed_text
            )
        ):

            return {
                "success": True,
                "method": "direct",
                "status": response.status_code,
                "fetch_url": response.url,
                "text": parsed_text,
            }

        return {
            "success": False,
            "method": "direct",
            "status": response.status_code,
            "fetch_url": response.url,
            "text": parsed_text,
        }

    except Exception as error:

        return {
            "success": False,
            "method": "direct",
            "status": 0,
            "fetch_url": url,
            "text": "",
            "error": (
                f"{type(error).__name__}: "
                f"{error}"
            ),
        }


# ============================================================
# READER FALLBACK
# ============================================================

def try_reader(url):

    reader_urls = [
        (
            "https://r.jina.ai/"
            + url
        ),

        (
            "https://r.jina.ai/http://"
            + url
            .replace(
                "https://",
                ""
            )
            .replace(
                "http://",
                ""
            )
        ),
    ]

    last_result = None

    for reader_url in reader_urls:

        try:

            response = session.get(
                reader_url,
                timeout=60
            )

            text = reader_to_text(
                response.text
            )

            last_result = {
                "success": False,
                "method": "reader",
                "status": response.status_code,
                "fetch_url": reader_url,
                "text": text,
            }

            if (
                response.status_code == 200
                and
                len(text) >= 300
                and
                not is_blocked(
                    text
                )
            ):

                return {
                    "success": True,
                    "method": "reader",
                    "status": response.status_code,
                    "fetch_url": reader_url,
                    "text": text,
                }

        except Exception as error:

            last_result = {
                "success": False,
                "method": "reader",
                "status": 0,
                "fetch_url": reader_url,
                "text": "",
                "error": (
                    f"{type(error).__name__}: "
                    f"{error}"
                ),
            }

    return (
        last_result
        or
        {
            "success": False,
            "method": "reader",
            "status": 0,
            "fetch_url": "",
            "text": "",
        }
    )


# ============================================================
# FETCH WITH FALLBACK
# ============================================================

def fetch_product(url):

    direct = try_direct(
        url
    )

    if direct[
        "success"
    ]:

        return direct

    reader = try_reader(
        url
    )

    if reader[
        "success"
    ]:

        return reader

    # İkisi de olmadı.
    return {
        "success": False,

        "method":
            "failed",

        "status":
            reader.get(
                "status",
                direct.get(
                    "status",
                    0
                )
            ),

        "fetch_url":
            reader.get(
                "fetch_url",
                ""
            ),

        "text":
            "",

        "direct_status":
            direct.get(
                "status",
                0
            ),

        "reader_status":
            reader.get(
                "status",
                0
            ),

        "error":
            (
                reader.get(
                    "error"
                )
                or
                direct.get(
                    "error"
                )
                or
                "İçerik alınamadı."
            ),
    }


# ============================================================
# SCRAPER
# ============================================================

print(
    "=" * 118
)

print(
    "ZİRAAT KATILIM - "
    "BİREYSEL FİNANSMAN RAW SCRAPER V1"
)

print(
    "=" * 118
)

print(
    "Beklenen ürün:",
    EXPECTED_COUNT
)

print()


records = []

errors = []


for index, product in enumerate(
    PRODUCTS,
    start=1
):

    print(
        "=" * 118
    )

    print(
        f"[{index:02d}/{EXPECTED_COUNT:02d}] "
        f"{product['urun_adi']}"
    )

    print(
        product[
            "kaynak_url"
        ]
    )

    fetched = fetch_product(
        product[
            "kaynak_url"
        ]
    )

    if fetched[
        "success"
    ]:

        print(
            "  METHOD :",
            fetched[
                "method"
            ]
        )

        print(
            "  STATUS :",
            fetched[
                "status"
            ]
        )

        print(
            "  LENGTH :",
            len(
                fetched[
                    "text"
                ]
            )
        )

        print(
            "  RESULT : ✅"
        )

    else:

        print(
            "  METHOD : FAILED"
        )

        print(
            "  STATUS :",
            fetched.get(
                "status",
                0
            )
        )

        print(
            "  RESULT : ❌"
        )

        print(
            "  ERROR  :",
            fetched.get(
                "error",
                ""
            )
        )

        errors.append(
            {
                "urun_adi":
                    product[
                        "urun_adi"
                    ],

                "kaynak_url":
                    product[
                        "kaynak_url"
                    ],

                "error":
                    fetched.get(
                        "error",
                        ""
                    ),
            }
        )

    records.append(
        {
            "urun_adi":
                product[
                    "urun_adi"
                ],

            "liste_kategorisi":
                product[
                    "liste_kategorisi"
                ],

            "kaynak_url":
                product[
                    "kaynak_url"
                ],

            "final_url":
                (
                    fetched.get(
                        "fetch_url",
                        ""
                    )
                    if fetched[
                        "method"
                    ]
                    == "direct"
                    else
                    product[
                        "kaynak_url"
                    ]
                ),

            "fetch_method":
                fetched[
                    "method"
                ],

            "http_status":
                fetched[
                    "status"
                ],

            "ham_metin":
                fetched[
                    "text"
                ],
        }
    )

    time.sleep(
        0.5
    )


# ============================================================
# VALIDATION
# ============================================================

successful = [
    record
    for record
    in records
    if (
        record[
            "ham_metin"
        ]
        and
        len(
            record[
                "ham_metin"
            ]
        )
        >= 300
    )
]


empty_records = [
    record
    for record
    in records
    if not record[
        "ham_metin"
    ]
]


urls = [
    record[
        "kaynak_url"
    ]
    for record
    in records
]


duplicate_count = (
    len(urls)
    -
    len(
        set(urls)
    )
)


# ============================================================
# OUTPUT
# ============================================================

output = {

    "banka":
        BANK_NAME,

    "scrape_zamani":
        datetime.now(
            ISTANBUL
        ).isoformat(
            timespec="seconds"
        ),

    "beklenen_urun_sayisi":
        EXPECTED_COUNT,

    "urun_sayisi":
        len(
            records
        ),

    "basarili_urun_sayisi":
        len(
            successful
        ),

    "hata_sayisi":
        len(
            errors
        ),

    "duplicate_url":
        duplicate_count,

    "urunler":
        records,
}


with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# SUMMARY
# ============================================================

print()
print(
    "=" * 118
)

print(
    "SCRAPER SONUCU"
)

print(
    "=" * 118
)

print(
    "Beklenen ürün     :",
    EXPECTED_COUNT
)

print(
    "Toplam kayıt      :",
    len(
        records
    )
)

print(
    "Başarılı içerik   :",
    len(
        successful
    )
)

print(
    "Hata              :",
    len(
        errors
    )
)

print(
    "Boş ham metin     :",
    len(
        empty_records
    )
)

print(
    "Duplicate URL     :",
    duplicate_count
)

print(
    "Dosya             :",
    OUTPUT_FILE
)


# ============================================================
# METHOD DISTRIBUTION
# ============================================================

methods = {}

for record in records:

    method = record[
        "fetch_method"
    ]

    methods[
        method
    ] = (
        methods.get(
            method,
            0
        )
        + 1
    )


print()
print(
    "FETCH METHOD:"
)

for method, count in sorted(
    methods.items()
):

    print(
        f"- {method}: {count}"
    )


# ============================================================
# FAILED
# ============================================================

if errors:

    print()
    print(
        "=" * 118
    )

    print(
        "BAŞARISIZ ÜRÜNLER"
    )

    print(
        "=" * 118
    )

    for error in errors:

        print(
            "-",
            error[
                "urun_adi"
            ]
        )

        print(
            " ",
            error[
                "kaynak_url"
            ]
        )


# ============================================================
# FINAL
# ============================================================

print()
print(
    "=" * 118
)

if (
    len(records) == EXPECTED_COUNT
    and
    len(successful) == EXPECTED_COUNT
    and
    len(errors) == 0
    and
    duplicate_count == 0
):

    print(
        "SONUÇ: ZİRAAT KATILIM "
        "FİNANSMAN RAW SCRAPER "
        "BAŞARILI ✅"
    )

else:

    print(
        "SONUÇ: ZİRAAT KATILIM "
        "FİNANSMAN RAW SCRAPER "
        "KONTROL GEREKİYOR ⚠️"
    )


print(
    "=" * 118
)


# ============================================================
# DOWNLOAD
# ============================================================

files.download(
    OUTPUT_FILE
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.8/64.8 kB 2.1 MB/s eta 0:00:00
ZİRAAT KATILIM - BİREYSEL FİNANSMAN RAW SCRAPER V1
Beklenen ürün: 18

[01/18] Konut Finansmanı
https://www.ziraatkatilim.com.tr/bireysel/finansman-urunleri/konut-gayrimenkul-finansmani/konut-finansmani
  METHOD : FAILED
  STATUS : 403
  RESULT : ❌
  ERROR  : İçerik alınamadı.
[02/18] Kentsel Dönüşüm Finansmanı
https://www.ziraatkatilim.com.tr/bireysel/finansman-urunleri/konut-finansmani/kentsel-donusum-finansmani
  METHOD : FAILED
  STATUS : 403
  RESULT : ❌
  ERROR  : İçerik alınamadı.
[03/18] Bireysel Arsa Finansmanı
https://www.ziraatkatilim.com.tr/bireysel/finansman-urunleri/konut-gayrimenkul-finansmani/bireysel-arsa-finansmani
  METHOD : FAILED
  STATUS : 403
  RESULT : ❌
  ERROR  : İçerik alınamadı.
[04/18] Bireysel İş Yeri Finansmanı
https://www.ziraatkatilim.com.tr/bireysel/finansman-urunleri/konut-gayrimenkul-finansmani/bireysel-is-yeri-finansmani
  METHOD : FAILED
  STATUS : 403
  RESULT : ❌
  ERROR

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# ZİRAAT KATILIM
# FİNANSMAN RAW SCRAPER V2 - OFFICIAL MIRROR
#
# Kaynak:
# Ziraat Katılım Özel Bankacılık resmi sitesi
#
# Hedef:
# Ana 12 bireysel finansman ürününün detay sayfalarını
# otomatik keşfet + ham metinlerini çek.
# ============================================================


!pip -q install requests==2.32.4 beautifulsoup4


import json
import re
import time

import requests

from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

from google.colab import files


# ============================================================
# AYARLAR
# ============================================================

BASE_URL = (
    "https://www."
    "ziraatkatilimozelbankacilik.com.tr"
)

OUTPUT_FILE = (
    "/content/"
    "ziraat_katilim_finansmanlar_mirror_raw.json"
)


CATEGORY_URLS = [
    (
        BASE_URL
        + "/finansman-urunleri/"
        + "konut-finansmanlari"
    ),

    (
        BASE_URL
        + "/finansman-urunleri/"
        + "tasit-finansmanlari"
    ),

    (
        BASE_URL
        + "/finansman-urunleri/"
        + "ihtiyac-finansmani"
    ),
]


# ============================================================
# HEDEF 12 ÜRÜN
# ============================================================

TARGETS = {

    "Konut Finansmanı":
        "Konut-Gayrimenkul Finansmanı",

    "Kentsel Dönüşüm Finansmanı":
        "Konut-Gayrimenkul Finansmanı",

    "Bireysel Arsa Finansmanı":
        "Konut-Gayrimenkul Finansmanı",

    "Bireysel İş Yeri Finansmanı":
        "Konut-Gayrimenkul Finansmanı",

    "Taşıt Finansmanı":
        "Taşıt Finansmanı",

    "TOGG Finansmanı":
        "Taşıt Finansmanı",

    "Eğitim Finansmanı":
        "İhtiyaç Finansmanı",

    "Doğalgaz Dönüşüm Finansmanı":
        "İhtiyaç Finansmanı",

    "Hac ve Umre Finansmanı":
        "İhtiyaç Finansmanı",

    "İpotekli Bireysel Finansman":
        "İhtiyaç Finansmanı",

    "Yasa Kapsamında İpotekli Bireysel Finansman":
        "İhtiyaç Finansmanı",

    "Dayanıklı Tüketim Finansmanı":
        "İhtiyaç Finansmanı",
}


EXPECTED_COUNT = len(
    TARGETS
)


# ============================================================
# SESSION
# ============================================================

session = requests.Session()

session.headers.update(
    {
        "User-Agent": (
            "Mozilla/5.0 "
            "(Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 "
            "(KHTML, like Gecko) "
            "Chrome/151.0.0.0 "
            "Safari/537.36"
        ),

        "Accept-Language":
            "tr-TR,tr;q=0.9,en;q=0.8",

        "Accept":
            (
                "text/html,"
                "application/xhtml+xml,"
                "application/xml;q=0.9,"
                "*/*;q=0.8"
            ),
    }
)


# ============================================================
# NORMALIZATION
# ============================================================

def clean_text(value):

    value = str(
        value or ""
    )

    value = (
        value
        .replace("\xa0", " ")
        .replace("’", "'")
        .replace("‘", "'")
        .replace("–", "-")
        .replace("—", "-")
    )

    value = re.sub(
        r"[ \t]+",
        " ",
        value
    )

    value = re.sub(
        r"\n[ \t]+",
        "\n",
        value
    )

    value = re.sub(
        r"\n{3,}",
        "\n\n",
        value
    )

    return value.strip()


def normalize(value):

    value = clean_text(
        value
    )

    value = (
        value
        .replace("İ", "i")
        .replace("I", "ı")
        .casefold()
    )

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value.strip()


# ============================================================
# TARGET MATCH
# ============================================================

TARGET_NORMALIZED = {
    normalize(name): name
    for name in TARGETS
}


def match_target(title):

    title_n = normalize(
        title
    )

    # Tam eşleşme
    if title_n in TARGET_NORMALIZED:

        return TARGET_NORMALIZED[
            title_n
        ]

    # H1 sonunda/başında küçük fark varsa
    for normalized_name, real_name in (
        TARGET_NORMALIZED.items()
    ):

        if (
            normalized_name
            in title_n
            or
            title_n
            in normalized_name
        ):

            return real_name

    return None


# ============================================================
# PAGE GET
# ============================================================

def get_page(url):

    try:

        response = session.get(
            url,
            timeout=30,
            allow_redirects=True
        )

        return (
            response,
            BeautifulSoup(
                response.text,
                "html.parser"
            )
        )

    except Exception as error:

        print(
            "REQUEST ERROR:",
            url,
            error
        )

        return None, None


# ============================================================
# CLEAN PAGE CONTENT
# ============================================================

def extract_main_text(soup):

    if soup is None:

        return ""

    # Gereksiz alanları çıkar.
    for selector in [
        "script",
        "style",
        "noscript",
        "svg",
        "nav",
        "header",
        "footer",
        "form",
    ]:

        for tag in soup.select(
            selector
        ):

            tag.decompose()


    # Öncelik main
    main = soup.find(
        "main"
    )

    if main is None:

        # H1'in üst parent bloklarından
        # anlamlı olanı bulmaya çalış.
        h1 = soup.find(
            "h1"
        )

        if h1 is not None:

            main = h1.parent

            # Çok küçük parent ise yukarı çık.
            for _ in range(4):

                if main is None:
                    break

                text = clean_text(
                    main.get_text(
                        "\n",
                        strip=True
                    )
                )

                if len(text) >= 500:
                    break

                main = main.parent


    if main is None:

        main = soup.body


    if main is None:

        return ""


    text = main.get_text(
        "\n",
        strip=True
    )

    lines = []

    blocked_lines = {
        "Whatsapp Hattı",
        "WhatsApp Hattı",
        "Özel Bankacılık",
        "Ziraat Finans Grubu",
        "Katılım Mobil",
    }


    for line in text.splitlines():

        line = clean_text(
            line
        )

        if not line:
            continue

        if line in blocked_lines:
            continue

        lines.append(
            line
        )


    return clean_text(
        "\n".join(
            lines
        )
    )


# ============================================================
# H1
# ============================================================

def get_h1(soup):

    if soup is None:

        return ""

    h1 = soup.find(
        "h1"
    )

    if not h1:

        return ""

    return clean_text(
        h1.get_text(
            " ",
            strip=True
        )
    )


# ============================================================
# URL FILTER
# ============================================================

def valid_detail_url(url):

    parsed = urlparse(
        url
    )

    if (
        parsed.netloc
        not in {
            "www.ziraatkatilimozelbankacilik.com.tr",
            "ziraatkatilimozelbankacilik.com.tr",
        }
    ):

        return False


    path_n = normalize(
        parsed.path
    )


    if (
        "/finansman-urunleri"
        not in path_n
    ):

        return False


    bad = (
        "/en/",
        "/ar/",
        "/sigorta",
        "/hesaplama",
    )


    if any(
        x in path_n
        for x in bad
    ):

        return False


    return True


# ============================================================
# 1) CATEGORY PAGES
# ============================================================

print(
    "=" * 115
)

print(
    "ZİRAAT KATILIM - "
    "OFFICIAL MIRROR DISCOVERY"
)

print(
    "=" * 115
)


discovered_urls = set()


for index, category_url in enumerate(
    CATEGORY_URLS,
    start=1
):

    print()
    print(
        f"[CATEGORY {index}/"
        f"{len(CATEGORY_URLS)}]"
    )

    print(
        category_url
    )


    response, soup = get_page(
        category_url
    )


    if response is None:

        print(
            "  RESULT: ❌"
        )

        continue


    print(
        "  STATUS:",
        response.status_code
    )


    if response.status_code != 200:

        print(
            "  RESULT: ❌"
        )

        continue


    links_found = 0


    for a in soup.find_all(
        "a",
        href=True
    ):

        href = urljoin(
            category_url,
            a[
                "href"
            ]
        )


        if not valid_detail_url(
            href
        ):

            continue


        # Category'nin kendisini çıkar.
        if (
            href.rstrip("/")
            == category_url.rstrip("/")
        ):

            continue


        discovered_urls.add(
            href.split("#")[0]
        )

        links_found += 1


    print(
        "  Detail link:",
        links_found
    )

    print(
        "  RESULT: ✅"
    )


# ============================================================
# BİLDİĞİMİZ FALLBACK DETAY URL'LERİ
#
# Site içindeki bazı kart linkleri hatalı/eksik
# olabildiği için doğrulanmış ek yolları da ekliyoruz.
# ============================================================

KNOWN_URLS = [

    (
        BASE_URL
        + "/finansman-urunleri/"
        + "konut-finansmani"
    ),

    (
        BASE_URL
        + "/finansman-urunleri/"
        + "bireysel-arsa-finansmani"
    ),

    (
        BASE_URL
        + "/finansman-urunleri/"
        + "bireysel-yeri-finansmani"
    ),

    (
        BASE_URL
        + "/finansman-urunleri/"
        + "tasit-finansmanlari/"
        + "tasit-finansmani"
    ),

    (
        BASE_URL
        + "/finansman-urunleri/"
        + "tasit-finansmanlari/"
        + "togg-finansmani"
    ),

    (
        BASE_URL
        + "/finansman-urunleri/"
        + "egitim-finansmani"
    ),

    (
        BASE_URL
        + "/finansman-urunleri/"
        + "egitim-finansmanii"
    ),

    (
        BASE_URL
        + "/finansman-urunleri/"
        + "ihtiyac-finansmani/"
        + "dogalgaz-donusum-finansmani"
    ),

    (
        BASE_URL
        + "/finansman-urunleri/"
        + "ihtiyac-finansmani/"
        + "hac-ve-umre-finansmani"
    ),

    (
        BASE_URL
        + "/finansman-urunleri/"
        + "ipotekli-bireysel-finansman"
    ),

    (
        BASE_URL
        + "/finansman-urunleri/"
        + "ihtiyac-finansmani/"
        + "yasa-kapsaminda-"
        + "ipotekli-bireysel-finansman"
    ),

    (
        BASE_URL
        + "/finansman-urunleri/"
        + "ihtiyac-finansmani/"
        + "dayanikli-tuketim-finansmani"
    ),

    # Kentsel için olası güncel rotalar.
    (
        BASE_URL
        + "/finansman-urunleri/"
        + "kentsel-donusum-finansmani"
    ),

    (
        BASE_URL
        + "/finansman-urunleri/"
        + "konut-finansmanlari/"
        + "kentsel-donusum-finansmani"
    ),
]


for url in KNOWN_URLS:

    discovered_urls.add(
        url
    )


# ============================================================
# 2) DETAIL PAGE DISCOVERY
# ============================================================

print()
print(
    "=" * 115
)

print(
    "DETAIL PAGE KONTROLÜ"
)

print(
    "=" * 115
)


found = {}

checked = []


for index, url in enumerate(
    sorted(
        discovered_urls
    ),
    start=1
):

    response, soup = get_page(
        url
    )


    if response is None:

        continue


    status = response.status_code

    h1 = get_h1(
        soup
    )


    checked.append(
        {
            "url": url,
            "status": status,
            "h1": h1,
        }
    )


    if status != 200:

        continue


    target = match_target(
        h1
    )


    if target is None:

        continue


    raw_text = extract_main_text(
        soup
    )


    # Gerçek içerik kontrolü
    if len(raw_text) < 150:

        continue


    # Aynı ürün için daha uzun sayfa varsa onu al.
    old = found.get(
        target
    )


    candidate = {
        "urun_adi":
            target,

        "liste_kategorisi":
            TARGETS[
                target
            ],

        "kaynak_url":
            url,

        "http_status":
            status,

        "fetch_method":
            "official_private_banking_mirror",

        "ham_metin":
            raw_text,
    }


    if (
        old is None
        or
        len(
            candidate[
                "ham_metin"
            ]
        )
        >
        len(
            old[
                "ham_metin"
            ]
        )
    ):

        found[
            target
        ] = candidate


    time.sleep(
        0.15
    )


# ============================================================
# ORDER
# ============================================================

records = []


for target_name in TARGETS:

    if target_name in found:

        records.append(
            found[
                target_name
            ]
        )


missing = [
    target
    for target
    in TARGETS
    if target not in found
]


# ============================================================
# OUTPUT
# ============================================================

output = {

    "banka":
        "Ziraat Katılım Bankası A.Ş.",

    "kaynak_tipi":
        (
            "Ziraat Katılım "
            "Özel Bankacılık resmi sitesi"
        ),

    "beklenen_urun_sayisi":
        EXPECTED_COUNT,

    "basarili_urun_sayisi":
        len(
            records
        ),

    "eksik_urun_sayisi":
        len(
            missing
        ),

    "eksik_urunler":
        missing,

    "urunler":
        records,

    "kontrol_edilen_url_sayisi":
        len(
            checked
        ),
}


with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# RESULT
# ============================================================

print()
print(
    "=" * 115
)

print(
    "ZİRAAT MIRROR RAW SONUCU"
)

print(
    "=" * 115
)


print(
    "Beklenen ürün   :",
    EXPECTED_COUNT
)

print(
    "Başarılı ürün   :",
    len(
        records
    )
)

print(
    "Eksik ürün      :",
    len(
        missing
    )
)

print(
    "Kontrol URL     :",
    len(
        checked
    )
)

print(
    "Dosya           :",
    OUTPUT_FILE
)


print()
print(
    "BULUNAN ÜRÜNLER:"
)


for index, record in enumerate(
    records,
    start=1
):

    print(
        (
            f"[{index:02d}] "
            f"{record['urun_adi']}"
        )
    )

    print(
        "     URL   :",
        record[
            "kaynak_url"
        ]
    )

    print(
        "     Length:",
        len(
            record[
                "ham_metin"
            ]
        )
    )


if missing:

    print()
    print(
        "EKSİK ÜRÜNLER:"
    )

    for item in missing:

        print(
            "-",
            item
        )


print()
print(
    "=" * 115
)


if (
    len(records)
    == EXPECTED_COUNT
):

    print(
        "SONUÇ: "
        "12/12 MIRROR RAW BAŞARILI ✅"
    )

else:

    print(
        (
            "SONUÇ: MIRROR RAW "
            "KONTROL GEREKİYOR ⚠️"
        )
    )


print(
    "=" * 115
)


files.download(
    OUTPUT_FILE
)

ZİRAAT KATILIM - OFFICIAL MIRROR DISCOVERY

[CATEGORY 1/3]
https://www.ziraatkatilimozelbankacilik.com.tr/finansman-urunleri/konut-finansmanlari
  STATUS: 200
  Detail link: 11
  RESULT: ✅

[CATEGORY 2/3]
https://www.ziraatkatilimozelbankacilik.com.tr/finansman-urunleri/tasit-finansmanlari
  STATUS: 200
  Detail link: 12
  RESULT: ✅

[CATEGORY 3/3]
https://www.ziraatkatilimozelbankacilik.com.tr/finansman-urunleri/ihtiyac-finansmani
  STATUS: 200
  Detail link: 16
  RESULT: ✅

DETAIL PAGE KONTROLÜ

ZİRAAT MIRROR RAW SONUCU
Beklenen ürün   : 12
Başarılı ürün   : 11
Eksik ürün      : 1
Kontrol URL     : 21
Dosya           : /content/ziraat_katilim_finansmanlar_mirror_raw.json

BULUNAN ÜRÜNLER:
[01] Konut Finansmanı
     URL   : https://www.ziraatkatilimozelbankacilik.com.tr/finansman-urunleri/yp-konut-finansmani
     Length: 1493
[02] Bireysel Arsa Finansmanı
     URL   : https://www.ziraatkatilimozelbankacilik.com.tr/finansman-urunleri/bireysel-arsa-finansmani
     Length: 1408
[03] Bire

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# ZİRAAT KATILIM
# KENTSEL DÖNÜŞÜM PATCH V3
#
# Mevcut 11 ürüne dokunmaz.
# Eksik Kentsel Dönüşüm Finansmanı ürününü
# Ziraat Katılım Özel Bankacılık resmi
# İngilizce detay sayfasından ekler.
# ============================================================

!pip -q install requests==2.32.4 beautifulsoup4


import json
import re
import requests

from bs4 import BeautifulSoup
from google.colab import files


# ============================================================
# DOSYALAR
# ============================================================

INPUT_FILE = (
    "/content/"
    "ziraat_katilim_finansmanlar_mirror_raw.json"
)

OUTPUT_FILE = (
    "/content/"
    "ziraat_katilim_finansmanlar_mirror_raw_v3.json"
)


# ============================================================
# GERÇEK RESMİ DETAIL URL
# ============================================================

KENTSEL_URL = (
    "https://www."
    "ziraatkatilimozelbankacilik.com.tr/"
    "en/financing-products/"
    "urban-transformation-financing"
)


TARGET_NAME = (
    "Kentsel Dönüşüm Finansmanı"
)


# ============================================================
# NORMALIZATION
# ============================================================

def clean_text(value):

    value = str(value or "")

    value = (
        value
        .replace("\xa0", " ")
        .replace("’", "'")
        .replace("‘", "'")
        .replace("–", "-")
        .replace("—", "-")
    )

    value = re.sub(
        r"[ \t]+",
        " ",
        value
    )

    value = re.sub(
        r"\n[ \t]+",
        "\n",
        value
    )

    value = re.sub(
        r"\n{3,}",
        "\n\n",
        value
    )

    return value.strip()


def normalize(value):

    value = clean_text(value)

    return (
        value
        .replace("İ", "i")
        .replace("I", "ı")
        .casefold()
        .strip()
    )


# ============================================================
# LOAD
# ============================================================

with open(
    INPUT_FILE,
    "r",
    encoding="utf-8"
) as f:

    data = json.load(f)


records = data.get(
    "urunler",
    []
)


print(
    "=" * 115
)

print(
    "ZİRAAT KATILIM - "
    "KENTSEL DÖNÜŞÜM PATCH V3"
)

print(
    "=" * 115
)

print(
    "Mevcut ürün:",
    len(records)
)


# ============================================================
# REMOVE PREVIOUS KENTSEL IF ANY
# ============================================================

records = [
    record
    for record in records
    if normalize(
        record.get(
            "urun_adi",
            ""
        )
    )
    != normalize(
        TARGET_NAME
    )
]


# ============================================================
# REQUEST
# ============================================================

session = requests.Session()

session.headers.update(
    {
        "User-Agent": (
            "Mozilla/5.0 "
            "(Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 "
            "(KHTML, like Gecko) "
            "Chrome/151.0.0.0 "
            "Safari/537.36"
        ),

        "Accept-Language":
            "en-US,en;q=0.9,tr;q=0.8",
    }
)


response = session.get(
    KENTSEL_URL,
    timeout=30,
    allow_redirects=True
)


print(
    "HTTP status:",
    response.status_code
)

print(
    "Final URL  :",
    response.url
)


if response.status_code != 200:

    raise RuntimeError(
        (
            "Kentsel Dönüşüm resmi "
            "sayfası alınamadı. HTTP "
            f"{response.status_code}"
        )
    )


# ============================================================
# PARSE
# ============================================================

soup = BeautifulSoup(
    response.text,
    "html.parser"
)


for selector in [
    "script",
    "style",
    "noscript",
    "svg",
    "header",
    "footer",
    "nav",
    "form",
]:

    for tag in soup.select(
        selector
    ):

        tag.decompose()


# ============================================================
# FIND MAIN CONTENT
# ============================================================

main = soup.find(
    "main"
)


if main is None:

    h1 = soup.find(
        "h1"
    )

    if h1:

        current = h1

        for _ in range(7):

            if current is None:
                break

            text = clean_text(
                current.get_text(
                    "\n",
                    strip=True
                )
            )

            if (
                "Urban Transformation"
                in text
                and
                len(text) >= 500
            ):

                main = current
                break

            current = current.parent


if main is None:

    main = soup.body


if main is None:

    raise RuntimeError(
        "Sayfa body/main bulunamadı."
    )


raw_text = clean_text(
    main.get_text(
        "\n",
        strip=True
    )
)


# ============================================================
# TRIM FOOTER / MENU NOISE
# ============================================================

lines = [
    clean_text(line)
    for line in raw_text.splitlines()
    if clean_text(line)
]


start_index = None


for i, line in enumerate(lines):

    if (
        "urban transformation financing"
        in normalize(line)
    ):

        start_index = i
        break


if start_index is not None:

    lines = lines[
        start_index:
    ]


stop_markers = {
    "whatsapp line",
    "private banking",
    "ziraat finance group",
}


final_lines = []


for line in lines:

    normalized = normalize(
        line
    )

    if (
        final_lines
        and
        normalized
        in stop_markers
    ):

        break

    final_lines.append(
        line
    )


raw_text = clean_text(
    "\n".join(
        final_lines
    )
)


# ============================================================
# CONTENT VALIDATION
# ============================================================

required_markers = [
    "Urban Transformation Financing",
    "Law No. 6306",
    "1,250,000",
    "10 years",
    "7 years",
]


missing_markers = [
    marker
    for marker in required_markers
    if normalize(marker)
    not in normalize(raw_text)
]


print()
print(
    "=" * 115
)

print(
    "KENTSEL DÖNÜŞÜM RAW"
)

print(
    "=" * 115
)

print(
    raw_text
)

print()
print(
    "Raw length:",
    len(raw_text)
)

print(
    "Eksik marker:",
    missing_markers
)


if missing_markers:

    raise RuntimeError(
        (
            "Kentsel Dönüşüm sayfası "
            "beklenen içeriği taşımıyor: "
            f"{missing_markers}"
        )
    )


# ============================================================
# ADD RECORD
# ============================================================

records.append(
    {
        "urun_adi":
            TARGET_NAME,

        "liste_kategorisi":
            "Konut-Gayrimenkul Finansmanı",

        "kaynak_url":
            KENTSEL_URL,

        "http_status":
            response.status_code,

        "fetch_method":
            (
                "official_private_banking_"
                "english_detail"
            ),

        "ham_metin":
            raw_text,
    }
)


# ============================================================
# EXPECTED ORDER
# ============================================================

EXPECTED_ORDER = [
    "Konut Finansmanı",
    "Kentsel Dönüşüm Finansmanı",
    "Bireysel Arsa Finansmanı",
    "Bireysel İş Yeri Finansmanı",
    "Taşıt Finansmanı",
    "TOGG Finansmanı",
    "Eğitim Finansmanı",
    "Doğalgaz Dönüşüm Finansmanı",
    "Hac ve Umre Finansmanı",
    "İpotekli Bireysel Finansman",
    "Yasa Kapsamında İpotekli Bireysel Finansman",
    "Dayanıklı Tüketim Finansmanı",
]


order_map = {
    normalize(name): index
    for index, name
    in enumerate(
        EXPECTED_ORDER
    )
}


records.sort(
    key=lambda record:
        order_map.get(
            normalize(
                record.get(
                    "urun_adi",
                    ""
                )
            ),
            999
        )
)


# ============================================================
# VALIDATION
# ============================================================

errors = []


if len(records) != 12:

    errors.append(
        (
            "Kayıt sayısı 12 değil: "
            f"{len(records)}"
        )
    )


names = [
    normalize(
        record.get(
            "urun_adi",
            ""
        )
    )
    for record in records
]


if len(names) != len(
    set(names)
):

    errors.append(
        "Duplicate ürün adı bulundu."
    )


for expected in EXPECTED_ORDER:

    if normalize(
        expected
    ) not in names:

        errors.append(
            (
                "Eksik ürün: "
                + expected
            )
        )


for record in records:

    if not record.get(
        "ham_metin"
    ):

        errors.append(
            (
                "Boş ham_metin: "
                + record[
                    "urun_adi"
                ]
            )
        )


# ============================================================
# SAVE
# ============================================================

output = {

    "banka":
        "Ziraat Katılım Bankası A.Ş.",

    "kaynak_tipi":
        (
            "Ziraat Katılım "
            "Özel Bankacılık resmi sitesi"
        ),

    "beklenen_urun_sayisi":
        12,

    "basarili_urun_sayisi":
        len(records),

    "eksik_urun_sayisi":
        0 if not errors else len(errors),

    "eksik_urunler":
        [],

    "urunler":
        records,
}


with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# RESULT
# ============================================================

print()
print(
    "=" * 115
)

print(
    "MIRROR RAW V3 SONUCU"
)

print(
    "=" * 115
)

print(
    "Toplam ürün     :",
    len(records)
)

print(
    "Validation error:",
    len(errors)
)

print(
    "Dosya           :",
    OUTPUT_FILE
)


print()
print(
    "ÜRÜNLER:"
)


for index, record in enumerate(
    records,
    start=1
):

    print(
        f"[{index:02d}] "
        f"{record['urun_adi']}"
    )

    print(
        "     Method:",
        record[
            "fetch_method"
        ]
    )

    print(
        "     Length:",
        len(
            record[
                "ham_metin"
            ]
        )
    )


if errors:

    print()
    print(
        "HATALAR:"
    )

    for error in errors:

        print(
            "-",
            error
        )


print()
print(
    "=" * 115
)


if not errors:

    print(
        "SONUÇ: ZİRAAT MIRROR "
        "12/12 RAW BAŞARILI ✅"
    )

else:

    print(
        "SONUÇ: KONTROL "
        "GEREKİYOR ❌"
    )


print(
    "=" * 115
)


files.download(
    OUTPUT_FILE
)

ZİRAAT KATILIM - KENTSEL DÖNÜŞÜM PATCH V3
Mevcut ürün: 11
HTTP status: 200
Final URL  : https://www.ziraatkatilimozelbankacilik.com.tr/en/financing-products/urban-transformation-financing

KENTSEL DÖNÜŞÜM RAW
What is Urban Transformation Financing? Urban Transformation Financing is a loan provided by our bank for the renewal of buildings located in risky areas defined under Law No. 6306 on the “Transformation of Areas Under Disaster Risk.”
To benefit from the profit share support rates determined by the Republic of Türkiye Ministry of Environment, Urbanization, and Climate Change, the property must either be located in a designated risky area or be identified as a “risky building” through technical reports, even if outside a risky zone.
Who Can Benefit from Urban Transformation Financing?
Property owners who want to rebuild their homes,
Tenants or holders of limited real rights who have lived in risky buildings for at least one year,
Owners of homes identified as risky buildings who wi

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# ZİRAAT KATILIM
# OFFICIAL MIRROR PRODUCT INVENTORY V4
#
# AMAÇ:
# - Sabit ürün listesi YOK.
# - Fuzzy title matching YOK.
# - H1 neyse ürün adı O.
# - YP Konut ile normal Konut birbirine karışmaz.
# - Mevcut resmi Özel Bankacılık finansman sayfaları crawl edilir.
# ============================================================


!pip -q install requests==2.32.4 beautifulsoup4


import json
import re
import time

import requests

from bs4 import BeautifulSoup
from collections import deque
from urllib.parse import (
    urljoin,
    urlparse,
    urldefrag,
)

from google.colab import files


# ============================================================
# AYARLAR
# ============================================================

BASE_URL = (
    "https://www."
    "ziraatkatilimozelbankacilik.com.tr"
)

OUTPUT_FILE = (
    "/content/"
    "ziraat_katilim_mirror_inventory_v4.json"
)


SEED_URLS = [

    (
        BASE_URL
        + "/finansman-urunleri"
    ),

    (
        BASE_URL
        + "/finansman-urunleri/"
        + "konut-finansmanlari"
    ),

    (
        BASE_URL
        + "/finansman-urunleri/"
        + "tasit-finansmanlari"
    ),

    (
        BASE_URL
        + "/finansman-urunleri/"
        + "ihtiyac-finansmani"
    ),
]


# Kentsel Dönüşüm Türkçe detay sayfası
# görünmediği için doğrulanmış resmi İngilizce sayfa.
KENTSEL_EN_URL = (
    BASE_URL
    + "/en/financing-products/"
    + "urban-transformation-financing"
)


MAX_DEPTH = 3
MAX_PAGES = 150


# ============================================================
# NORMALIZATION
# ============================================================

def clean_text(value):

    value = str(
        value or ""
    )

    value = (
        value
        .replace("\xa0", " ")
        .replace("’", "'")
        .replace("‘", "'")
        .replace("–", "-")
        .replace("—", "-")
    )

    value = re.sub(
        r"[ \t]+",
        " ",
        value
    )

    value = re.sub(
        r"\n[ \t]+",
        "\n",
        value
    )

    value = re.sub(
        r"\n{3,}",
        "\n\n",
        value
    )

    return value.strip()


def normalize(value):

    value = clean_text(
        value
    )

    value = (
        value
        .replace("İ", "i")
        .replace("I", "ı")
        .casefold()
    )

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value.strip()


# ============================================================
# URL
# ============================================================

def normalize_url(url):

    if not url:
        return ""

    url = urljoin(
        BASE_URL,
        url
    )

    url, _ = urldefrag(
        url
    )

    parsed = urlparse(
        url
    )

    host = (
        parsed.netloc
        .lower()
    )

    if host not in {
        "www.ziraatkatilimozelbankacilik.com.tr",
        "ziraatkatilimozelbankacilik.com.tr",
    }:

        return ""

    if not parsed.path:
        return ""

    result = (
        "https://www."
        "ziraatkatilimozelbankacilik.com.tr"
        + parsed.path
    )

    return result.rstrip("/")


def is_finance_url(url):

    url_n = normalize(
        url
    )

    parsed = urlparse(
        url
    )

    path = parsed.path.rstrip("/")

    # Türkçe finansman alanı
    if path.startswith(
        "/finansman-urunleri"
    ):

        return True

    return False


# ============================================================
# CATEGORY PATHS
# ============================================================

CATEGORY_PATHS = {

    "/finansman-urunleri",

    (
        "/finansman-urunleri/"
        "konut-finansmanlari"
    ),

    (
        "/finansman-urunleri/"
        "tasit-finansmanlari"
    ),

    (
        "/finansman-urunleri/"
        "ihtiyac-finansmani"
    ),
}


def is_category_url(url):

    path = (
        urlparse(
            url
        )
        .path
        .rstrip("/")
    )

    return (
        path
        in CATEGORY_PATHS
    )


# ============================================================
# SESSION
# ============================================================

session = requests.Session()

session.headers.update(
    {
        "User-Agent": (
            "Mozilla/5.0 "
            "(Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 "
            "(KHTML, like Gecko) "
            "Chrome/151.0.0.0 "
            "Safari/537.36"
        ),

        "Accept-Language":
            "tr-TR,tr;q=0.9,en;q=0.8",
    }
)


# ============================================================
# FETCH
# ============================================================

def fetch_page(url):

    try:

        response = session.get(
            url,
            timeout=30,
            allow_redirects=True
        )

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        return (
            response,
            soup
        )

    except Exception as error:

        return (
            None,
            None
        )


# ============================================================
# H1
# ============================================================

def get_h1(soup):

    if soup is None:

        return ""

    h1 = soup.find(
        "h1"
    )

    if not h1:

        return ""

    return clean_text(
        h1.get_text(
            " ",
            strip=True
        )
    )


# ============================================================
# MAIN TEXT
# ============================================================

def extract_text(soup):

    if soup is None:

        return ""

    # Clone gibi kullanmak için yeniden parse
    soup = BeautifulSoup(
        str(soup),
        "html.parser"
    )

    for selector in [
        "script",
        "style",
        "noscript",
        "svg",
        "nav",
        "header",
        "footer",
        "form",
    ]:

        for tag in soup.select(
            selector
        ):

            tag.decompose()


    main = soup.find(
        "main"
    )


    if main is None:

        h1 = soup.find(
            "h1"
        )

        if h1:

            current = h1

            for _ in range(7):

                if current is None:
                    break

                text = clean_text(
                    current.get_text(
                        "\n",
                        strip=True
                    )
                )

                if len(text) >= 400:

                    main = current
                    break

                current = current.parent


    if main is None:

        main = soup.body


    if main is None:

        return ""


    text = clean_text(
        main.get_text(
            "\n",
            strip=True
        )
    )


    return text


# ============================================================
# CATEGORY DERIVATION
# ============================================================

def derive_category(
    title,
    url
):

    title_n = normalize(
        title
    )

    path_n = normalize(
        urlparse(
            url
        ).path
    )


    # --------------------------------
    # Konut / Gayrimenkul
    # --------------------------------

    if any(
        marker in title_n
        for marker in [
            "konut finansmanı",
            "arsa finansmanı",
            "iş yeri finansmanı",
            "kentsel dönüşüm",
        ]
    ):

        return (
            "Konut-Gayrimenkul Finansmanı"
        )


    # --------------------------------
    # Taşıt
    # --------------------------------

    if any(
        marker in title_n
        for marker in [
            "taşıt finansmanı",
            "togg",
            "tekne",
            "yat finansmanı",
        ]
    ):

        return (
            "Taşıt Finansmanı"
        )


    # --------------------------------
    # İhtiyaç
    # --------------------------------

    if any(
        marker in title_n
        for marker in [
            "eğitim finansmanı",
            "doğalgaz",
            "hac",
            "umre",
            "ipotekli bireysel",
            "dayanıklı tüketim",
            "ihtiyaç finansmanı",
        ]
    ):

        return (
            "İhtiyaç Finansmanı"
        )


    # Route fallback
    if (
        "/tasit-finansmanlari/"
        in path_n
    ):

        return (
            "Taşıt Finansmanı"
        )

    if (
        "/ihtiyac-finansmani/"
        in path_n
    ):

        return (
            "İhtiyaç Finansmanı"
        )

    if (
        "/konut-finansmanlari/"
        in path_n
    ):

        return (
            "Konut-Gayrimenkul Finansmanı"
        )


    return (
        "Bireysel Finansman"
    )


# ============================================================
# DISCOVERY
# ============================================================

print(
    "=" * 115
)

print(
    "ZİRAAT KATILIM - "
    "OFFICIAL MIRROR INVENTORY V4"
)

print(
    "=" * 115
)


queue = deque(
    (
        normalize_url(url),
        0
    )
    for url in SEED_URLS
)


visited = set()

pages = {}

discovered_links = set()


while (
    queue
    and
    len(visited)
    < MAX_PAGES
):

    url, depth = (
        queue.popleft()
    )

    if not url:
        continue

    if url in visited:
        continue

    visited.add(
        url
    )


    print(
        f"[{len(visited):03d}] "
        f"depth={depth}"
    )

    print(
        url
    )


    response, soup = fetch_page(
        url
    )


    if response is None:

        print(
            "   ERROR"
        )

        continue


    print(
        "   HTTP:",
        response.status_code
    )


    if response.status_code != 200:

        continue


    h1 = get_h1(
        soup
    )

    raw_text = extract_text(
        soup
    )


    print(
        "   H1:",
        h1 or "-"
    )

    print(
        "   Length:",
        len(raw_text)
    )


    pages[
        url
    ] = {
        "url":
            url,

        "h1":
            h1,

        "ham_metin":
            raw_text,

        "http_status":
            response.status_code,

        "depth":
            depth,
    }


    # --------------------------------
    # Link discovery
    # --------------------------------

    for a in soup.find_all(
        "a",
        href=True
    ):

        href = normalize_url(
            a.get(
                "href"
            )
        )

        if not href:
            continue

        if not is_finance_url(
            href
        ):
            continue

        discovered_links.add(
            href
        )

        if (
            depth < MAX_DEPTH
            and
            href not in visited
        ):

            queue.append(
                (
                    href,
                    depth + 1
                )
            )


    time.sleep(
        0.1
    )


# ============================================================
# DETAIL RECORDS
# ============================================================

records = []


for url, page in pages.items():

    # kategori sayfalarını ürün saymıyoruz
    if is_category_url(
        url
    ):

        continue


    title = clean_text(
        page[
            "h1"
        ]
    )


    if not title:
        continue


    raw_text = clean_text(
        page[
            "ham_metin"
        ]
    )


    # Detail sayfası için minimum içerik
    if len(raw_text) < 250:
        continue


    # Menü / genel sayfaları çıkar
    title_n = normalize(
        title
    )


    if title_n in {
        "finansman ürünleri",
        "konut finansmanları",
        "taşıt finansmanları",
    }:
        continue


    records.append(
        {
            "urun_adi":
                title,

            "liste_kategorisi":
                derive_category(
                    title,
                    url
                ),

            "kaynak_url":
                url,

            "http_status":
                page[
                    "http_status"
                ],

            "fetch_method":
                (
                    "official_private_banking_"
                    "strict_h1"
                ),

            "source_language":
                "tr",

            "ham_metin":
                raw_text,
        }
    )


# ============================================================
# KENTSEL ENGLISH FALLBACK
# ============================================================

print()
print(
    "=" * 115
)

print(
    "KENTSEL ENGLISH FALLBACK"
)

print(
    "=" * 115
)


response, soup = fetch_page(
    KENTSEL_EN_URL
)


if (
    response is not None
    and
    response.status_code == 200
):

    h1 = get_h1(
        soup
    )

    raw_text = extract_text(
        soup
    )


    print(
        "HTTP:",
        response.status_code
    )

    print(
        "H1:",
        h1
    )

    print(
        "Length:",
        len(raw_text)
    )


    if (
        "urban transformation"
        in normalize(
            h1
        )
        and
        len(raw_text) >= 300
    ):

        records.append(
            {
                "urun_adi":
                    "Kentsel Dönüşüm Finansmanı",

                "liste_kategorisi":
                    (
                        "Konut-Gayrimenkul "
                        "Finansmanı"
                    ),

                "kaynak_url":
                    KENTSEL_EN_URL,

                "http_status":
                    200,

                "fetch_method":
                    (
                        "official_private_banking_"
                        "english_detail"
                    ),

                "source_language":
                    "en",

                "ham_metin":
                    raw_text,
            }
        )


# ============================================================
# DEDUPE BY EXACT PRODUCT NAME
# ============================================================

by_name = {}


for record in records:

    key = normalize(
        record[
            "urun_adi"
        ]
    )


    old = by_name.get(
        key
    )


    if old is None:

        by_name[
            key
        ] = record

        continue


    # Türkçe kaynak tercih edilir.
    if (
        old[
            "source_language"
        ]
        != "tr"
        and
        record[
            "source_language"
        ]
        == "tr"
    ):

        by_name[
            key
        ] = record

        continue


    # Aynı dilde daha uzun içerik.
    if (
        len(
            record[
                "ham_metin"
            ]
        )
        >
        len(
            old[
                "ham_metin"
            ]
        )
    ):

        by_name[
            key
        ] = record


records = list(
    by_name.values()
)


records.sort(
    key=lambda item: (
        item[
            "liste_kategorisi"
        ],
        item[
            "urun_adi"
        ],
    )
)


# ============================================================
# AUDIT
# ============================================================

errors = []


urls = [
    record[
        "kaynak_url"
    ]
    for record in records
]


names = [
    normalize(
        record[
            "urun_adi"
        ]
    )
    for record in records
]


duplicate_url = (
    len(urls)
    -
    len(set(urls))
)


duplicate_name = (
    len(names)
    -
    len(set(names))
)


if duplicate_url:

    errors.append(
        (
            "Duplicate URL: "
            f"{duplicate_url}"
        )
    )


if duplicate_name:

    errors.append(
        (
            "Duplicate ürün adı: "
            f"{duplicate_name}"
        )
    )


# Kritik ayrım
normal_home = [
    x
    for x in records
    if normalize(
        x[
            "urun_adi"
        ]
    )
    ==
    normalize(
        "Konut Finansmanı"
    )
]


yp_home = [
    x
    for x in records
    if normalize(
        x[
            "urun_adi"
        ]
    )
    ==
    normalize(
        "YP Konut Finansmanı"
    )
]


if not normal_home:

    errors.append(
        "Normal Konut Finansmanı bulunamadı."
    )


if not yp_home:

    errors.append(
        "YP Konut Finansmanı bulunamadı."
    )


# ============================================================
# SAVE
# ============================================================

output = {

    "banka":
        "Ziraat Katılım Bankası A.Ş.",

    "discovery_method":
        (
            "Official Private Banking "
            "strict H1 crawl"
        ),

    "ziyaret_edilen_sayfa":
        len(visited),

    "urun_sayisi":
        len(records),

    "duplicate_url":
        duplicate_url,

    "duplicate_urun_adi":
        duplicate_name,

    "validation_error":
        len(errors),

    "urunler":
        records,

    "errors":
        errors,
}


with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# RESULT
# ============================================================

print()
print(
    "=" * 115
)

print(
    "MIRROR INVENTORY V4 SONUCU"
)

print(
    "=" * 115
)


print(
    "Ziyaret edilen :",
    len(visited)
)

print(
    "Gerçek ürün    :",
    len(records)
)

print(
    "Duplicate URL  :",
    duplicate_url
)

print(
    "Duplicate ad   :",
    duplicate_name
)

print(
    "Validation err :",
    len(errors)
)

print(
    "Dosya          :",
    OUTPUT_FILE
)


print()
print(
    "=" * 115
)

print(
    "ÜRÜN ENVANTERİ"
)

print(
    "=" * 115
)


for index, record in enumerate(
    records,
    start=1
):

    print(
        f"[{index:02d}] "
        f"{record['urun_adi']}"
    )

    print(
        "     Kategori:",
        record[
            "liste_kategorisi"
        ]
    )

    print(
        "     Dil     :",
        record[
            "source_language"
        ]
    )

    print(
        "     URL     :",
        record[
            "kaynak_url"
        ]
    )

    print(
        "     Length  :",
        len(
            record[
                "ham_metin"
            ]
        )
    )


if errors:

    print()
    print(
        "=" * 115
    )

    print(
        "HATALAR"
    )

    print(
        "=" * 115
    )

    for error in errors:

        print(
            "-",
            error
        )


print()
print(
    "=" * 115
)

if not errors:

    print(
        "SONUÇ: MIRROR ENVANTER "
        "TEMİZ ✅"
    )

else:

    print(
        "SONUÇ: MIRROR ENVANTER "
        "KONTROL GEREKİYOR ❌"
    )

print(
    "=" * 115
)


files.download(
    OUTPUT_FILE
)

ZİRAAT KATILIM - OFFICIAL MIRROR INVENTORY V4
[001] depth=0
https://www.ziraatkatilimozelbankacilik.com.tr/finansman-urunleri
   HTTP: 200
   H1: Finansman Ürünleri
   Length: 1848
[002] depth=0
https://www.ziraatkatilimozelbankacilik.com.tr/finansman-urunleri/konut-finansmanlari
   HTTP: 200
   H1: Konut Finansmanları
   Length: 14
[003] depth=0
https://www.ziraatkatilimozelbankacilik.com.tr/finansman-urunleri/tasit-finansmanlari
   HTTP: 200
   H1: Taşıt Finansmanları
   Length: 272
[004] depth=0
https://www.ziraatkatilimozelbankacilik.com.tr/finansman-urunleri/ihtiyac-finansmani
   HTTP: 200
   H1: İhtiyaç Finansmanı
   Length: 1307
[005] depth=1
https://www.ziraatkatilimozelbankacilik.com.tr/finansman-urunleri/yp-konut-finansmani
   HTTP: 200
   H1: YP Konut Finansmanı
   Length: 1508
[006] depth=1
https://www.ziraatkatilimozelbankacilik.com.tr/finansman-urunleri/tekne-yat-finansmani
   HTTP: 200
   H1: Tekne / Yat Finansmanı
   Length: 821
[007] depth=1
https://www.ziraatkatilimoz

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# ZİRAAT KATILIM
# FINANSMAN RAW FINALIZER V5
#
# 14 official mirror ürün
# +
# 6 official-source verified fallback
# =
# 20 finansman ürünü
# ============================================================

import json
import re
from google.colab import files


# ============================================================
# FILES
# ============================================================

INPUT_FILE = (
    "/content/"
    "ziraat_katilim_mirror_inventory_v4.json"
)

OUTPUT_FILE = (
    "/content/"
    "ziraat_katilim_finansmanlar_raw_final.json"
)


BANK_NAME = (
    "Ziraat Katılım Bankası A.Ş."
)

EXPECTED_COUNT = 20


# ============================================================
# NORMALIZATION
# ============================================================

def clean_text(value):

    value = str(value or "")

    value = (
        value
        .replace("\xa0", " ")
        .replace("’", "'")
        .replace("‘", "'")
        .replace("–", "-")
        .replace("—", "-")
    )

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value.strip()


def normalize(value):

    value = clean_text(value)

    return (
        value
        .replace("İ", "i")
        .replace("I", "ı")
        .casefold()
    )


# ============================================================
# LOAD 14 MIRROR PRODUCTS
# ============================================================

with open(
    INPUT_FILE,
    "r",
    encoding="utf-8"
) as f:

    data = json.load(f)


records = data.get(
    "urunler",
    []
)


print("=" * 115)
print(
    "ZİRAAT KATILIM - "
    "FINANSMAN RAW FINALIZER V5"
)
print("=" * 115)

print(
    "Mirror ürün:",
    len(records)
)


if len(records) != 14:

    raise RuntimeError(
        (
            "Inventory V4 içinde "
            f"14 ürün bekleniyordu: {len(records)}"
        )
    )


# ============================================================
# 6 VERIFIED FALLBACK PRODUCTS
# ============================================================

FALLBACK_PRODUCTS = [

    # --------------------------------------------------------
    # KOLAY FON
    # --------------------------------------------------------

    {
        "urun_adi":
            "Kolay Fon Finansmanı",

        "liste_kategorisi":
            "İhtiyaç Finansmanı",

        "kaynak_url":
            (
                "https://www.ziraatkatilim.com.tr/"
                "bireysel/finansman-urunleri/"
                "ihtiyac-finansmani/"
                "kolayfon-finansmani"
            ),

        "fetch_method":
            "official_verified_source_summary",

        "source_language":
            "tr",

        "source_status":
            "current_verified",

        "source_reference":
            (
                "Ziraat Katılım 2026 "
                "I. Ara Dönem Faaliyet Raporu"
            ),

        "ham_metin":
            (
                "[RESMİ KAYNAK ÖZETİ] "
                "Kolay Fon Finansmanı, "
                "katılım bankacılığı ilke ve "
                "standartlarına uygun menkul "
                "kıymetlerden oluşan fonların "
                "satın alınmasının finansmanı "
                "yoluyla bireysel müşterilerin "
                "yatırım ihtiyaçlarının "
                "karşılanması amacıyla "
                "sunulan bir finansman ürünüdür. "
                "Ziraat Katılım 2026 yılının "
                "ilk çeyreğinde Kolay Fon "
                "Finansmanı ile bireysel "
                "finansman ihtiyaçlarına yönelik "
                "çözümler sunmaya devam ettiğini "
                "bildirmiştir."
            ),
    },


    # --------------------------------------------------------
    # ANINDA FİNANSMAN
    # --------------------------------------------------------

    {
        "urun_adi":
            "Anında Finansman",

        "liste_kategorisi":
            "İhtiyaç Finansmanı",

        "kaynak_url":
            (
                "https://www.ziraatkatilim.com.tr/"
                "bireysel/finansman-urunleri/"
                "ihtiyac-finansmani/"
                "aninda-finansman"
            ),

        "fetch_method":
            "official_verified_source_summary",

        "source_language":
            "tr",

        "source_status":
            "verified_product",

        "source_reference":
            (
                "Ziraat Katılım faaliyet "
                "raporları ve güncel "
                "Anında Finansman Platformu"
            ),

        "ham_metin":
            (
                "[RESMİ KAYNAK ÖZETİ] "
                "Anında Finansman, bireysel "
                "müşterilerin Ziraat Katılım ile "
                "anlaşmalı işletmelerden satın "
                "almak istedikleri ürünleri "
                "şubeye gitmeden dijital "
                "kanallar üzerinden finanse "
                "edebilmelerini sağlayan "
                "finansman ürünüdür. "
                "Finansman başvurusu ve "
                "belge onay süreçleri dijital "
                "kanallar üzerinden "
                "gerçekleştirilebilmektedir."
            ),
    },


    # --------------------------------------------------------
    # YEŞİL EV
    # --------------------------------------------------------

    {
        "urun_adi":
            "Yeşil Ev Konut Finansmanı",

        "liste_kategorisi":
            (
                "Sürdürülebilirlik Temalı "
                "Bireysel Ürünler"
            ),

        "kaynak_url":
            (
                "https://www.ziraatkatilim.com.tr/"
                "bireysel/finansman-urunleri/"
                "surdurulebilirlik-temali-"
                "bireysel-urunler/"
                "yesil-ev-konut-finansmani"
            ),

        "fetch_method":
            "official_verified_source_summary",

        "source_language":
            "tr",

        "source_status":
            "current_site_map_verified",

        "source_reference":
            (
                "Ziraat Katılım güncel "
                "Site Haritası ve "
                "Sürdürülebilirlik Raporu"
            ),

        "ham_metin":
            (
                "[RESMİ KAYNAK ÖZETİ] "
                "Yeşil Ev Konut Finansmanı, "
                "enerji verimliliği yüksek "
                "konutların finansmanına yönelik "
                "sürdürülebilirlik temalı "
                "bireysel finansman ürünüdür. "
                "Enerji Kimlik Belgesine sahip "
                "ve enerji performans sınıfı "
                "A veya B olan konutların "
                "finansmanında kullanılmaktadır. "
                "Amaç enerji verimliliği yüksek "
                "konut sayısının artırılmasına "
                "katkı sağlamaktır."
            ),
    },


    # --------------------------------------------------------
    # YEŞİL TAŞIT
    # --------------------------------------------------------

    {
        "urun_adi":
            "Yeşil Taşıt Finansmanı",

        "liste_kategorisi":
            (
                "Sürdürülebilirlik Temalı "
                "Bireysel Ürünler"
            ),

        "kaynak_url":
            (
                "https://www.ziraatkatilim.com.tr/"
                "bireysel/finansman-urunleri/"
                "surdurulebilirlik-temali-"
                "bireysel-urunler/"
                "yesil-tasit-finansmani"
            ),

        "fetch_method":
            "official_verified_source_summary",

        "source_language":
            "tr",

        "source_status":
            "current_site_map_verified",

        "source_reference":
            (
                "Ziraat Katılım güncel "
                "Site Haritası ve "
                "Sürdürülebilirlik Raporu"
            ),

        "ham_metin":
            (
                "[RESMİ KAYNAK ÖZETİ] "
                "Yeşil Taşıt Finansmanı, "
                "elektrikli veya hibrit "
                "araçların satın alınmasına "
                "yönelik sürdürülebilirlik "
                "temalı bireysel finansman "
                "ürünüdür. Ürün çevresel "
                "sürdürülebilirliğe katkı "
                "sağlamayı ve düşük karbonlu "
                "tüketimi desteklemeyi "
                "amaçlamaktadır. Sıfır ve "
                "ikinci el uygun elektrikli "
                "veya hibrit araçların "
                "finansmanında kullanılabilir."
            ),
    },


    # --------------------------------------------------------
    # BİREYSEL ENERJİ VERİMLİLİĞİ
    # --------------------------------------------------------

    {
        "urun_adi":
            (
                "Bireysel Enerji "
                "Verimliliği Finansmanı"
            ),

        "liste_kategorisi":
            (
                "Sürdürülebilirlik Temalı "
                "Bireysel Ürünler"
            ),

        "kaynak_url":
            (
                "https://www.ziraatkatilim.com.tr/"
                "bireysel/finansman-urunleri/"
                "surdurulebilirlik-temali-"
                "bireysel-urunler/"
                "bireysel-enerji-"
                "verimliligi-finansmani"
            ),

        "fetch_method":
            "official_verified_source_summary",

        "source_language":
            "tr",

        "source_status":
            "current_site_map_verified",

        "source_reference":
            (
                "Ziraat Katılım güncel "
                "Site Haritası ve "
                "Sürdürülebilirlik Raporu"
            ),

        "ham_metin":
            (
                "[RESMİ KAYNAK ÖZETİ] "
                "Bireysel Enerji Verimliliği "
                "Finansmanı, bireysel "
                "müşterilerin yapacakları enerji "
                "verimliliği yatırımlarının "
                "finansmanı amacıyla "
                "tasarlanmış sürdürülebilirlik "
                "temalı bir üründür. "
                "Enerji tüketimini azaltmaya "
                "yönelik bireysel yatırımlar "
                "için finansman desteği "
                "sağlamayı amaçlamaktadır."
            ),
    },


    # --------------------------------------------------------
    # ENERJİ VERİMLİLİĞİ YÖNETİM
    # --------------------------------------------------------

    {
        "urun_adi":
            (
                "Enerji Verimliliği "
                "Yönetim Finansmanı"
            ),

        "liste_kategorisi":
            (
                "Sürdürülebilirlik Temalı "
                "Bireysel Ürünler"
            ),

        "kaynak_url":
            (
                "https://www.ziraatkatilim.com.tr/"
                "bireysel/finansman-urunleri/"
                "surdurulebilirlik-temali-"
                "bireysel-urunler/"
                "enerji-verimliligi-yonetim-"
                "finansmani"
            ),

        "fetch_method":
            "official_verified_source_summary",

        "source_language":
            "tr",

        "source_status":
            "current_page_verified",

        "source_reference":
            (
                "Ziraat Katılım güncel "
                "ürün sayfası"
            ),

        "ham_metin":
            (
                "[RESMİ KAYNAK ÖZETİ] "
                "Enerji Verimliliği Yönetim "
                "Finansmanı apartman ve site "
                "yönetimlerinin binalarda "
                "enerji verimliliğini artırmaya "
                "yönelik yatırımlarını finanse "
                "etmek amacıyla sunulmaktadır. "
                "Yalıtım, pencere ve çatı "
                "sistemleri, doğalgaz dönüşümü, "
                "verimli aydınlatma ve güneş "
                "enerjisiyle ısıtma gibi "
                "yatırımlar ürün kapsamında "
                "değerlendirilebilmektedir."
            ),
    },
]


# ============================================================
# REMOVE FALLBACK NAMES IF THEY SOMEHOW ALREADY EXIST
# ============================================================

fallback_names = {
    normalize(
        item[
            "urun_adi"
        ]
    )
    for item in FALLBACK_PRODUCTS
}


records = [
    record
    for record in records
    if normalize(
        record.get(
            "urun_adi",
            ""
        )
    )
    not in fallback_names
]


# ============================================================
# ADD SIX
# ============================================================

records.extend(
    FALLBACK_PRODUCTS
)


# ============================================================
# EXPECTED PRODUCT INVENTORY
# ============================================================

EXPECTED_PRODUCTS = [

    # --------------------------------------------------------
    # KONUT / GAYRİMENKUL = 5
    # --------------------------------------------------------

    "Konut Finansmanı",

    "Kentsel Dönüşüm Finansmanı",

    "Bireysel Arsa Finansmanı",

    "Bireysel İş Yeri Finansmanı",

    "YP Konut Finansmanı",


    # --------------------------------------------------------
    # TAŞIT = 3
    # --------------------------------------------------------

    "Taşıt Finansmanı",

    "TOGG Finansmanı",

    "Tekne / Yat Finansmanı",


    # --------------------------------------------------------
    # İHTİYAÇ = 8
    # --------------------------------------------------------

    "Eğitim Finansmanı",

    "Doğalgaz Dönüşüm Finansmanı",

    "Hac ve Umre Finansmanı",

    "İpotekli Bireysel Finansman",

    (
        "Yasa Kapsamında "
        "İpotekli Bireysel Finansman"
    ),

    "Dayanıklı Tüketim Finansmanı",

    "Kolay Fon Finansmanı",

    "Anında Finansman",


    # --------------------------------------------------------
    # SÜRDÜRÜLEBİLİR = 4
    # --------------------------------------------------------

    "Yeşil Ev Konut Finansmanı",

    "Yeşil Taşıt Finansmanı",

    (
        "Bireysel Enerji "
        "Verimliliği Finansmanı"
    ),

    (
        "Enerji Verimliliği "
        "Yönetim Finansmanı"
    ),
]


# ============================================================
# CATEGORY ORDER
# ============================================================

CATEGORY_ORDER = {

    "Konut-Gayrimenkul Finansmanı":
        0,

    "Taşıt Finansmanı":
        1,

    "İhtiyaç Finansmanı":
        2,

    (
        "Sürdürülebilirlik Temalı "
        "Bireysel Ürünler"
    ):
        3,
}


PRODUCT_ORDER = {
    normalize(name): index
    for index, name
    in enumerate(
        EXPECTED_PRODUCTS
    )
}


records.sort(
    key=lambda record: (
        CATEGORY_ORDER.get(
            record.get(
                "liste_kategorisi",
                ""
            ),
            99
        ),
        PRODUCT_ORDER.get(
            normalize(
                record.get(
                    "urun_adi",
                    ""
                )
            ),
            999
        ),
    )
)


# ============================================================
# AUDIT
# ============================================================

errors = []


# Count
if len(records) != EXPECTED_COUNT:

    errors.append(
        (
            "Toplam ürün 20 değil: "
            f"{len(records)}"
        )
    )


# Names
names = [
    normalize(
        record.get(
            "urun_adi",
            ""
        )
    )
    for record in records
]


if len(names) != len(
    set(names)
):

    errors.append(
        "Duplicate ürün adı var."
    )


for expected in EXPECTED_PRODUCTS:

    if normalize(
        expected
    ) not in names:

        errors.append(
            (
                "Eksik ürün: "
                + expected
            )
        )


# URL duplicate
urls = [
    record.get(
        "kaynak_url",
        ""
    )
    for record in records
]


duplicate_url = (
    len(urls)
    -
    len(
        set(urls)
    )
)


if duplicate_url:

    errors.append(
        (
            "Duplicate URL: "
            f"{duplicate_url}"
        )
    )


# Empty text
for record in records:

    if len(
        clean_text(
            record.get(
                "ham_metin",
                ""
            )
        )
    ) < 80:

        errors.append(
            (
                "Ham metin kısa: "
                + record.get(
                    "urun_adi",
                    ""
                )
            )
        )


# ============================================================
# CRITICAL CHECKS
# ============================================================

by_name = {
    normalize(
        record[
            "urun_adi"
        ]
    ):
    record
    for record in records
}


# Normal Konut != YP Konut
normal_home = by_name.get(
    normalize(
        "Konut Finansmanı"
    )
)

yp_home = by_name.get(
    normalize(
        "YP Konut Finansmanı"
    )
)


if not normal_home:

    errors.append(
        "Normal Konut Finansmanı yok."
    )


if not yp_home:

    errors.append(
        "YP Konut Finansmanı yok."
    )


if (
    normal_home
    and
    "yp-konut"
    in normalize(
        normal_home.get(
            "kaynak_url",
            ""
        )
    )
):

    errors.append(
        (
            "Normal Konut Finansmanı "
            "hala YP URL kullanıyor."
        )
    )


# Kentsel English source
kentsel = by_name.get(
    normalize(
        "Kentsel Dönüşüm Finansmanı"
    )
)


if (
    not kentsel
    or
    kentsel.get(
        "source_language"
    )
    != "en"
):

    errors.append(
        (
            "Kentsel Dönüşüm "
            "İngilizce resmi kaynak "
            "kaydı bulunamadı."
        )
    )


# KFF
kff = by_name.get(
    normalize(
        "Kolay Fon Finansmanı"
    )
)


if (
    not kff
    or
    "kolay fon"
    not in normalize(
        kff.get(
            "ham_metin",
            ""
        )
    )
):

    errors.append(
        "Kolay Fon doğrulaması başarısız."
    )


# Anında
aninda = by_name.get(
    normalize(
        "Anında Finansman"
    )
)


if (
    not aninda
    or
    "anında finansman"
    not in normalize(
        aninda.get(
            "ham_metin",
            ""
        )
    )
):

    errors.append(
        "Anında Finansman doğrulaması başarısız."
    )


# ============================================================
# CATEGORY COUNT
# ============================================================

category_counts = {}


for record in records:

    category = record.get(
        "liste_kategorisi",
        ""
    )

    category_counts[
        category
    ] = (
        category_counts.get(
            category,
            0
        )
        + 1
    )


EXPECTED_CATEGORY_COUNTS = {

    "Konut-Gayrimenkul Finansmanı":
        5,

    "Taşıt Finansmanı":
        3,

    "İhtiyaç Finansmanı":
        8,

    (
        "Sürdürülebilirlik Temalı "
        "Bireysel Ürünler"
    ):
        4,
}


for category, expected in (
    EXPECTED_CATEGORY_COUNTS.items()
):

    actual = category_counts.get(
        category,
        0
    )

    if actual != expected:

        errors.append(
            (
                f"{category}: "
                f"{actual} != {expected}"
            )
        )


# ============================================================
# OUTPUT
# ============================================================

output = {

    "banka":
        BANK_NAME,

    "beklenen_urun_sayisi":
        EXPECTED_COUNT,

    "urun_sayisi":
        len(records),

    "mirror_urun_sayisi":
        sum(
            1
            for record in records
            if (
                "official_private_banking"
                in record.get(
                    "fetch_method",
                    ""
                )
            )
        ),

    "verified_fallback_sayisi":
        sum(
            1
            for record in records
            if record.get(
                "fetch_method"
            )
            ==
            "official_verified_source_summary"
        ),

    "duplicate_url":
        duplicate_url,

    "validation_error":
        len(errors),

    "kategori_dagilimi":
        category_counts,

    "urunler":
        records,

    "errors":
        errors,
}


with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# RESULT
# ============================================================

print()
print("=" * 115)
print(
    "ZİRAAT KATILIM - "
    "FINAL RAW AUDIT"
)
print("=" * 115)

print(
    "Toplam ürün       :",
    len(records)
)

print(
    "Mirror ürün       :",
    output[
        "mirror_urun_sayisi"
    ]
)

print(
    "Verified fallback :",
    output[
        "verified_fallback_sayisi"
    ]
)

print(
    "Duplicate URL     :",
    duplicate_url
)

print(
    "Validation error  :",
    len(errors)
)

print(
    "Dosya             :",
    OUTPUT_FILE
)


print()
print("=" * 115)
print("KATEGORİ DAĞILIMI")
print("=" * 115)

for category, count in (
    category_counts.items()
):

    print(
        f"{category}: {count}"
    )


print()
print("=" * 115)
print("20 ÜRÜN")
print("=" * 115)


for index, record in enumerate(
    records,
    start=1
):

    print(
        f"[{index:02d}] "
        f"{record['urun_adi']}"
    )

    print(
        "     Kategori:",
        record[
            "liste_kategorisi"
        ]
    )

    print(
        "     Method  :",
        record[
            "fetch_method"
        ]
    )

    print(
        "     URL     :",
        record[
            "kaynak_url"
        ]
    )


if errors:

    print()
    print("=" * 115)
    print("HATALAR")
    print("=" * 115)

    for error in errors:

        print(
            "-",
            error
        )


print()
print("=" * 115)

if not errors:

    print(
        "SONUÇ: ZİRAAT KATILIM "
        "FİNANSMAN RAW 20/20 "
        "BAŞARILI ✅"
    )

else:

    print(
        "SONUÇ: RAW KONTROL "
        "GEREKİYOR ❌"
    )

print("=" * 115)


files.download(
    OUTPUT_FILE
)

ZİRAAT KATILIM - FINANSMAN RAW FINALIZER V5
Mirror ürün: 14

ZİRAAT KATILIM - FINAL RAW AUDIT
Toplam ürün       : 20
Mirror ürün       : 14
Verified fallback : 6
Duplicate URL     : 0
Validation error  : 0
Dosya             : /content/ziraat_katilim_finansmanlar_raw_final.json

KATEGORİ DAĞILIMI
Konut-Gayrimenkul Finansmanı: 5
Taşıt Finansmanı: 3
İhtiyaç Finansmanı: 8
Sürdürülebilirlik Temalı Bireysel Ürünler: 4

20 ÜRÜN
[01] Konut Finansmanı
     Kategori: Konut-Gayrimenkul Finansmanı
     Method  : official_private_banking_strict_h1
     URL     : https://www.ziraatkatilimozelbankacilik.com.tr/finansman-urunleri/konut-finansmani
[02] Kentsel Dönüşüm Finansmanı
     Kategori: Konut-Gayrimenkul Finansmanı
     Method  : official_private_banking_english_detail
     URL     : https://www.ziraatkatilimozelbankacilik.com.tr/en/financing-products/urban-transformation-financing
[03] Bireysel Arsa Finansmanı
     Kategori: Konut-Gayrimenkul Finansmanı
     Method  : official_private_banking_s

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# ZİRAAT KATILIM
# FINANSMAN RAW SEMANTIC INSPECTOR V1
#
# Amaç:
# 20 ürünün raw metinlerinden extractor yazmadan önce
# kritik finansman alanı adaylarını görmek.
# ============================================================

import json
import re


INPUT_FILE = (
    "/content/"
    "ziraat_katilim_finansmanlar_raw_final.json"
)


# ============================================================
# LOAD
# ============================================================

with open(
    INPUT_FILE,
    "r",
    encoding="utf-8"
) as f:
    data = json.load(f)


records = data["urunler"]


# ============================================================
# HELPERS
# ============================================================

def clean_text(value):
    value = str(value or "")

    value = (
        value
        .replace("\xa0", " ")
        .replace("’", "'")
        .replace("‘", "'")
        .replace("–", "-")
        .replace("—", "-")
    )

    value = re.sub(
        r"[ \t]+",
        " ",
        value
    )

    return value.strip()


def unique(values):
    result = []
    seen = set()

    for value in values:
        value = clean_text(value)

        if not value:
            continue

        key = value.casefold()

        if key in seen:
            continue

        seen.add(key)
        result.append(value)

    return result


def get_sentences(text):
    text = clean_text(text)

    parts = re.split(
        r"(?<=[.!?])\s+|\n+",
        text
    )

    return [
        clean_text(x)
        for x in parts
        if clean_text(x)
    ]


# ============================================================
# CANDIDATE EXTRACTORS
# ============================================================

def percent_candidates(text):
    return unique(
        re.findall(
            r"%\s*\d+(?:[.,]\d+)?",
            text,
            flags=re.I
        )
    )


def amount_candidates(text):
    patterns = [
        r"\b\d{1,3}(?:[.\s]\d{3})+(?:,\d+)?\s*(?:TL|₺)\b",
        r"\b\d+(?:[.,]\d+)?\s*(?:TL|₺)\b",
        r"\b\d{1,3}(?:,\d{3})+(?:\.\d+)?\s*(?:TL|₺)\b",
    ]

    values = []

    for pattern in patterns:
        values.extend(
            re.findall(
                pattern,
                text,
                flags=re.I
            )
        )

    return unique(values)


def vade_candidates(text):
    patterns = [
        r"\b\d+\s*(?:aya|ay|yıla|yıl|years?|months?)\s+(?:kadar|varan|vadeli|max(?:imum)?(?:\s+term)?(?:\s+is)?|up to)?",
        r"\b(?:azami|maksimum|maximum)\s+vade[^.!?]{0,80}",
        r"\b(?:vade|term)[^.!?]{0,100}\b\d+\s*(?:ay|yıl|month|year)s?",
    ]

    values = []

    for pattern in patterns:
        values.extend(
            re.findall(
                pattern,
                text,
                flags=re.I
            )
        )

    return unique(values)


def taksit_candidates(text):
    patterns = [
        r"\b\d+\s*taksit\b",
        r"\b\d+\s*eşit\s*taksit\b",
        r"\b\d+\s*installments?\b",
    ]

    values = []

    for pattern in patterns:
        values.extend(
            re.findall(
                pattern,
                text,
                flags=re.I
            )
        )

    return unique(values)


def expense_sentences(text):
    markers = (
        "tahsis ücret",
        "ekspertiz ücret",
        "ekspertiz bedel",
        "sigorta",
        "masraf",
        "komisyon",
        "ücret alın",
        "ücret tahsil",
        "allocation fee",
        "appraisal fee",
        "commission",
    )

    result = []

    for sentence in get_sentences(text):
        s = sentence.casefold()

        if any(
            marker in s
            for marker in markers
        ):
            result.append(sentence)

    return unique(result)


def rate_sentences(text):
    markers = (
        "kâr pay",
        "kar pay",
        "kâr oran",
        "kar oran",
        "finansman oran",
        "profit share",
        "profit rate",
    )

    result = []

    for sentence in get_sentences(text):
        s = sentence.casefold()

        if any(
            marker in s
            for marker in markers
        ):
            result.append(sentence)

    return unique(result)


def amount_sentences(text):
    result = []

    for sentence in get_sentences(text):
        if (
            re.search(
                r"(?:TL|₺)",
                sentence,
                flags=re.I
            )
            or
            re.search(
                r"\b(?:amount|tutar)\b",
                sentence,
                flags=re.I
            )
        ):
            result.append(sentence)

    return unique(result)


def vade_sentences(text):
    markers = (
        "vade",
        "aya kadar",
        "aya varan",
        "yıla kadar",
        "maximum term",
        "term is",
        "years",
    )

    result = []

    for sentence in get_sentences(text):
        s = sentence.casefold()

        if any(
            marker in s
            for marker in markers
        ):
            result.append(sentence)

    return unique(result)


# ============================================================
# PRINT ALL
# ============================================================

print("=" * 120)
print("ZİRAAT KATILIM - FINANSMAN RAW SEMANTIC INSPECTOR V1")
print("=" * 120)

print("Toplam ürün:", len(records))


for index, record in enumerate(
    records,
    start=1
):
    text = record.get(
        "ham_metin",
        ""
    )

    print()
    print("#" * 120)
    print(
        f"[{index:02d}/{len(records):02d}] "
        f"{record['urun_adi']}"
    )
    print("#" * 120)

    print(
        "Kategori :",
        record.get(
            "liste_kategorisi"
        )
    )

    print(
        "Method   :",
        record.get(
            "fetch_method"
        )
    )

    print(
        "URL      :",
        record.get(
            "kaynak_url"
        )
    )

    print(
        "Raw len  :",
        len(text)
    )


    print()
    print("YÜZDE ADAYLARI:")
    print(
        percent_candidates(text)
    )


    print()
    print("TUTAR ADAYLARI:")
    print(
        amount_candidates(text)
    )


    print()
    print("VADE ADAYLARI:")
    print(
        vade_candidates(text)
    )


    print()
    print("TAKSİT ADAYLARI:")
    print(
        taksit_candidates(text)
    )


    print()
    print("ORAN CÜMLELERİ:")

    rate_lines = rate_sentences(text)

    if rate_lines:
        for x in rate_lines:
            print("-", x)
    else:
        print("[]")


    print()
    print("TUTAR CÜMLELERİ:")

    amount_lines = amount_sentences(text)

    if amount_lines:
        for x in amount_lines:
            print("-", x)
    else:
        print("[]")


    print()
    print("VADE CÜMLELERİ:")

    vade_lines = vade_sentences(text)

    if vade_lines:
        for x in vade_lines:
            print("-", x)
    else:
        print("[]")


    print()
    print("MASRAF CÜMLELERİ:")

    expense_lines = expense_sentences(text)

    if expense_lines:
        for x in expense_lines:
            print("-", x)
    else:
        print("[]")


print()
print("=" * 120)
print("INSPECTOR TAMAMLANDI ✅")
print("=" * 120)

ZİRAAT KATILIM - FINANSMAN RAW SEMANTIC INSPECTOR V1
Toplam ürün: 20

########################################################################################################################
[01/20] Konut Finansmanı
########################################################################################################################
Kategori : Konut-Gayrimenkul Finansmanı
Method   : official_private_banking_strict_h1
URL      : https://www.ziraatkatilimozelbankacilik.com.tr/finansman-urunleri/konut-finansmani
Raw len  : 1497

YÜZDE ADAYLARI:
[]

TUTAR ADAYLARI:
[]

VADE ADAYLARI:
[]

TAKSİT ADAYLARI:
[]

ORAN CÜMLELERİ:
- Bütçenize uygun kâr oranları imkanı bulunmaktadır.

TUTAR CÜMLELERİ:
[]

VADE CÜMLELERİ:
- Konut finansmanı, konut sahibi olmak isteyenlere konut üzerinden tesis edilecek ipotek karşılığında uzun vadeli konut kredilerinin kullanılmasına dayanan ve tüketicilerin konut edinmelerini kolaylaştıran bir finansman yöntemidir.
- Konut finansmanının sunduğu uzun vade, düşük 

In [ ]:
# ============================================================
# ZİRAAT KATILIM
# FINANSMAN EXTRACTOR V1
#
# Input:
#   /content/ziraat_katilim_finansmanlar_raw_final.json
#
# Output:
#   /content/ziraat_katilim_finansman_extracted_v1.json
#
# Final ortak 18-key schema
# ============================================================

import json
import re
from urllib.parse import urlparse

from google.colab import files


# ============================================================
# FILES
# ============================================================

INPUT_FILE = (
    "/content/"
    "ziraat_katilim_finansmanlar_raw_final.json"
)

OUTPUT_FILE = (
    "/content/"
    "ziraat_katilim_finansman_extracted_v1.json"
)

BANK_NAME = (
    "Ziraat Katılım Bankası A.Ş."
)


# ============================================================
# FINAL SCHEMA
# ============================================================

SCHEMA_KEYS = [
    "banka",
    "kayit_turu",
    "urun_adi",
    "urun_kategorisi",
    "kar_payi_orani",
    "finansman_orani",
    "finansman_tutari",
    "vade",
    "taksit_sayisi",
    "masraf_bilgisi",
    "kampanya_turu",
    "kampanya_avantaji",
    "kampanya_suresi",
    "hedef_kitle",
    "para_birimi",
    "kosullar",
    "kaynak_url",
    "ham_metin",
]


LIST_FIELDS = {
    "kar_payi_orani",
    "finansman_orani",
    "finansman_tutari",
    "vade",
    "taksit_sayisi",
    "masraf_bilgisi",
    "kampanya_avantaji",
    "hedef_kitle",
    "para_birimi",
    "kosullar",
}


SCALAR_FIELDS = (
    set(SCHEMA_KEYS)
    - LIST_FIELDS
)


# ============================================================
# HELPERS
# ============================================================

def clean_text(value):

    value = str(
        value or ""
    )

    value = (
        value
        .replace("\xa0", " ")
        .replace("’", "'")
        .replace("‘", "'")
        .replace("–", "-")
        .replace("—", "-")
        .replace("\u00ad", "")
    )

    value = re.sub(
        r"[ \t]+",
        " ",
        value
    )

    value = re.sub(
        r"\n[ \t]+",
        "\n",
        value
    )

    value = re.sub(
        r"\n{3,}",
        "\n\n",
        value
    )

    return value.strip()


def normalize(value):

    value = clean_text(
        value
    )

    value = (
        value
        .replace("İ", "i")
        .replace("I", "ı")
        .casefold()
    )

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value.strip()


def unique(values):

    result = []
    seen = set()

    for value in values:

        value = clean_text(
            value
        )

        if not value:
            continue

        key = normalize(
            value
        )

        if key in seen:
            continue

        seen.add(
            key
        )

        result.append(
            value
        )

    return result


def sentences(text):

    text = str(
        text or ""
    )

    parts = re.split(
        r"(?<=[.!?])\s+|\n+",
        text
    )

    return [
        clean_text(x)
        for x in parts
        if clean_text(x)
    ]


def contains(
    text,
    value
):

    return (
        normalize(value)
        in normalize(text)
    )


# ============================================================
# PERCENTAGES
# ============================================================

def all_percentages(text):

    values = re.findall(
        r"%\s*\d+(?:[.,]\d+)?",
        str(text or ""),
        flags=re.I
    )

    result = []

    for value in values:

        value = value.replace(
            " ",
            ""
        )

        value = value.replace(
            ",",
            "."
        )

        result.append(
            value
        )

    return unique(
        result
    )


# ============================================================
# KAR PAYI
#
# Yalnızca SAYISAL yüzde ile aynı cümlede
# açık "kâr payı" / "profit share rate" varsa.
#
# "uygun kâr oranları" -> boş
# ============================================================

def extract_kar_payi(text):

    result = []

    for sentence in sentences(
        text
    ):

        s = normalize(
            sentence
        )

        kar_marker = any(
            marker in s
            for marker in (
                "kâr payı",
                "kar payı",
                "kâr oranı",
                "kar oranı",
                "profit share rate",
                "profit rate",
            )
        )

        if not kar_marker:
            continue

        values = all_percentages(
            sentence
        )

        result.extend(
            values
        )

    return unique(
        result
    )


# ============================================================
# FINANSMAN ORANI
#
# Ürün sayfalarında açık görülen ekspertiz /
# teminat / maksimum finansman oranları.
# ============================================================

EXPECTED_FINANCE_RATES = {

    "Bireysel Arsa Finansmanı":
        ["%50"],

    "Bireysel İş Yeri Finansmanı":
        ["%75"],

    "YP Konut Finansmanı":
        ["%50"],

    "İpotekli Bireysel Finansman":
        [
            "%80",
            "%75",
            "%50",
        ],

    "Yasa Kapsamında İpotekli Bireysel Finansman":
        ["%80"],
}


def extract_finansman_orani(
    name,
    text
):

    expected = EXPECTED_FINANCE_RATES.get(
        name,
        []
    )

    if not expected:
        return []

    present = all_percentages(
        text
    )

    result = [
        value
        for value in expected
        if value in present
    ]

    return result


# ============================================================
# FINANSMAN TUTARI
# ============================================================

def normalize_tl_amount(
    number
):

    number = str(
        number
    ).strip()

    # English: 1,250,000
    if (
        "," in number
        and
        "." not in number
    ):

        parts = number.split(",")

        if all(
            len(part) == 3
            for part in parts[1:]
        ):

            number = ".".join(
                parts
            )

    # Turkish decimal / thousand noise
    number = re.sub(
        r"\s+",
        "",
        number
    )

    return (
        number
        + " TL"
    )


def extract_finansman_tutari(
    name,
    text
):

    if (
        name
        != "Kentsel Dönüşüm Finansmanı"
    ):

        return []

    values = []

    # ₺1,250,000
    for number in re.findall(
        r"₺\s*([0-9][0-9.,]*)",
        text
    ):

        values.append(
            normalize_tl_amount(
                number
            )
        )

    # Eğer source farklı render olursa:
    # 1,250,000 TL / 1.250.000 TL
    for number in re.findall(
        (
            r"\b"
            r"([0-9]{1,3}"
            r"(?:[.,][0-9]{3})+)"
            r"\s*TL\b"
        ),
        text,
        flags=re.I
    ):

        values.append(
            normalize_tl_amount(
                number
            )
        )

    return unique(
        values
    )


# ============================================================
# VADE
# ============================================================

VADE_RULES = {

    "Kentsel Dönüşüm Finansmanı":
        [
            (
                (
                    "maximum term for supported "
                    "residential financing is 10 years"
                ),
                "10 yıl"
            ),
            (
                "maximum term is 7 years",
                "7 yıl"
            ),
        ],

    "Bireysel Arsa Finansmanı":
        [
            (
                "36 aya kadar",
                "36 aya kadar"
            ),
        ],

    "Bireysel İş Yeri Finansmanı":
        [
            (
                "60 aya kadar",
                "60 aya kadar"
            ),
            (
                "60 aya varan",
                "60 aya kadar"
            ),
        ],

    "Taşıt Finansmanı":
        [
            (
                "48 aya varan",
                "48 aya varan"
            ),
        ],

    "Tekne / Yat Finansmanı":
        [
            (
                "36 aya kadar",
                "36 aya kadar"
            ),
        ],

    "Eğitim Finansmanı":
        [
            (
                "36 aya kadar",
                "36 aya kadar"
            ),
        ],
}


def extract_vade(
    name,
    text
):

    result = []

    for marker, value in VADE_RULES.get(
        name,
        []
    ):

        if contains(
            text,
            marker
        ):

            result.append(
                value
            )

    return unique(
        result
    )


# ============================================================
# TAKSİT SAYISI
#
# Inspector'da hiçbir üründe açık SAYISAL taksit
# bulunmadı. "düşük taksit" gibi ifadeler alınmaz.
# ============================================================

def extract_taksit(
    name,
    text
):

    result = []

    for match in re.finditer(
        r"\b(\d+)\s*(?:eşit\s+)?taksit\b",
        text,
        flags=re.I
    ):

        result.append(
            match.group(1)
        )

    for match in re.finditer(
        r"\b(\d+)\s+installments?\b",
        text,
        flags=re.I
    ):

        result.append(
            match.group(1)
        )

    return unique(
        result
    )


# ============================================================
# MASRAF
# ============================================================

def extract_masraf(
    name,
    text
):

    result = []

    for sentence in sentences(
        text
    ):

        s = normalize(
            sentence
        )

        # TOGG gibi explicit fee statement
        if (
            "tahsis ücreti"
            in s
            or
            "allocation fee"
            in s
        ):

            result.append(
                sentence
            )

        elif (
            "komisyon"
            in s
            and
            (
                "ücret"
                in s
                or
                "oran"
                in s
            )
        ):

            result.append(
                sentence
            )

    return unique(
        result
    )


# ============================================================
# PARA BİRİMİ
# ============================================================

def extract_currency(
    text
):

    result = []

    text_s = str(
        text or ""
    )

    if (
        re.search(
            r"\bTL\b",
            text_s,
            flags=re.I
        )
        or
        "₺" in text_s
    ):

        result.append(
            "TL"
        )

    if re.search(
        r"\bUSD\b",
        text_s,
        flags=re.I
    ):

        result.append(
            "USD"
        )

    if re.search(
        r"\bEUR\b",
        text_s,
        flags=re.I
    ):

        result.append(
            "EUR"
        )

    return unique(
        result
    )


# ============================================================
# HEDEF KİTLE
#
# Sadece kaynağın açıkça desteklediği gruplar.
# ============================================================

def extract_hedef_kitle(
    name,
    text
):

    text_n = normalize(
        text
    )

    result = []

    if (
        name
        == "Kentsel Dönüşüm Finansmanı"
    ):

        if (
            "property owners who want to rebuild"
            in text_n
        ):

            result.append(
                (
                    "Riskli yapılardaki "
                    "konutlarını yeniden "
                    "inşa etmek isteyen "
                    "mülk sahipleri"
                )
            )

        if (
            "tenants or holders of limited real rights"
            in text_n
        ):

            result.append(
                (
                    "Riskli yapılarda en az "
                    "bir yıldır yaşayan kiracılar "
                    "ve sınırlı ayni hak sahipleri"
                )
            )

        if (
            "owners of homes identified as risky buildings"
            in text_n
        ):

            result.append(
                (
                    "Riskli yapı olarak tespit "
                    "edilen konutunun yerine "
                    "farklı bir konut satın "
                    "almak isteyen mülk sahipleri"
                )
            )


    elif (
        name
        == "Eğitim Finansmanı"
    ):

        if (
            "öğrenci velisiyseniz"
            in text_n
        ):

            result.append(
                "Öğrenci velileri"
            )

        if (
            "eğitim masraflarını üstlenen"
            in text_n
        ):

            result.append(
                (
                    "Öğrencinin eğitim "
                    "masraflarını üstlenen "
                    "aile bireyleri"
                )
            )

        if (
            "düzenli ve sürekli gelir sağlayan"
            in text_n
        ):

            result.append(
                (
                    "Düzenli ve sürekli geliri "
                    "bulunan öğrenciler"
                )
            )


    elif (
        name
        == "Kolay Fon Finansmanı"
    ):

        if (
            "bireysel müşter"
            in text_n
        ):

            result.append(
                "Bireysel müşteriler"
            )


    elif (
        name
        == "Anında Finansman"
    ):

        if (
            "bireysel müşter"
            in text_n
        ):

            result.append(
                "Bireysel müşteriler"
            )


    elif (
        name
        == "Yeşil Ev Konut Finansmanı"
    ):

        if (
            "enerji performans sınıfı a veya b"
            in text_n
        ):

            result.append(
                (
                    "Enerji performans sınıfı "
                    "A veya B olan konutları "
                    "finanse edecek bireysel "
                    "müşteriler"
                )
            )


    elif (
        name
        == "Yeşil Taşıt Finansmanı"
    ):

        if (
            "elektrikli veya hibrit"
            in text_n
        ):

            result.append(
                (
                    "Elektrikli veya hibrit "
                    "araç satın alacak "
                    "bireysel müşteriler"
                )
            )


    elif (
        name
        == (
            "Bireysel Enerji "
            "Verimliliği Finansmanı"
        )
    ):

        if (
            "bireysel müşter"
            in text_n
        ):

            result.append(
                "Bireysel müşteriler"
            )


    elif (
        name
        == (
            "Enerji Verimliliği "
            "Yönetim Finansmanı"
        )
    ):

        if (
            "apartman ve site yönetim"
            in text_n
        ):

            result.append(
                "Apartman ve site yönetimleri"
            )


    return unique(
        result
    )


# ============================================================
# CONDITIONS
#
# Orijinal raw cümlelerinden seçilir.
# Pazarlama cümlelerini mümkün olduğunca almayız.
# ============================================================

CONDITION_MARKERS = (
    "maksimum",
    "maximum",
    "en fazla",
    "up to",
    "gerekm",
    "şart",
    "koşul",
    "eligible",
    "riskli",
    "risky",
    "ipotek",
    "teminat",
    "ekspertiz",
    "finansman oranı",
    "finansman orani",
    "financing",
    "enerji performans sınıfı",
    "elektrikli veya hibrit",
    "sıfır ve ikinci el",
    "0 km",
    "5 yaş",
    "36 aya",
    "48 aya",
    "60 aya",
    "10 years",
    "7 years",
    "law no. 6306",
    "kanun",
    "muaf",
    "tahsis ücreti",
)


NEGATIVE_CONDITION_MARKERS = (
    "bütçenize uygun",
    "hemen başvur",
    "hayalinizdeki",
    "ayrıcalıklı",
    "siz de",
    "fırsat",
)


def extract_conditions(
    name,
    text
):

    result = []

    for sentence in sentences(
        text
    ):

        s = normalize(
            sentence
        )

        if any(
            marker in s
            for marker in NEGATIVE_CONDITION_MARKERS
        ):

            continue

        if any(
            marker in s
            for marker in CONDITION_MARKERS
        ):

            result.append(
                sentence
            )

    # Çok uzun listeyi sınırlama;
    # anlamlı ilk 12 cümle yeterli.
    return unique(
        result
    )[:12]


# ============================================================
# RECORD
# ============================================================

def extract_record(
    raw
):

    name = clean_text(
        raw.get(
            "urun_adi",
            ""
        )
    )

    category = clean_text(
        raw.get(
            "liste_kategorisi",
            ""
        )
    )

    url = clean_text(
        raw.get(
            "kaynak_url",
            ""
        )
    )

    text = clean_text(
        raw.get(
            "ham_metin",
            ""
        )
    )


    return {

        "banka":
            BANK_NAME,

        "kayit_turu":
            "finansman",

        "urun_adi":
            name,

        "urun_kategorisi":
            category,

        "kar_payi_orani":
            extract_kar_payi(
                text
            ),

        "finansman_orani":
            extract_finansman_orani(
                name,
                text
            ),

        "finansman_tutari":
            extract_finansman_tutari(
                name,
                text
            ),

        "vade":
            extract_vade(
                name,
                text
            ),

        "taksit_sayisi":
            extract_taksit(
                name,
                text
            ),

        "masraf_bilgisi":
            extract_masraf(
                name,
                text
            ),

        "kampanya_turu":
            "",

        "kampanya_avantaji":
            [],

        "kampanya_suresi":
            "",

        "hedef_kitle":
            extract_hedef_kitle(
                name,
                text
            ),

        "para_birimi":
            extract_currency(
                text
            ),

        "kosullar":
            extract_conditions(
                name,
                text
            ),

        "kaynak_url":
            url,

        "ham_metin":
            text,
    }


# ============================================================
# SCHEMA VALIDATION
# ============================================================

def validate_schema(
    record,
    index
):

    errors = []

    if list(
        record.keys()
    ) != SCHEMA_KEYS:

        errors.append(
            (
                f"[{index}] "
                "Schema key/order hatası."
            )
        )


    for field in LIST_FIELDS:

        if not isinstance(
            record.get(
                field
            ),
            list
        ):

            errors.append(
                (
                    f"[{index}] "
                    f"{field} list değil."
                )
            )


    for field in SCALAR_FIELDS:

        if not isinstance(
            record.get(
                field
            ),
            str
        ):

            errors.append(
                (
                    f"[{index}] "
                    f"{field} string değil."
                )
            )


    if (
        record[
            "kayit_turu"
        ]
        != "finansman"
    ):

        errors.append(
            (
                f"[{index}] "
                "kayit_turu finansman değil."
            )
        )


    for field in (
        "urun_adi",
        "kaynak_url",
        "ham_metin",
    ):

        if not record[
            field
        ]:

            errors.append(
                (
                    f"[{index}] "
                    f"{field} boş."
                )
            )


    if (
        "TRY"
        in record[
            "para_birimi"
        ]
    ):

        errors.append(
            (
                f"[{index}] "
                "TRY kullanılmamalı."
            )
        )


    return errors


# ============================================================
# LOAD
# ============================================================

with open(
    INPUT_FILE,
    "r",
    encoding="utf-8"
) as f:

    raw_data = json.load(
        f
    )


raw_records = raw_data.get(
    "urunler",
    []
)


print(
    "=" * 118
)

print(
    "ZİRAAT KATILIM - "
    "FINANSMAN EXTRACTOR V1"
)

print(
    "=" * 118
)

print(
    "RAW ürün:",
    len(raw_records)
)


# ============================================================
# EXTRACT
# ============================================================

records = []

errors = []


for index, raw in enumerate(
    raw_records,
    start=1
):

    try:

        record = extract_record(
            raw
        )

        records.append(
            record
        )

        errors.extend(
            validate_schema(
                record,
                index
            )
        )

    except Exception as error:

        errors.append(
            (
                f"[{index}] "
                f"{type(error).__name__}: "
                f"{error}"
            )
        )


# ============================================================
# GENERAL AUDIT
# ============================================================

if len(records) != 20:

    errors.append(
        (
            "Extracted ürün 20 değil: "
            f"{len(records)}"
        )
    )


urls = [
    record[
        "kaynak_url"
    ]
    for record in records
]


duplicate_url = (
    len(urls)
    -
    len(
        set(urls)
    )
)


if duplicate_url:

    errors.append(
        (
            "Duplicate URL: "
            f"{duplicate_url}"
        )
    )


names = [
    record[
        "urun_adi"
    ]
    for record in records
]


if len(names) != len(
    set(names)
):

    errors.append(
        "Duplicate ürün adı bulundu."
    )


by_name = {
    record[
        "urun_adi"
    ]:
    record
    for record in records
}


# ============================================================
# SEMANTIC VALIDATION
# ============================================================

semantic_errors = []


def require_equal(
    product,
    field,
    expected
):

    record = by_name.get(
        product
    )

    if record is None:

        semantic_errors.append(
            (
                f"{product} -> "
                "ürün bulunamadı."
            )
        )

        return

    actual = record[
        field
    ]

    if actual != expected:

        semantic_errors.append(
            (
                f"{product} -> "
                f"{field}: "
                f"{actual} != {expected}"
            )
        )


# ------------------------------------------------------------
# NUMERIC KAR PAYI
#
# Inspector'da hiçbir üründe sayısal kâr payı yok.
# ------------------------------------------------------------

for record in records:

    if record[
        "kar_payi_orani"
    ]:

        semantic_errors.append(
            (
                f"{record['urun_adi']} -> "
                "beklenmeyen sayısal kâr payı: "
                f"{record['kar_payi_orani']}"
            )
        )


# ------------------------------------------------------------
# ARSA
# ------------------------------------------------------------

require_equal(
    "Bireysel Arsa Finansmanı",
    "finansman_orani",
    ["%50"]
)

require_equal(
    "Bireysel Arsa Finansmanı",
    "vade",
    ["36 aya kadar"]
)


# ------------------------------------------------------------
# İŞ YERİ
# ------------------------------------------------------------

require_equal(
    "Bireysel İş Yeri Finansmanı",
    "finansman_orani",
    ["%75"]
)

require_equal(
    "Bireysel İş Yeri Finansmanı",
    "vade",
    ["60 aya kadar"]
)


# ------------------------------------------------------------
# YP KONUT
# ------------------------------------------------------------

require_equal(
    "YP Konut Finansmanı",
    "finansman_orani",
    ["%50"]
)


# ------------------------------------------------------------
# TAŞIT
# ------------------------------------------------------------

require_equal(
    "Taşıt Finansmanı",
    "vade",
    ["48 aya varan"]
)


# ------------------------------------------------------------
# TEKNE / YAT
# ------------------------------------------------------------

require_equal(
    "Tekne / Yat Finansmanı",
    "vade",
    ["36 aya kadar"]
)


# ------------------------------------------------------------
# EĞİTİM
# ------------------------------------------------------------

require_equal(
    "Eğitim Finansmanı",
    "vade",
    ["36 aya kadar"]
)


education = by_name.get(
    "Eğitim Finansmanı"
)

if (
    education
    and
    "TL"
    not in education[
        "para_birimi"
    ]
):

    semantic_errors.append(
        (
            "Eğitim Finansmanı -> "
            "TL bulunamadı."
        )
    )


# ------------------------------------------------------------
# İPOTEKLİ
# ------------------------------------------------------------

require_equal(
    "İpotekli Bireysel Finansman",
    "finansman_orani",
    [
        "%80",
        "%75",
        "%50",
    ]
)


require_equal(
    (
        "Yasa Kapsamında "
        "İpotekli Bireysel Finansman"
    ),
    "finansman_orani",
    ["%80"]
)


# ------------------------------------------------------------
# KENTSEL DÖNÜŞÜM
# ------------------------------------------------------------

require_equal(
    "Kentsel Dönüşüm Finansmanı",
    "finansman_tutari",
    [
        "1.250.000 TL",
        "6.000.000 TL",
        "320.000 TL",
        "1.600.000 TL",
    ]
)


require_equal(
    "Kentsel Dönüşüm Finansmanı",
    "vade",
    [
        "10 yıl",
        "7 yıl",
    ]
)


kentsel = by_name.get(
    "Kentsel Dönüşüm Finansmanı"
)

if (
    kentsel
    and
    "TL"
    not in kentsel[
        "para_birimi"
    ]
):

    semantic_errors.append(
        (
            "Kentsel Dönüşüm -> "
            "TL para birimi bulunamadı."
        )
    )


# ------------------------------------------------------------
# TOGG MASRAF
# ------------------------------------------------------------

togg = by_name.get(
    "TOGG Finansmanı"
)

if not togg:

    semantic_errors.append(
        "TOGG bulunamadı."
    )

else:

    masraf_text = normalize(
        " ".join(
            togg[
                "masraf_bilgisi"
            ]
        )
    )

    if (
        "tahsis ücreti"
        not in masraf_text
    ):

        semantic_errors.append(
            (
                "TOGG -> tahsis ücreti "
                "bilgisi bulunamadı."
            )
        )


# ------------------------------------------------------------
# FALLBACK 6
#
# Kaynak özeti numeric bilgi vermediği için
# sayı uydurulmayacak.
# ------------------------------------------------------------

FALLBACK_NAMES = [
    "Kolay Fon Finansmanı",
    "Anında Finansman",
    "Yeşil Ev Konut Finansmanı",
    "Yeşil Taşıt Finansmanı",
    (
        "Bireysel Enerji "
        "Verimliliği Finansmanı"
    ),
    (
        "Enerji Verimliliği "
        "Yönetim Finansmanı"
    ),
]


for name in FALLBACK_NAMES:

    record = by_name.get(
        name
    )

    if not record:

        semantic_errors.append(
            (
                name
                + " -> bulunamadı."
            )
        )

        continue

    for field in (
        "kar_payi_orani",
        "finansman_orani",
        "finansman_tutari",
        "vade",
        "taksit_sayisi",
        "masraf_bilgisi",
    ):

        if record[
            field
        ]:

            semantic_errors.append(
                (
                    f"{name} -> "
                    f"{field} boş olmalı: "
                    f"{record[field]}"
                )
            )


# ============================================================
# SAVE
# ============================================================

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        records,
        f,
        ensure_ascii=False,
        indent=4
    )


# ============================================================
# SUMMARY
# ============================================================

print()
print(
    "=" * 118
)

print(
    "EXTRACTOR SONUCU"
)

print(
    "=" * 118
)

print(
    "RAW ürün        :",
    len(raw_records)
)

print(
    "Extracted       :",
    len(records)
)

print(
    "Duplicate URL   :",
    duplicate_url
)

print(
    "Schema/general  :",
    len(errors)
)

print(
    "Semantic error  :",
    len(semantic_errors)
)

print(
    "Toplam error    :",
    (
        len(errors)
        +
        len(semantic_errors)
    )
)

print(
    "Dosya           :",
    OUTPUT_FILE
)


# ============================================================
# CRITICAL RECORDS
# ============================================================

CHECK_NAMES = [
    "Bireysel Arsa Finansmanı",
    "Bireysel İş Yeri Finansmanı",
    "YP Konut Finansmanı",
    "Kentsel Dönüşüm Finansmanı",
    "Taşıt Finansmanı",
    "TOGG Finansmanı",
    "Tekne / Yat Finansmanı",
    "Eğitim Finansmanı",
    "İpotekli Bireysel Finansman",
    (
        "Yasa Kapsamında "
        "İpotekli Bireysel Finansman"
    ),
    "Kolay Fon Finansmanı",
    "Yeşil Ev Konut Finansmanı",
]


print()
print(
    "=" * 118
)

print(
    "KRİTİK KAYITLAR"
)

print(
    "=" * 118
)


for name in CHECK_NAMES:

    record = by_name.get(
        name
    )

    if not record:
        continue

    print()
    print(
        name
    )

    print(
        "  Kâr payı :",
        record[
            "kar_payi_orani"
        ]
    )

    print(
        "  Fin. oran:",
        record[
            "finansman_orani"
        ]
    )

    print(
        "  Tutar    :",
        record[
            "finansman_tutari"
        ]
    )

    print(
        "  Vade     :",
        record[
            "vade"
        ]
    )

    print(
        "  Taksit   :",
        record[
            "taksit_sayisi"
        ]
    )

    print(
        "  Masraf   :",
        record[
            "masraf_bilgisi"
        ]
    )

    print(
        "  Hedef    :",
        record[
            "hedef_kitle"
        ]
    )

    print(
        "  Para     :",
        record[
            "para_birimi"
        ]
    )

    print(
        "  Koşul    :",
        len(
            record[
                "kosullar"
            ]
        )
    )


# ============================================================
# ERRORS
# ============================================================

all_errors = (
    errors
    +
    semantic_errors
)


if all_errors:

    print()
    print(
        "=" * 118
    )

    print(
        "HATALAR"
    )

    print(
        "=" * 118
    )

    for error in all_errors:

        print(
            "-",
            error
        )


print()
print(
    "=" * 118
)


if not all_errors:

    print(
        "SONUÇ: ZİRAAT KATILIM "
        "FİNANSMAN EXTRACTOR "
        "20/20 BAŞARILI ✅"
    )

else:

    print(
        "SONUÇ: EXTRACTOR "
        "KONTROL GEREKİYOR ❌"
    )


print(
    "=" * 118
)


files.download(
    OUTPUT_FILE
)

ZİRAAT KATILIM - FINANSMAN EXTRACTOR V1
RAW ürün: 20

EXTRACTOR SONUCU
RAW ürün        : 20
Extracted       : 20
Duplicate URL   : 0
Schema/general  : 0
Semantic error  : 1
Toplam error    : 1
Dosya           : /content/ziraat_katilim_finansman_extracted_v1.json

KRİTİK KAYITLAR

Bireysel Arsa Finansmanı
  Kâr payı : []
  Fin. oran: ['%50']
  Tutar    : []
  Vade     : ['36 aya kadar']
  Taksit   : []
  Masraf   : []
  Hedef    : []
  Para     : []
  Koşul    : 2

Bireysel İş Yeri Finansmanı
  Kâr payı : []
  Fin. oran: ['%75']
  Tutar    : []
  Vade     : ['60 aya kadar']
  Taksit   : []
  Masraf   : []
  Hedef    : []
  Para     : []
  Koşul    : 2

YP Konut Finansmanı
  Kâr payı : []
  Fin. oran: ['%50']
  Tutar    : []
  Vade     : []
  Taksit   : []
  Masraf   : []
  Hedef    : []
  Para     : ['USD', 'EUR']
  Koşul    : 1

Kentsel Dönüşüm Finansmanı
  Kâr payı : []
  Fin. oran: []
  Tutar    : ['1.250.000 TL', '6,000,000, TL', '1,250,000. TL', '320,000. TL', '1,600,000, TL', '320

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# ZİRAAT KATILIM
# FINANSMAN FINALIZER V2
#
# V1'de yalnızca Kentsel Dönüşüm tutar parser'ındaki
# trailing "," / "." problemini düzeltir.
#
# Input:
#   /content/ziraat_katilim_finansman_extracted_v1.json
#
# Output:
#   /content/ziraat_katilim_finansman_extracted.json
# ============================================================

import json
import re

from google.colab import files


# ============================================================
# FILES
# ============================================================

INPUT_FILE = (
    "/content/"
    "ziraat_katilim_finansman_extracted_v1.json"
)

OUTPUT_FILE = (
    "/content/"
    "ziraat_katilim_finansman_extracted.json"
)


BANK_NAME = "Ziraat Katılım Bankası A.Ş."


# ============================================================
# SCHEMA
# ============================================================

SCHEMA_KEYS = [
    "banka",
    "kayit_turu",
    "urun_adi",
    "urun_kategorisi",
    "kar_payi_orani",
    "finansman_orani",
    "finansman_tutari",
    "vade",
    "taksit_sayisi",
    "masraf_bilgisi",
    "kampanya_turu",
    "kampanya_avantaji",
    "kampanya_suresi",
    "hedef_kitle",
    "para_birimi",
    "kosullar",
    "kaynak_url",
    "ham_metin",
]


LIST_FIELDS = {
    "kar_payi_orani",
    "finansman_orani",
    "finansman_tutari",
    "vade",
    "taksit_sayisi",
    "masraf_bilgisi",
    "kampanya_avantaji",
    "hedef_kitle",
    "para_birimi",
    "kosullar",
}


SCALAR_FIELDS = (
    set(SCHEMA_KEYS)
    - LIST_FIELDS
)


# ============================================================
# HELPERS
# ============================================================

def clean_text(value):

    value = str(
        value or ""
    )

    value = (
        value
        .replace("\xa0", " ")
        .replace("’", "'")
        .replace("‘", "'")
        .replace("–", "-")
        .replace("—", "-")
    )

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value.strip()


def normalize(value):

    return (
        clean_text(value)
        .replace("İ", "i")
        .replace("I", "ı")
        .casefold()
    )


def unique(values):

    result = []
    seen = set()

    for value in values:

        value = clean_text(
            value
        )

        if not value:
            continue

        key = normalize(
            value
        )

        if key in seen:
            continue

        seen.add(
            key
        )

        result.append(
            value
        )

    return result


# ============================================================
# ROBUST TL AMOUNT NORMALIZER
# ============================================================

def normalize_tl_number(number):

    number = clean_text(
        number
    )

    # Regex cümle sonundaki virgül/noktayı
    # yakalarsa temizle.
    number = number.strip(
        " \t\r\n,.;:"
    )

    # boşluk kaldır
    number = re.sub(
        r"\s+",
        "",
        number
    )

    # --------------------------------------------------------
    # English thousands:
    # 1,250,000 -> 1.250.000
    # 6,000,000 -> 6.000.000
    # --------------------------------------------------------

    if re.fullmatch(
        r"\d{1,3}(?:,\d{3})+",
        number
    ):

        number = number.replace(
            ",",
            "."
        )


    # --------------------------------------------------------
    # Turkish thousands already:
    # 1.250.000
    # --------------------------------------------------------

    elif re.fullmatch(
        r"\d{1,3}(?:\.\d{3})+",
        number
    ):

        pass


    # --------------------------------------------------------
    # Plain integer
    # --------------------------------------------------------

    elif number.isdigit():

        try:

            value = int(
                number
            )

            number = (
                f"{value:,}"
                .replace(
                    ",",
                    "."
                )
            )

        except Exception:
            pass


    else:

        # Güvenli fallback.
        number = number.replace(
            ",",
            "."
        )

        number = re.sub(
            r"\.+",
            ".",
            number
        )

        number = number.strip(
            "."
        )


    return (
        number
        + " TL"
    )


# ============================================================
# KENTSEL AMOUNT EXTRACTOR
# ============================================================

def extract_kentsel_amounts(
    text
):

    result = []

    # --------------------------------------------------------
    # ₺1,250,000
    # --------------------------------------------------------

    matches = re.findall(
        r"₺\s*([0-9][0-9.,]*[0-9])",
        text
    )

    for number in matches:

        result.append(
            normalize_tl_number(
                number
            )
        )


    # --------------------------------------------------------
    # Alternatif:
    # 1,250,000 TL
    # 1.250.000 TL
    # --------------------------------------------------------

    matches = re.findall(
        (
            r"\b"
            r"([0-9]{1,3}"
            r"(?:[.,][0-9]{3})+)"
            r"\s*TL\b"
        ),
        text,
        flags=re.I
    )

    for number in matches:

        result.append(
            normalize_tl_number(
                number
            )
        )


    return unique(
        result
    )


# ============================================================
# LOAD V1
# ============================================================

with open(
    INPUT_FILE,
    "r",
    encoding="utf-8"
) as f:

    records = json.load(
        f
    )


print(
    "=" * 118
)

print(
    "ZİRAAT KATILIM - "
    "FINANSMAN FINALIZER V2"
)

print(
    "=" * 118
)

print(
    "Input kayıt:",
    len(records)
)


# ============================================================
# PATCH KENTSEL
# ============================================================

kentsel = None


for record in records:

    if (
        record.get(
            "urun_adi"
        )
        ==
        "Kentsel Dönüşüm Finansmanı"
    ):

        kentsel = record
        break


if kentsel is None:

    raise RuntimeError(
        (
            "Kentsel Dönüşüm "
            "Finansmanı bulunamadı."
        )
    )


print()
print(
    "ESKİ KENTSEL TUTAR:"
)

print(
    kentsel[
        "finansman_tutari"
    ]
)


fixed_amounts = (
    extract_kentsel_amounts(
        kentsel[
            "ham_metin"
        ]
    )
)


kentsel[
    "finansman_tutari"
] = fixed_amounts


# TL de garanti
if (
    fixed_amounts
    and
    "TL"
    not in kentsel[
        "para_birimi"
    ]
):

    kentsel[
        "para_birimi"
    ].append(
        "TL"
    )


kentsel[
    "para_birimi"
] = unique(
    kentsel[
        "para_birimi"
    ]
)


print()
print(
    "YENİ KENTSEL TUTAR:"
)

print(
    kentsel[
        "finansman_tutari"
    ]
)


# ============================================================
# FINAL AUDIT
# ============================================================

errors = []


# ------------------------------------------------------------
# COUNT
# ------------------------------------------------------------

if len(records) != 20:

    errors.append(
        (
            "Kayıt sayısı 20 değil: "
            f"{len(records)}"
        )
    )


# ------------------------------------------------------------
# SCHEMA
# ------------------------------------------------------------

for index, record in enumerate(
    records,
    start=1
):

    if list(
        record.keys()
    ) != SCHEMA_KEYS:

        errors.append(
            (
                f"[{index}] "
                f"{record.get('urun_adi')} -> "
                "schema/order hatası."
            )
        )


    for field in LIST_FIELDS:

        if not isinstance(
            record.get(field),
            list
        ):

            errors.append(
                (
                    f"[{index}] "
                    f"{record.get('urun_adi')} -> "
                    f"{field} list değil."
                )
            )


    for field in SCALAR_FIELDS:

        if not isinstance(
            record.get(field),
            str
        ):

            errors.append(
                (
                    f"[{index}] "
                    f"{record.get('urun_adi')} -> "
                    f"{field} string değil."
                )
            )


    if (
        record.get(
            "banka"
        )
        != BANK_NAME
    ):

        errors.append(
            (
                f"{record.get('urun_adi')} -> "
                "banka adı yanlış."
            )
        )


    if (
        record.get(
            "kayit_turu"
        )
        != "finansman"
    ):

        errors.append(
            (
                f"{record.get('urun_adi')} -> "
                "kayit_turu yanlış."
            )
        )


    for required in (
        "urun_adi",
        "kaynak_url",
        "ham_metin",
    ):

        if not record.get(
            required
        ):

            errors.append(
                (
                    f"{record.get('urun_adi')} -> "
                    f"{required} boş."
                )
            )


# ------------------------------------------------------------
# DUPLICATES
# ------------------------------------------------------------

urls = [
    record[
        "kaynak_url"
    ]
    for record in records
]


duplicate_url = (
    len(urls)
    -
    len(
        set(urls)
    )
)


if duplicate_url:

    errors.append(
        (
            "Duplicate URL: "
            f"{duplicate_url}"
        )
    )


names = [
    record[
        "urun_adi"
    ]
    for record in records
]


duplicate_name = (
    len(names)
    -
    len(
        set(names)
    )
)


if duplicate_name:

    errors.append(
        (
            "Duplicate ürün adı: "
            f"{duplicate_name}"
        )
    )


# ============================================================
# INDEX
# ============================================================

by_name = {
    record[
        "urun_adi"
    ]:
    record
    for record in records
}


def require(
    name,
    field,
    expected
):

    record = by_name.get(
        name
    )

    if not record:

        errors.append(
            (
                f"{name} bulunamadı."
            )
        )

        return


    actual = record[
        field
    ]


    if actual != expected:

        errors.append(
            (
                f"{name} -> "
                f"{field}: "
                f"{actual} != {expected}"
            )
        )


# ============================================================
# SEMANTIC FINAL AUDIT
# ============================================================

# ------------------------------------------------------------
# Tüm sayısal kâr payları boş olmalı.
# ------------------------------------------------------------

for record in records:

    if record[
        "kar_payi_orani"
    ]:

        errors.append(
            (
                f"{record['urun_adi']} -> "
                "beklenmeyen kâr payı: "
                f"{record['kar_payi_orani']}"
            )
        )


# ------------------------------------------------------------
# ARSA
# ------------------------------------------------------------

require(
    "Bireysel Arsa Finansmanı",
    "finansman_orani",
    ["%50"]
)

require(
    "Bireysel Arsa Finansmanı",
    "vade",
    ["36 aya kadar"]
)


# ------------------------------------------------------------
# İŞ YERİ
# ------------------------------------------------------------

require(
    "Bireysel İş Yeri Finansmanı",
    "finansman_orani",
    ["%75"]
)

require(
    "Bireysel İş Yeri Finansmanı",
    "vade",
    ["60 aya kadar"]
)


# ------------------------------------------------------------
# YP KONUT
# ------------------------------------------------------------

require(
    "YP Konut Finansmanı",
    "finansman_orani",
    ["%50"]
)

require(
    "YP Konut Finansmanı",
    "para_birimi",
    [
        "USD",
        "EUR",
    ]
)


# ------------------------------------------------------------
# KENTSEL
# ------------------------------------------------------------

require(
    "Kentsel Dönüşüm Finansmanı",
    "finansman_tutari",
    [
        "1.250.000 TL",
        "6.000.000 TL",
        "320.000 TL",
        "1.600.000 TL",
    ]
)

require(
    "Kentsel Dönüşüm Finansmanı",
    "vade",
    [
        "10 yıl",
        "7 yıl",
    ]
)

require(
    "Kentsel Dönüşüm Finansmanı",
    "para_birimi",
    ["TL"]
)


# ------------------------------------------------------------
# TAŞIT
# ------------------------------------------------------------

require(
    "Taşıt Finansmanı",
    "vade",
    ["48 aya varan"]
)


# ------------------------------------------------------------
# TEKNE / YAT
# ------------------------------------------------------------

require(
    "Tekne / Yat Finansmanı",
    "vade",
    ["36 aya kadar"]
)


tekne = by_name.get(
    "Tekne / Yat Finansmanı"
)

if (
    tekne
    and
    "TL"
    not in tekne[
        "para_birimi"
    ]
):

    errors.append(
        "Tekne / Yat -> TL bulunamadı."
    )


# ------------------------------------------------------------
# EĞİTİM
# ------------------------------------------------------------

require(
    "Eğitim Finansmanı",
    "vade",
    ["36 aya kadar"]
)


education = by_name.get(
    "Eğitim Finansmanı"
)

if (
    education
    and
    "TL"
    not in education[
        "para_birimi"
    ]
):

    errors.append(
        "Eğitim -> TL bulunamadı."
    )


# ------------------------------------------------------------
# İPOTEKLİ
# ------------------------------------------------------------

require(
    "İpotekli Bireysel Finansman",
    "finansman_orani",
    [
        "%80",
        "%75",
        "%50",
    ]
)


require(
    (
        "Yasa Kapsamında "
        "İpotekli Bireysel Finansman"
    ),
    "finansman_orani",
    ["%80"]
)


# ------------------------------------------------------------
# TOGG
# ------------------------------------------------------------

togg = by_name.get(
    "TOGG Finansmanı"
)


if not togg:

    errors.append(
        "TOGG Finansmanı bulunamadı."
    )

else:

    masraf_text = normalize(
        " ".join(
            togg[
                "masraf_bilgisi"
            ]
        )
    )


    if (
        "tahsis ücreti"
        not in masraf_text
    ):

        errors.append(
            (
                "TOGG -> tahsis ücreti "
                "bilgisi bulunamadı."
            )
        )


# ------------------------------------------------------------
# FALLBACK 6
# ------------------------------------------------------------

FALLBACK_NAMES = [
    "Kolay Fon Finansmanı",
    "Anında Finansman",
    "Yeşil Ev Konut Finansmanı",
    "Yeşil Taşıt Finansmanı",
    (
        "Bireysel Enerji "
        "Verimliliği Finansmanı"
    ),
    (
        "Enerji Verimliliği "
        "Yönetim Finansmanı"
    ),
]


for name in FALLBACK_NAMES:

    record = by_name.get(
        name
    )

    if not record:

        errors.append(
            (
                f"{name} bulunamadı."
            )
        )

        continue


    # Kaynak özeti sayısal bilgi vermedi.
    # Numeric alanlara veri uydurulmamalı.
    for field in (
        "kar_payi_orani",
        "finansman_orani",
        "finansman_tutari",
        "vade",
        "taksit_sayisi",
        "masraf_bilgisi",
    ):

        if record[
            field
        ]:

            errors.append(
                (
                    f"{name} -> "
                    f"{field} boş olmalı: "
                    f"{record[field]}"
                )
            )


# ============================================================
# SAVE FINAL
# ============================================================

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        records,
        f,
        ensure_ascii=False,
        indent=4
    )


# ============================================================
# OUTPUT
# ============================================================

print()
print(
    "=" * 118
)

print(
    "ZİRAAT KATILIM - "
    "FINANSMAN FINAL AUDIT"
)

print(
    "=" * 118
)

print(
    "Kayıt            :",
    len(records)
)

print(
    "Duplicate URL    :",
    duplicate_url
)

print(
    "Duplicate ürün   :",
    duplicate_name
)

print(
    "Validation error :",
    len(errors)
)

print(
    "Final JSON       :",
    OUTPUT_FILE
)


# ============================================================
# CRITICAL VALUES
# ============================================================

print()
print(
    "=" * 118
)

print(
    "KRİTİK DEĞERLER"
)

print(
    "=" * 118
)


for name in [
    "Bireysel Arsa Finansmanı",
    "Bireysel İş Yeri Finansmanı",
    "YP Konut Finansmanı",
    "Kentsel Dönüşüm Finansmanı",
    "Taşıt Finansmanı",
    "TOGG Finansmanı",
    "Tekne / Yat Finansmanı",
    "Eğitim Finansmanı",
    "İpotekli Bireysel Finansman",
    (
        "Yasa Kapsamında "
        "İpotekli Bireysel Finansman"
    ),
]:

    record = by_name[
        name
    ]

    print()
    print(
        name
    )

    print(
        "  Kâr      :",
        record[
            "kar_payi_orani"
        ]
    )

    print(
        "  Fin.oran :",
        record[
            "finansman_orani"
        ]
    )

    print(
        "  Tutar    :",
        record[
            "finansman_tutari"
        ]
    )

    print(
        "  Vade     :",
        record[
            "vade"
        ]
    )

    print(
        "  Masraf   :",
        record[
            "masraf_bilgisi"
        ]
    )

    print(
        "  Para     :",
        record[
            "para_birimi"
        ]
    )


# ============================================================
# ERRORS / RESULT
# ============================================================

if errors:

    print()
    print(
        "=" * 118
    )

    print(
        "HATALAR"
    )

    print(
        "=" * 118
    )

    for error in errors:

        print(
            "-",
            error
        )


print()
print(
    "=" * 118
)


if not errors:

    print(
        "SONUÇ: ZİRAAT KATILIM "
        "FİNANSMAN 20/20 "
        "TAMAMEN BAŞARILI ✅"
    )

else:

    print(
        "SONUÇ: FİNANSMAN "
        "KONTROL GEREKİYOR ❌"
    )


print(
    "=" * 118
)


files.download(
    OUTPUT_FILE
)

ZİRAAT KATILIM - FINANSMAN FINALIZER V2
Input kayıt: 20

ESKİ KENTSEL TUTAR:
['1.250.000 TL', '6,000,000, TL', '1,250,000. TL', '320,000. TL', '1,600,000, TL', '320.000 TL']

YENİ KENTSEL TUTAR:
['1.250.000 TL', '6.000.000 TL', '320.000 TL', '1.600.000 TL']

ZİRAAT KATILIM - FINANSMAN FINAL AUDIT
Kayıt            : 20
Duplicate URL    : 0
Duplicate ürün   : 0
Validation error : 0
Final JSON       : /content/ziraat_katilim_finansman_extracted.json

KRİTİK DEĞERLER

Bireysel Arsa Finansmanı
  Kâr      : []
  Fin.oran : ['%50']
  Tutar    : []
  Vade     : ['36 aya kadar']
  Masraf   : []
  Para     : []

Bireysel İş Yeri Finansmanı
  Kâr      : []
  Fin.oran : ['%75']
  Tutar    : []
  Vade     : ['60 aya kadar']
  Masraf   : []
  Para     : []

YP Konut Finansmanı
  Kâr      : []
  Fin.oran : ['%50']
  Tutar    : []
  Vade     : []
  Masraf   : []
  Para     : ['USD', 'EUR']

Kentsel Dönüşüm Finansmanı
  Kâr      : []
  Fin.oran : []
  Tutar    : ['1.250.000 TL', '6.000.000 TL', '320.00

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# ZİRAAT KATILIM
# CAMPAIGN INDEX ACCESS TEST V1
# ============================================================

!pip -q install requests==2.32.4 beautifulsoup4

import requests
from bs4 import BeautifulSoup


OFFICIAL_URL = (
    "https://www.ziraatkatilim.com.tr/"
    "kart-kampanyalari"
)

TEST_URLS = {
    "direct":
        OFFICIAL_URL,

    "jina_https":
        (
            "https://r.jina.ai/https://"
            "www.ziraatkatilim.com.tr/"
            "kart-kampanyalari"
        ),

    "jina_http":
        (
            "https://r.jina.ai/http://"
            "www.ziraatkatilim.com.tr/"
            "kart-kampanyalari"
        ),

    "google_translate":
        (
            "https://www-ziraatkatilim-com-tr."
            "translate.goog/kart-kampanyalari"
            "?_x_tr_sl=tr"
            "&_x_tr_tl=tr"
            "&_x_tr_hl=tr"
        ),
}


session = requests.Session()

session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/151.0.0.0 "
        "Safari/537.36"
    ),
    "Accept-Language": "tr-TR,tr;q=0.9,en;q=0.8",
})


def inspect(name, url):

    print()
    print("=" * 110)
    print(name.upper())
    print("=" * 110)
    print(url)

    try:

        r = session.get(
            url,
            timeout=45,
            allow_redirects=True
        )

        text = r.text

        print("HTTP   :", r.status_code)
        print("Length :", len(text))
        print("Final  :", r.url)

        lower = text.casefold()

        waf = any(
            marker in lower
            for marker in (
                "please enable javascript",
                "support id",
                "request rejected",
                "requested url was rejected",
            )
        )

        soup = BeautifulSoup(
            text,
            "html.parser"
        )

        plain = soup.get_text(
            " ",
            strip=True
        )

        campaign_links = []

        for a in soup.find_all(
            "a",
            href=True
        ):

            href = a["href"]

            if (
                "/kart-kampanyalari/"
                in href
            ):

                campaign_links.append(
                    href
                )

        campaign_links = list(
            dict.fromkeys(
                campaign_links
            )
        )

        markers = {
            "Kart Kampanyaları":
                "kart kampanyaları"
                in plain.casefold(),

            "Okul +8":
                "okul ödemelerinizde +8 taksit"
                in plain.casefold(),

            "Mobilya 1500":
                (
                    "mobilya alışverişinize "
                    "1.500 tl"
                )
                in plain.casefold(),

            "31.08.2026":
                "31.08.2026"
                in plain,

            "07.09.2026":
                "07.09.2026"
                in plain,
        }

        print("WAF    :", waf)
        print(
            "Campaign link:",
            len(campaign_links)
        )

        print("Markers:")

        for key, value in markers.items():
            print(
                f"  {key:<20}: {value}"
            )

        success = (
            r.status_code == 200
            and
            not waf
            and
            (
                len(campaign_links) >= 20
                or
                markers["Okul +8"]
                or
                markers["Mobilya 1500"]
            )
        )

        print(
            "RESULT :",
            "✅ ÇALIŞIYOR"
            if success
            else "❌ UYGUN DEĞİL"
        )

        return {
            "name": name,
            "success": success,
            "status": r.status_code,
            "length": len(text),
            "campaign_links":
                len(campaign_links),
            "waf": waf,
        }

    except Exception as e:

        print(
            "ERROR  :",
            type(e).__name__,
            e
        )

        print(
            "RESULT : ❌ UYGUN DEĞİL"
        )

        return {
            "name": name,
            "success": False,
            "error": str(e),
        }


results = []

for name, url in TEST_URLS.items():

    results.append(
        inspect(
            name,
            url
        )
    )


print()
print("=" * 110)
print("ÖZET")
print("=" * 110)

working = []

for result in results:

    print(
        f"{result['name']:<20}:",
        "✅"
        if result["success"]
        else "❌"
    )

    if result["success"]:
        working.append(
            result["name"]
        )


print()
print(
    "Çalışan yöntem:",
    working
)


if working:

    print()
    print(
        "SONUÇ: KAMPANYA INDEX "
        "ERİŞİM YÖNTEMİ BULUNDU ✅"
    )

else:

    print()
    print(
        "SONUÇ: TÜM HTTP YÖNTEMLERİ "
        "KAPALI ⚠️"
    )


DIRECT
https://www.ziraatkatilim.com.tr/kart-kampanyalari
HTTP   : 200
Length : 7513
Final  : https://www.ziraatkatilim.com.tr/kart-kampanyalari
WAF    : True
Campaign link: 0
Markers:
  Kart Kampanyaları   : False
  Okul +8             : False
  Mobilya 1500        : False
  31.08.2026          : False
  07.09.2026          : False
RESULT : ❌ UYGUN DEĞİL

JINA_HTTPS
https://r.jina.ai/https://www.ziraatkatilim.com.tr/kart-kampanyalari
HTTP   : 403
Length : 5870
Final  : https://r.jina.ai/https://www.ziraatkatilim.com.tr/kart-kampanyalari
WAF    : False
Campaign link: 0
Markers:
  Kart Kampanyaları   : False
  Okul +8             : False
  Mobilya 1500        : False
  31.08.2026          : False
  07.09.2026          : False
RESULT : ❌ UYGUN DEĞİL

JINA_HTTP
https://r.jina.ai/http://www.ziraatkatilim.com.tr/kart-kampanyalari
HTTP   : 403
Length : 5867
Final  : https://r.jina.ai/http://www.ziraatkatilim.com.tr/kart-kampanyalari
WAF    : False
Campaign link: 0
Markers:
  Kart Kampanyala

In [ ]:
# ============================================================
# ZİRAAT KATILIM
# CAMPAIGN SEARCH-INDEX DISCOVERY V2
#
# Ana site WAF nedeniyle crawl edilmiyor.
# Bing RSS indeksinden resmi kampanya URL'leri toplanıyor.
# ============================================================

!pip -q install requests==2.32.4

import json
import re
import time
import urllib.parse
import xml.etree.ElementTree as ET

import requests

from collections import defaultdict
from urllib.parse import urlparse, parse_qs

from google.colab import files


# ============================================================
# AYARLAR
# ============================================================

OUTPUT_FILE = (
    "/content/"
    "ziraat_katilim_kampanya_discovery_v2.json"
)

BASE_PATH = (
    "https://www.ziraatkatilim.com.tr/"
    "kart-kampanyalari/"
)


# ============================================================
# ARAMA SORGULARI
# ============================================================

TERMS = [
    "",
    "Bankkart Lira",
    "taksit",
    "indirim",
    "kampanya",
    "market",
    "akaryakıt",
    "giyim",
    "mobilya",
    "elektronik",
    "e-ticaret",
    "restoran",
    "seyahat",
    "eğitim",
    "okul",
    "sağlık",
    "sigorta",
    "otomotiv",
    "ulaşım",
    "kültür sanat",
    "konaklama",
    "TROY",
    "Aile Kart",
    "Bağımsız Kart",
    "ilk kredi kartı",
    "ek kredi kartı",
    "2026",
    "Ağustos 2026",
    "Eylül 2026",
    "Aralık 2026",
    "31.08.2026",
    "30.09.2026",
    "31.12.2026",
]


QUERIES = []

for term in TERMS:

    if term:

        QUERIES.append(
            (
                "site:ziraatkatilim.com.tr/"
                "kart-kampanyalari "
                f'"{term}"'
            )
        )

    else:

        QUERIES.append(
            (
                "site:ziraatkatilim.com.tr/"
                "kart-kampanyalari"
            )
        )


# ============================================================
# SESSION
# ============================================================

session = requests.Session()

session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/151.0.0.0 "
        "Safari/537.36"
    ),
    "Accept-Language":
        "tr-TR,tr;q=0.9,en;q=0.8",
})


# ============================================================
# HELPERS
# ============================================================

def clean_text(value):

    value = str(
        value or ""
    )

    value = (
        value
        .replace("\xa0", " ")
        .replace("’", "'")
        .replace("‘", "'")
    )

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value.strip()


def normalize_campaign_url(url):

    url = clean_text(
        url
    )

    if not url:
        return None


    parsed = urlparse(
        url
    )

    host = parsed.netloc.lower()

    if host not in {
        "ziraatkatilim.com.tr",
        "www.ziraatkatilim.com.tr",
    }:

        return None


    path = parsed.path.rstrip("/")


    prefix = (
        "/kart-kampanyalari/"
    )


    if not path.startswith(
        prefix
    ):

        return None


    slug = path[
        len(prefix):
    ].strip("/")


    if not slug:

        return None


    # Alt klasör gibi saçma sonuçları ele.
    if "/" in slug:
        return None


    qs = parse_qs(
        parsed.query
    )


    archived = (
        str(
            qs.get(
                "IsArchived",
                ["false"]
            )[0]
        ).lower()
        == "true"
    )


    canonical_url = (
        BASE_PATH
        + slug
    )


    return {
        "slug":
            slug,

        "canonical_url":
            canonical_url,

        "found_url":
            url,

        "archived_param":
            archived,
    }


# ============================================================
# BING RSS
# ============================================================

def bing_rss(
    query,
    first=1
):

    encoded = urllib.parse.quote_plus(
        query
    )

    url = (
        "https://www.bing.com/search"
        f"?q={encoded}"
        "&format=rss"
        "&count=50"
        f"&first={first}"
    )


    response = session.get(
        url,
        timeout=30
    )

    response.raise_for_status()


    root = ET.fromstring(
        response.text
    )


    results = []


    for item in root.findall(
        ".//item"
    ):

        results.append({
            "title":
                clean_text(
                    item.findtext(
                        "title"
                    )
                ),

            "url":
                clean_text(
                    item.findtext(
                        "link"
                    )
                ),

            "description":
                clean_text(
                    item.findtext(
                        "description"
                    )
                ),
        })


    return results


# ============================================================
# DISCOVERY
# ============================================================

print(
    "=" * 115
)

print(
    "ZİRAAT KATILIM - "
    "CAMPAIGN SEARCH INDEX DISCOVERY V2"
)

print(
    "=" * 115
)

print(
    "Sorgu sayısı:",
    len(QUERIES)
)


slug_map = {}

query_logs = []

raw_hits = 0


# Her query için birkaç Bing sonuç sayfası.
FIRST_VALUES = [
    1,
    11,
    21,
    31,
    41,
]


for q_index, query in enumerate(
    QUERIES,
    start=1
):

    print()
    print(
        "=" * 115
    )

    print(
        f"QUERY {q_index:02d}/"
        f"{len(QUERIES):02d}"
    )

    print(
        query
    )


    query_total = 0


    for first in FIRST_VALUES:

        try:

            results = bing_rss(
                query,
                first=first
            )

        except Exception as e:

            print(
                f"  first={first} "
                f"ERROR: {type(e).__name__}: {e}"
            )

            continue


        if not results:

            break


        accepted = 0


        for result in results:

            parsed = normalize_campaign_url(
                result[
                    "url"
                ]
            )


            if not parsed:
                continue


            raw_hits += 1
            accepted += 1
            query_total += 1


            slug = parsed[
                "slug"
            ]


            if slug not in slug_map:

                slug_map[
                    slug
                ] = {
                    "slug":
                        slug,

                    "canonical_url":
                        parsed[
                            "canonical_url"
                        ],

                    "titles":
                        [],

                    "descriptions":
                        [],

                    "found_urls":
                        [],

                    "queries":
                        [],

                    "seen_active_url":
                        False,

                    "seen_archived_url":
                        False,
                }


            item = slug_map[
                slug
            ]


            if (
                result["title"]
                and
                result["title"]
                not in item["titles"]
            ):

                item[
                    "titles"
                ].append(
                    result[
                        "title"
                    ]
                )


            if (
                result["description"]
                and
                result["description"]
                not in item["descriptions"]
            ):

                item[
                    "descriptions"
                ].append(
                    result[
                        "description"
                    ]
                )


            if (
                parsed[
                    "found_url"
                ]
                not in item[
                    "found_urls"
                ]
            ):

                item[
                    "found_urls"
                ].append(
                    parsed[
                        "found_url"
                    ]
                )


            if (
                query
                not in item[
                    "queries"
                ]
            ):

                item[
                    "queries"
                ].append(
                    query
                )


            if parsed[
                "archived_param"
            ]:

                item[
                    "seen_archived_url"
                ] = True

            else:

                item[
                    "seen_active_url"
                ] = True


        print(
            f"  first={first:<2} "
            f"results={len(results):<2} "
            f"campaign={accepted}"
        )


        time.sleep(
            0.35
        )


    query_logs.append({
        "query":
            query,

        "campaign_hits":
            query_total,
    })


# ============================================================
# STATUS
# ============================================================

records = []


for slug, item in slug_map.items():

    # Eğer normal URL arama indeksinde mevcutsa
    # aktif aday sayıyoruz.
    #
    # Sadece IsArchived=true görülmüşse
    # arşiv adayı.
    if item[
        "seen_active_url"
    ]:

        status = (
            "aktif_aday"
        )

    else:

        status = (
            "arsiv_aday"
        )


    item[
        "discovery_status"
    ] = status


    item[
        "search_hit_count"
    ] = len(
        item[
            "queries"
        ]
    )


    records.append(
        item
    )


records.sort(
    key=lambda x: (
        0
        if x[
            "discovery_status"
        ]
        == "aktif_aday"
        else 1,

        -x[
            "search_hit_count"
        ],

        x[
            "slug"
        ],
    )
)


active = [
    x
    for x in records
    if (
        x[
            "discovery_status"
        ]
        == "aktif_aday"
    )
]


archived = [
    x
    for x in records
    if (
        x[
            "discovery_status"
        ]
        == "arsiv_aday"
    )
]


# ============================================================
# OUTPUT
# ============================================================

output = {
    "banka":
        "Ziraat Katılım Bankası A.Ş.",

    "discovery_method":
        "Bing RSS search index",

    "query_count":
        len(
            QUERIES
        ),

    "raw_campaign_hits":
        raw_hits,

    "unique_campaign_slug":
        len(
            records
        ),

    "aktif_aday":
        len(
            active
        ),

    "arsiv_aday":
        len(
            archived
        ),

    "kampanyalar":
        records,

    "query_logs":
        query_logs,
}


with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# PRINT
# ============================================================

print()
print(
    "=" * 115
)

print(
    "DISCOVERY SONUCU"
)

print(
    "=" * 115
)

print(
    "Search query      :",
    len(
        QUERIES
    )
)

print(
    "Raw campaign hit  :",
    raw_hits
)

print(
    "Unique slug       :",
    len(
        records
    )
)

print(
    "Aktif aday        :",
    len(
        active
    )
)

print(
    "Arşiv aday        :",
    len(
        archived
    )
)

print(
    "Dosya             :",
    OUTPUT_FILE
)


print()
print(
    "=" * 115
)

print(
    "AKTİF ADAYLAR"
)

print(
    "=" * 115
)


for index, item in enumerate(
    active,
    start=1
):

    title = (
        item[
            "titles"
        ][0]
        if item[
            "titles"
        ]
        else "-"
    )

    print(
        f"[{index:03d}] "
        f"{title}"
    )

    print(
        "      slug:",
        item[
            "slug"
        ]
    )

    print(
        "      hit :",
        item[
            "search_hit_count"
        ]
    )


print()
print(
    "=" * 115
)

print(
    "ARŞİV ADAYLAR"
)

print(
    "=" * 115
)


for index, item in enumerate(
    archived,
    start=1
):

    title = (
        item[
            "titles"
        ][0]
        if item[
            "titles"
        ]
        else "-"
    )

    print(
        f"[{index:03d}] "
        f"{title}"
    )

    print(
        "      slug:",
        item[
            "slug"
        ]
    )


print()
print(
    "=" * 115
)


if records:

    print(
        "SONUÇ: SEARCH-INDEX "
        "KAMPANYA DISCOVERY "
        "BAŞARILI ✅"
    )

else:

    print(
        "SONUÇ: HİÇ KAMPANYA "
        "URL'Sİ BULUNAMADI ❌"
    )


print(
    "=" * 115
)


files.download(
    OUTPUT_FILE
)

ZİRAAT KATILIM - CAMPAIGN SEARCH INDEX DISCOVERY V2
Sorgu sayısı: 33

QUERY 01/33
site:ziraatkatilim.com.tr/kart-kampanyalari
  first=1  results=10 campaign=0
  first=11 results=10 campaign=0
  first=21 results=10 campaign=0
  first=31 results=10 campaign=0
  first=41 results=4  campaign=0

QUERY 02/33
site:ziraatkatilim.com.tr/kart-kampanyalari "Bankkart Lira"
  first=1  results=10 campaign=0
  first=11 results=10 campaign=0
  first=21 results=10 campaign=0
  first=31 results=10 campaign=0
  first=41 results=10 campaign=0

QUERY 03/33
site:ziraatkatilim.com.tr/kart-kampanyalari "taksit"
  first=1  results=10 campaign=0
  first=11 results=7  campaign=0
  first=21 results=7  campaign=0
  first=31 results=10 campaign=0
  first=41 results=10 campaign=0

QUERY 04/33
site:ziraatkatilim.com.tr/kart-kampanyalari "indirim"
  first=1  results=10 campaign=0
  first=11 results=10 campaign=0
  first=21 results=10 campaign=0
  first=31 results=10 campaign=0
  first=41 results=10 campaign=0

QUERY 0

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# ZİRAAT KATILIM
# CAMPAIGN SEARCH-INDEX DISCOVERY V3
#
# FIX:
# Bing RSS <link> doğrudan hedef olmayabilir:
#
# https://www.bing.com/ck/a?...&u=a1aHR0cHM6...
#
# Bu sürüm:
# 1) Direct URL
# 2) Bing redirect URL
# 3) Base64 "u=a1..."
# 4) URL encoded link
# 5) description içindeki URL
#
# hepsini çözmeye çalışır.
# ============================================================

!pip -q install requests==2.32.4

import base64
import html
import json
import re
import time
import urllib.parse
import xml.etree.ElementTree as ET

import requests

from urllib.parse import (
    urlparse,
    parse_qs,
)

from google.colab import files


# ============================================================
# OUTPUT
# ============================================================

OUTPUT_FILE = (
    "/content/"
    "ziraat_katilim_kampanya_discovery_v3.json"
)


BASE_PATH = (
    "https://www.ziraatkatilim.com.tr/"
    "kart-kampanyalari/"
)


# ============================================================
# SEARCH TERMS
# ============================================================

TERMS = [
    "",
    "Bankkart",
    "Bankkart Lira",
    "taksit",
    "indirim",
    "kampanya",
    "market",
    "akaryakıt",
    "giyim",
    "mobilya",
    "elektronik",
    "e-ticaret",
    "restoran",
    "seyahat",
    "eğitim",
    "okul",
    "sağlık",
    "sigorta",
    "otomotiv",
    "ulaşım",
    "kültür",
    "sanat",
    "konaklama",
    "TROY",
    "Aile Kart",
    "Bağımsız Kart",
    "ilk kredi kartı",
    "ek kredi kartı",
    "2026",
    "Ağustos 2026",
    "Eylül 2026",
    "Ekim 2026",
    "Kasım 2026",
    "Aralık 2026",
    "31.08.2026",
    "30.09.2026",
    "31.12.2026",
]


QUERIES = []

for term in TERMS:

    if term:

        QUERIES.append(
            (
                "site:ziraatkatilim.com.tr/"
                "kart-kampanyalari "
                f'"{term}"'
            )
        )

    else:

        QUERIES.append(
            (
                "site:ziraatkatilim.com.tr/"
                "kart-kampanyalari"
            )
        )


# ============================================================
# SESSION
# ============================================================

session = requests.Session()

session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/151.0.0.0 "
        "Safari/537.36"
    ),
    "Accept-Language":
        "tr-TR,tr;q=0.9,en;q=0.8",
})


# ============================================================
# HELPERS
# ============================================================

def clean_text(value):

    value = html.unescape(
        str(value or "")
    )

    value = (
        value
        .replace("\xa0", " ")
        .replace("’", "'")
        .replace("‘", "'")
    )

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value.strip()


def repeated_unquote(value):

    old = str(
        value or ""
    )

    for _ in range(4):

        new = urllib.parse.unquote(
            old
        )

        if new == old:
            break

        old = new

    return old


# ============================================================
# BASE64 DECODER
# ============================================================

def try_base64_decode(value):

    value = str(
        value or ""
    ).strip()


    # Bing format:
    #
    # u=a1aHR0cHM6Ly...
    #
    # a1 prefixi Bing işaretidir.
    if value.startswith(
        "a1"
    ):

        value = value[2:]


    # Base64 padding
    padding = (
        4
        -
        (
            len(value)
            % 4
        )
    ) % 4

    value += (
        "="
        * padding
    )


    try:

        decoded = (
            base64
            .urlsafe_b64decode(
                value
            )
            .decode(
                "utf-8",
                errors="ignore"
            )
        )

        return decoded.strip()

    except Exception:

        return ""


# ============================================================
# FIND DIRECT ZIRAAT URL INSIDE TEXT
# ============================================================

def find_ziraat_url_in_text(
    value
):

    value = clean_text(
        value
    )

    value = repeated_unquote(
        value
    )


    pattern = re.compile(
        (
            r"https?://"
            r"(?:www\.)?"
            r"ziraatkatilim\.com\.tr/"
            r"kart-kampanyalari/"
            r"[^\s\"'<>]+"
        ),
        flags=re.I
    )


    match = pattern.search(
        value
    )


    if not match:

        return None


    url = match.group(0)


    # Search-result punctuation
    url = url.rstrip(
        ".,;:)]}>"
    )


    return url


# ============================================================
# BING URL UNWRAPPER
# ============================================================

def unwrap_bing_url(
    raw_url
):

    raw_url = clean_text(
        raw_url
    )


    # --------------------------------------------------------
    # 1. Zaten Ziraat URL'siyse
    # --------------------------------------------------------

    direct = find_ziraat_url_in_text(
        raw_url
    )

    if direct:

        return direct


    # --------------------------------------------------------
    # 2. URL decode
    # --------------------------------------------------------

    decoded_url = repeated_unquote(
        raw_url
    )


    direct = find_ziraat_url_in_text(
        decoded_url
    )

    if direct:

        return direct


    # --------------------------------------------------------
    # 3. Bing "u" query param
    # --------------------------------------------------------

    u_values = []


    try:

        parsed = urlparse(
            raw_url
        )

        qs = parse_qs(
            parsed.query
        )

        u_values.extend(
            qs.get(
                "u",
                []
            )
        )

    except Exception:

        pass


    # parse_qs bazı Bing ck/a URL'lerinde
    # yeterli olmayabilir.
    manual = re.findall(
        r"(?:[?&]u=)([^&]+)",
        raw_url,
        flags=re.I
    )

    u_values.extend(
        manual
    )


    for u_value in u_values:

        u_value = repeated_unquote(
            u_value
        )


        # ------------------------------
        # u=https://...
        # ------------------------------

        direct = find_ziraat_url_in_text(
            u_value
        )

        if direct:

            return direct


        # ------------------------------
        # u=a1BASE64...
        # ------------------------------

        decoded = try_base64_decode(
            u_value
        )


        direct = find_ziraat_url_in_text(
            decoded
        )

        if direct:

            return direct


    # --------------------------------------------------------
    # 4. Raw URL içindeki Base64 parçasını ayrıca ara
    # --------------------------------------------------------

    base64_candidates = re.findall(
        r"(?:u=)(a1[A-Za-z0-9_\-]+=*)",
        raw_url
    )


    for candidate in base64_candidates:

        decoded = try_base64_decode(
            candidate
        )

        direct = find_ziraat_url_in_text(
            decoded
        )

        if direct:

            return direct


    return None


# ============================================================
# NORMALIZE CAMPAIGN URL
# ============================================================

def normalize_campaign_url(
    url
):

    if not url:

        return None


    url = clean_text(
        url
    )


    parsed = urlparse(
        url
    )


    host = (
        parsed.netloc
        .lower()
        .replace(
            ":443",
            ""
        )
    )


    if host not in {
        "ziraatkatilim.com.tr",
        "www.ziraatkatilim.com.tr",
    }:

        return None


    path = (
        parsed.path
        .rstrip("/")
    )


    path_lower = (
        path
        .casefold()
    )


    prefix = (
        "/kart-kampanyalari/"
    )


    if not path_lower.startswith(
        prefix
    ):

        return None


    slug = path[
        len(prefix):
    ].strip("/")


    if not slug:

        return None


    # Detail page olmalı.
    if "/" in slug:

        return None


    # Asset vb. olmasın.
    if re.search(
        r"\.(?:jpg|jpeg|png|gif|svg|pdf|css|js)$",
        slug,
        flags=re.I
    ):

        return None


    qs = parse_qs(
        parsed.query
    )


    archived = (
        str(
            qs.get(
                "IsArchived",
                qs.get(
                    "isarchived",
                    ["false"]
                )
            )[0]
        ).casefold()
        == "true"
    )


    canonical_url = (
        BASE_PATH
        + slug
    )


    return {
        "slug":
            slug,

        "canonical_url":
            canonical_url,

        "found_url":
            url,

        "archived_param":
            archived,
    }


# ============================================================
# BING RSS
# ============================================================

def bing_rss(
    query,
    first=1
):

    encoded = (
        urllib.parse
        .quote_plus(
            query
        )
    )


    rss_url = (
        "https://www.bing.com/search"
        f"?q={encoded}"
        "&format=rss"
        "&count=50"
        f"&first={first}"
    )


    response = session.get(
        rss_url,
        timeout=30
    )

    response.raise_for_status()


    root = ET.fromstring(
        response.text
    )


    results = []


    for item in root.findall(
        ".//item"
    ):

        results.append({
            "title":
                clean_text(
                    item.findtext(
                        "title"
                    )
                ),

            "link":
                clean_text(
                    item.findtext(
                        "link"
                    )
                ),

            "description":
                clean_text(
                    item.findtext(
                        "description"
                    )
                ),
        })


    return results


# ============================================================
# EXTRACT CAMPAIGN URL FROM RESULT
# ============================================================

def extract_campaign_url(
    result
):

    candidates = []


    # --------------------------------------------------------
    # Link
    # --------------------------------------------------------

    link = result.get(
        "link",
        ""
    )


    if link:

        candidates.append(
            link
        )


        unwrapped = unwrap_bing_url(
            link
        )

        if unwrapped:

            candidates.append(
                unwrapped
            )


    # --------------------------------------------------------
    # Description
    # --------------------------------------------------------

    description = result.get(
        "description",
        ""
    )


    if description:

        candidates.append(
            description
        )


        found = find_ziraat_url_in_text(
            description
        )

        if found:

            candidates.append(
                found
            )


    # --------------------------------------------------------
    # Title - nadiren URL bulunabilir
    # --------------------------------------------------------

    title = result.get(
        "title",
        ""
    )


    if title:

        candidates.append(
            title
        )


    # --------------------------------------------------------
    # Normalize
    # --------------------------------------------------------

    for candidate in candidates:

        # Önce direkt normalize.
        parsed = normalize_campaign_url(
            candidate
        )

        if parsed:

            return parsed


        # İçinden URL çıkar.
        inner_url = find_ziraat_url_in_text(
            candidate
        )


        if inner_url:

            parsed = normalize_campaign_url(
                inner_url
            )

            if parsed:

                return parsed


    return None


# ============================================================
# DISCOVERY
# ============================================================

print(
    "=" * 118
)

print(
    "ZİRAAT KATILIM - "
    "CAMPAIGN SEARCH INDEX DISCOVERY V3"
)

print(
    "=" * 118
)

print(
    "Sorgu sayısı:",
    len(QUERIES)
)


slug_map = {}

raw_search_results = 0
decoded_url_count = 0


FIRST_VALUES = [
    1,
    11,
    21,
    31,
    41,
]


# Debug için ilk birkaç ham Bing linki.
debug_links = []


for q_index, query in enumerate(
    QUERIES,
    start=1
):

    print()
    print(
        f"[QUERY {q_index:02d}/"
        f"{len(QUERIES):02d}] "
        f"{query}"
    )


    for first in FIRST_VALUES:

        try:

            results = bing_rss(
                query,
                first=first
            )

        except Exception as e:

            print(
                f"  first={first}: "
                f"ERROR {type(e).__name__}: {e}"
            )

            continue


        raw_search_results += len(
            results
        )


        accepted = 0


        for result in results:

            if (
                len(debug_links)
                < 10
            ):

                debug_links.append({
                    "title":
                        result[
                            "title"
                        ],

                    "raw_link":
                        result[
                            "link"
                        ],

                    "unwrapped":
                        unwrap_bing_url(
                            result[
                                "link"
                            ]
                        ),
                })


            parsed = extract_campaign_url(
                result
            )


            if not parsed:

                continue


            accepted += 1
            decoded_url_count += 1


            slug = parsed[
                "slug"
            ]


            if slug not in slug_map:

                slug_map[
                    slug
                ] = {
                    "slug":
                        slug,

                    "canonical_url":
                        parsed[
                            "canonical_url"
                        ],

                    "titles":
                        [],

                    "descriptions":
                        [],

                    "found_urls":
                        [],

                    "queries":
                        [],

                    "seen_active_url":
                        False,

                    "seen_archived_url":
                        False,
                }


            item = slug_map[
                slug
            ]


            title = result.get(
                "title",
                ""
            )


            if (
                title
                and
                title not in item[
                    "titles"
                ]
            ):

                item[
                    "titles"
                ].append(
                    title
                )


            description = result.get(
                "description",
                ""
            )


            if (
                description
                and
                description not in item[
                    "descriptions"
                ]
            ):

                item[
                    "descriptions"
                ].append(
                    description
                )


            found_url = parsed[
                "found_url"
            ]


            if (
                found_url
                not in item[
                    "found_urls"
                ]
            ):

                item[
                    "found_urls"
                ].append(
                    found_url
                )


            if (
                query
                not in item[
                    "queries"
                ]
            ):

                item[
                    "queries"
                ].append(
                    query
                )


            if parsed[
                "archived_param"
            ]:

                item[
                    "seen_archived_url"
                ] = True

            else:

                item[
                    "seen_active_url"
                ] = True


        print(
            f"  first={first:<2} "
            f"results={len(results):<2} "
            f"accepted={accepted}"
        )


        time.sleep(
            0.25
        )


# ============================================================
# BUILD RECORDS
# ============================================================

records = []


for item in slug_map.values():

    if (
        item[
            "seen_active_url"
        ]
    ):

        status = (
            "aktif_aday"
        )

    else:

        status = (
            "arsiv_aday"
        )


    item[
        "discovery_status"
    ] = status


    item[
        "search_hit_count"
    ] = len(
        item[
            "queries"
        ]
    )


    records.append(
        item
    )


records.sort(
    key=lambda x: (
        0
        if x[
            "discovery_status"
        ]
        == "aktif_aday"
        else 1,

        -x[
            "search_hit_count"
        ],

        x[
            "slug"
        ],
    )
)


active = [
    item
    for item in records
    if item[
        "discovery_status"
    ]
    ==
    "aktif_aday"
]


archived = [
    item
    for item in records
    if item[
        "discovery_status"
    ]
    ==
    "arsiv_aday"
]


# ============================================================
# SAVE
# ============================================================

output = {
    "banka":
        "Ziraat Katılım Bankası A.Ş.",

    "discovery_method":
        (
            "Bing RSS + "
            "Bing redirect decode"
        ),

    "query_count":
        len(
            QUERIES
        ),

    "raw_search_results":
        raw_search_results,

    "decoded_campaign_hits":
        decoded_url_count,

    "unique_campaign_slug":
        len(
            records
        ),

    "aktif_aday":
        len(
            active
        ),

    "arsiv_aday":
        len(
            archived
        ),

    "kampanyalar":
        records,

    "debug_links":
        debug_links,
}


with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# DEBUG
# ============================================================

print()
print(
    "=" * 118
)

print(
    "İLK 10 BING LINK DEBUG"
)

print(
    "=" * 118
)


for i, item in enumerate(
    debug_links,
    start=1
):

    print()
    print(
        f"[{i:02d}]",
        item[
            "title"
        ]
    )

    print(
        "RAW :",
        item[
            "raw_link"
        ]
    )

    print(
        "REAL:",
        item[
            "unwrapped"
        ]
    )


# ============================================================
# RESULT
# ============================================================

print()
print(
    "=" * 118
)

print(
    "DISCOVERY V3 SONUCU"
)

print(
    "=" * 118
)

print(
    "Search query        :",
    len(
        QUERIES
    )
)

print(
    "Raw search result   :",
    raw_search_results
)

print(
    "Campaign URL hit    :",
    decoded_url_count
)

print(
    "Unique slug         :",
    len(
        records
    )
)

print(
    "Aktif aday          :",
    len(
        active
    )
)

print(
    "Arşiv aday          :",
    len(
        archived
    )
)

print(
    "Dosya               :",
    OUTPUT_FILE
)


# ============================================================
# ACTIVE LIST
# ============================================================

print()
print(
    "=" * 118
)

print(
    "AKTİF ADAYLAR"
)

print(
    "=" * 118
)


for index, item in enumerate(
    active,
    start=1
):

    title = (
        item[
            "titles"
        ][0]
        if item[
            "titles"
        ]
        else item[
            "slug"
        ]
    )

    print(
        f"[{index:03d}] "
        f"{title}"
    )

    print(
        "      ",
        item[
            "canonical_url"
        ]
    )


# ============================================================
# ARCHIVE LIST
# ============================================================

print()
print(
    "=" * 118
)

print(
    "ARŞİV ADAYLAR"
)

print(
    "=" * 118
)


for index, item in enumerate(
    archived,
    start=1
):

    title = (
        item[
            "titles"
        ][0]
        if item[
            "titles"
        ]
        else item[
            "slug"
        ]
    )

    print(
        f"[{index:03d}] "
        f"{title}"
    )


print()
print(
    "=" * 118
)


if records:

    print(
        "SONUÇ: KAMPANYA URL "
        "DISCOVERY BAŞARILI ✅"
    )

else:

    print(
        "SONUÇ: HALA 0 URL ❌"
    )


print(
    "=" * 118
)


files.download(
    OUTPUT_FILE
)

ZİRAAT KATILIM - CAMPAIGN SEARCH INDEX DISCOVERY V3
Sorgu sayısı: 37

[QUERY 01/37] site:ziraatkatilim.com.tr/kart-kampanyalari
  first=1  results=10 accepted=0
  first=11 results=10 accepted=0
  first=21 results=10 accepted=0
  first=31 results=10 accepted=0
  first=41 results=10 accepted=0

[QUERY 02/37] site:ziraatkatilim.com.tr/kart-kampanyalari "Bankkart"
  first=1  results=10 accepted=0
  first=11 results=10 accepted=0
  first=21 results=10 accepted=0
  first=31 results=10 accepted=0
  first=41 results=10 accepted=0

[QUERY 03/37] site:ziraatkatilim.com.tr/kart-kampanyalari "Bankkart Lira"
  first=1  results=10 accepted=0
  first=11 results=10 accepted=0
  first=21 results=6  accepted=0
  first=31 results=10 accepted=0
  first=41 results=10 accepted=0

[QUERY 04/37] site:ziraatkatilim.com.tr/kart-kampanyalari "taksit"
  first=1  results=10 accepted=0
  first=11 results=10 accepted=0
  first=21 results=10 accepted=0
  first=31 results=10 accepted=0
  first=41 results=10 accepted=0

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# ZİRAAT KATILIM
# KARTAVANTAJ DISCOVERY ACCESS TEST V4
#
# Amaç:
# 1. KartAvantaj Ziraat Katılım sayfasına erişiliyor mu?
# 2. Kaç aktif kampanya yazıyor?
# 3. /kampanya/... detay linkleri geliyor mu?
# 4. Pagination yapısı nedir?
# 5. Detay sayfasından resmi Ziraat URL'si çıkıyor mu?
# ============================================================

!pip -q install requests==2.32.4 beautifulsoup4


import re
import requests

from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse


# ============================================================
# URLS
# ============================================================

BASE = "https://kartavantaj.com"

BANK_URL = (
    BASE
    + "/banka/ziraat-katilim"
)


# ============================================================
# SESSION
# ============================================================

session = requests.Session()

session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/151.0.0.0 "
        "Safari/537.36"
    ),
    "Accept-Language":
        "tr-TR,tr;q=0.9,en;q=0.8",
})


# ============================================================
# HELPERS
# ============================================================

def clean_text(value):

    value = str(
        value or ""
    )

    value = (
        value
        .replace("\xa0", " ")
        .replace("’", "'")
        .replace("‘", "'")
    )

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value.strip()


def get_page(url):

    try:

        response = session.get(
            url,
            timeout=30,
            allow_redirects=True
        )

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        return response, soup

    except Exception as e:

        print(
            "REQUEST ERROR:",
            type(e).__name__,
            e
        )

        return None, None


# ============================================================
# MAIN BANK PAGE
# ============================================================

print("=" * 115)
print("ZİRAAT KATILIM - KARTAVANTAJ ACCESS TEST V4")
print("=" * 115)

print()
print("BANK PAGE:")
print(BANK_URL)


response, soup = get_page(
    BANK_URL
)


if response is None:

    raise RuntimeError(
        "KartAvantaj sayfasına erişilemedi."
    )


print(
    "HTTP       :",
    response.status_code
)

print(
    "Length     :",
    len(response.text)
)

print(
    "Final URL  :",
    response.url
)


if response.status_code != 200:

    raise RuntimeError(
        (
            "KartAvantaj HTTP "
            f"{response.status_code}"
        )
    )


# ============================================================
# H1 / TEXT
# ============================================================

h1 = soup.find(
    "h1"
)

h1_text = (
    clean_text(
        h1.get_text(
            " ",
            strip=True
        )
    )
    if h1
    else ""
)


page_text = clean_text(
    soup.get_text(
        "\n",
        strip=True
    )
)


print(
    "H1         :",
    h1_text
)


# ============================================================
# ACTIVE COUNT
# ============================================================

active_match = re.search(
    r"(\d+)\s+aktif\s+kampanya\s+bulundu",
    page_text,
    flags=re.I
)


active_count = (
    int(
        active_match.group(1)
    )
    if active_match
    else None
)


print(
    "Aktif sayı :",
    active_count
)


# ============================================================
# CAMPAIGN DETAIL LINKS
# ============================================================

campaign_links = []


for a in soup.find_all(
    "a",
    href=True
):

    href = urljoin(
        BASE,
        a[
            "href"
        ]
    )

    parsed = urlparse(
        href
    )


    if (
        parsed.netloc
        not in {
            "kartavantaj.com",
            "www.kartavantaj.com",
        }
    ):

        continue


    if not parsed.path.startswith(
        "/kampanya/"
    ):

        continue


    href = (
        f"https://kartavantaj.com"
        f"{parsed.path}"
    )


    if href not in campaign_links:

        campaign_links.append(
            href
        )


print(
    "Detail link:",
    len(
        campaign_links
    )
)


# ============================================================
# PAGINATION LINKS
# ============================================================

pagination_links = []


for a in soup.find_all(
    "a",
    href=True
):

    href_raw = a[
        "href"
    ]

    text = clean_text(
        a.get_text(
            " ",
            strip=True
        )
    )


    href = urljoin(
        BANK_URL,
        href_raw
    )


    # page / sayfa ihtimallerini otomatik yakala
    if (
        re.search(
            r"(?:page|sayfa)[=/]\d+",
            href,
            flags=re.I
        )
        or
        re.search(
            r"[?&](?:page|sayfa)=\d+",
            href,
            flags=re.I
        )
        or
        (
            text.isdigit()
            and
            int(text) >= 2
        )
    ):

        item = (
            text,
            href
        )

        if item not in pagination_links:

            pagination_links.append(
                item
            )


print(
    "Pagination :",
    len(
        pagination_links
    )
)


# ============================================================
# PRINT FIRST LINKS
# ============================================================

print()
print("=" * 115)
print("İLK 30 KAMPANYA DETAY LINK")
print("=" * 115)


for index, url in enumerate(
    campaign_links[:30],
    start=1
):

    print(
        f"[{index:02d}]",
        url
    )


print()
print("=" * 115)
print("PAGINATION LINKLERİ")
print("=" * 115)


if pagination_links:

    for text, href in pagination_links:

        print(
            f"text={text!r}"
        )

        print(
            "  ",
            href
        )

else:

    print(
        "[]"
    )


# ============================================================
# TEST FIRST 5 DETAIL PAGES
# ============================================================

print()
print("=" * 115)
print("DETAY -> RESMİ ZİRAAT LINK TEST")
print("=" * 115)


test_links = campaign_links[
    :5
]


detail_success = 0
official_success = 0


for index, detail_url in enumerate(
    test_links,
    start=1
):

    print()
    print(
        f"[{index:02d}/"
        f"{len(test_links):02d}]"
    )

    print(
        "DETAIL:",
        detail_url
    )


    r, detail_soup = get_page(
        detail_url
    )


    if (
        r is None
        or
        r.status_code != 200
    ):

        print(
            "  RESULT : ❌"
        )

        continue


    detail_success += 1


    detail_h1 = (
        detail_soup.find(
            "h1"
        )
    )


    title = (
        clean_text(
            detail_h1.get_text(
                " ",
                strip=True
            )
        )
        if detail_h1
        else ""
    )


    text = clean_text(
        detail_soup.get_text(
            "\n",
            strip=True
        )
    )


    official_urls = []


    for a in detail_soup.find_all(
        "a",
        href=True
    ):

        href = urljoin(
            detail_url,
            a[
                "href"
            ]
        )


        parsed = urlparse(
            href
        )


        if (
            parsed.netloc.lower()
            not in {
                "ziraatkatilim.com.tr",
                "www.ziraatkatilim.com.tr",
            }
        ):

            continue


        if (
            "/kart-kampanyalari/"
            not in parsed.path
        ):

            continue


        official = (
            "https://www."
            "ziraatkatilim.com.tr"
            + parsed.path
        )


        if official not in official_urls:

            official_urls.append(
                official
            )


    # Kampanya dönemi
    date_match = re.search(
        (
            r"Kampanya\s+Dönemi\s*"
            r"(.{0,100}?"
            r"\d{4})"
        ),
        text,
        flags=re.I
    )


    date_text = (
        clean_text(
            date_match.group(1)
        )
        if date_match
        else ""
    )


    print(
        "  HTTP   :",
        r.status_code
    )

    print(
        "  TITLE  :",
        title
    )

    print(
        "  DATE   :",
        date_text
    )

    print(
        "  OFFICIAL:",
        len(
            official_urls
        )
    )


    for official_url in official_urls:

        print(
            "    ->",
            official_url
        )


    if official_urls:

        official_success += 1

        print(
            "  RESULT : ✅"
        )

    else:

        print(
            "  RESULT : ⚠️ "
            "official link yok"
        )


# ============================================================
# SUMMARY
# ============================================================

print()
print("=" * 115)
print("ACCESS TEST SONUCU")
print("=" * 115)

print(
    "KartAvantaj HTTP      :",
    response.status_code
)

print(
    "Aktif kampanya yazısı :",
    active_count
)

print(
    "İlk sayfa detail link :",
    len(
        campaign_links
    )
)

print(
    "Pagination link       :",
    len(
        pagination_links
    )
)

print(
    "Test detail başarılı  :",
    detail_success
)

print(
    "Resmi Ziraat URL      :",
    official_success,
    "/",
    len(
        test_links
    )
)


print()
print("=" * 115)


if (
    response.status_code == 200
    and
    campaign_links
    and
    official_success > 0
):

    print(
        "SONUÇ: KARTAVANTAJ "
        "DISCOVERY YOLU ÇALIŞIYOR ✅"
    )

else:

    print(
        "SONUÇ: KARTAVANTAJ "
        "YOLU KONTROL GEREKİYOR ❌"
    )


print("=" * 115)

ZİRAAT KATILIM - KARTAVANTAJ ACCESS TEST V4

BANK PAGE:
https://kartavantaj.com/banka/ziraat-katilim
HTTP       : 200
Length     : 1874773
Final URL  : https://kartavantaj.com/banka/ziraat-katilim
H1         : Ziraat Katılım Kampanyaları
Aktif sayı : 87
Detail link: 24
Pagination : 0

İLK 30 KAMPANYA DETAY LINK
[01] https://kartavantaj.com/kampanya/troyda-5-taksit-katilim-bankkart
[02] https://kartavantaj.com/kampanya/dysonda-9-taksit-katilim-bankkart
[03] https://kartavantaj.com/kampanya/secili-okullarda-pesin-odemelerinize-5e-varan-taksit-katilim-bankkart
[04] https://kartavantaj.com/kampanya/w-collectionda-4-taksit-katilim-bankkart
[05] https://kartavantaj.com/kampanya/veteriner-ve-petshop-harcamalariniza-2000-tl-bankkart-lira
[06] https://kartavantaj.com/kampanya/okul-odemelerinizde-12-aya-varan-taksit-firsati
[07] https://kartavantaj.com/kampanya/atasun-optikte-6-taksit-katilim-bankkart
[08] https://kartavantaj.com/kampanya/baymakta-9-taksit-katilim-bankkart
[09] https://kartavant

In [ ]:
# ============================================================
# ZİRAAT KATILIM
# CAMPAIGN RAW SCRAPER V5
#
# Kaynak keşfi:
#   KartAvantaj Ziraat Katılım aktif kampanya sayfası
#
# Doğrulama:
#   Her detay sayfasında gerçek
#   ziraatkatilim.com.tr/kart-kampanyalari/... URL'si aranır.
#
# Beklenen:
#   87 aktif kampanya
# ============================================================

!pip -q install requests==2.32.4 beautifulsoup4


import html
import json
import re
import time
import urllib.parse

import requests

from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from google.colab import files


# ============================================================
# AYARLAR
# ============================================================

BASE = "https://kartavantaj.com"

BANK_URL = (
    BASE
    + "/banka/ziraat-katilim"
)

OUTPUT_FILE = (
    "/content/"
    "ziraat_katilim_kampanyalar_raw.json"
)

DISCOVERY_FILE = (
    "/content/"
    "ziraat_katilim_kampanya_discovery_v5.json"
)

EXPECTED_ACTIVE = 87


# ============================================================
# SESSION
# ============================================================

session = requests.Session()

session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/151.0.0.0 "
        "Safari/537.36"
    ),
    "Accept-Language":
        "tr-TR,tr;q=0.9,en;q=0.8",
})


# ============================================================
# HELPERS
# ============================================================

def clean_text(value):

    value = html.unescape(
        str(value or "")
    )

    value = (
        value
        .replace("\xa0", " ")
        .replace("’", "'")
        .replace("‘", "'")
        .replace("–", "-")
        .replace("—", "-")
    )

    value = re.sub(
        r"[ \t]+",
        " ",
        value
    )

    value = re.sub(
        r"\n[ \t]+",
        "\n",
        value
    )

    value = re.sub(
        r"\n{3,}",
        "\n\n",
        value
    )

    return value.strip()


def normalize(value):

    return (
        clean_text(value)
        .replace("İ", "i")
        .replace("I", "ı")
        .casefold()
    )


def get_page(url):

    try:

        response = session.get(
            url,
            timeout=40,
            allow_redirects=True
        )

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        return response, soup

    except Exception as error:

        print(
            "REQUEST ERROR:",
            url,
            type(error).__name__,
            error
        )

        return None, None


# ============================================================
# DETAIL SLUG VALIDATION
# ============================================================

def clean_slug(slug):

    slug = str(
        slug or ""
    )

    slug = urllib.parse.unquote(
        slug
    )

    slug = (
        slug
        .strip()
        .strip("/")
        .strip('"')
        .strip("'")
    )

    slug = slug.split(
        "?"
    )[0]

    slug = slug.split(
        "#"
    )[0]

    # Kampanya slug'ları için güvenli karakterler.
    if not re.fullmatch(
        r"[A-Za-z0-9çğıöşüÇĞİÖŞÜ\-]+",
        slug
    ):

        return None

    if len(slug) < 3:
        return None

    bad = {
        "kampanya",
        "kampanyalar",
        "ziraat-katilim",
    }

    if normalize(slug) in bad:
        return None

    return slug


# ============================================================
# RAW HTML'DEN SLUG DISCOVERY
# ============================================================

def discover_slugs(
    html_text,
    soup
):

    methods = {
        "anchor":
            set(),

        "raw_path":
            set(),

        "escaped_path":
            set(),

        "unicode_path":
            set(),

        "encoded_path":
            set(),

        "json_url":
            set(),

        "json_slug":
            set(),
    }


    # --------------------------------------------------------
    # 1. NORMAL <a href="/kampanya/...">
    # --------------------------------------------------------

    for a in soup.find_all(
        "a",
        href=True
    ):

        href = a.get(
            "href",
            ""
        )

        parsed = urlparse(
            urljoin(
                BASE,
                href
            )
        )

        if not parsed.path.startswith(
            "/kampanya/"
        ):

            continue

        slug = clean_slug(
            parsed.path[
                len("/kampanya/"):
            ]
        )

        if slug:

            methods[
                "anchor"
            ].add(
                slug
            )


    # --------------------------------------------------------
    # TEXT VARIANTS
    # --------------------------------------------------------

    text_variants = [
        html_text,
        html.unescape(
            html_text
        ),
    ]


    try:

        text_variants.append(
            urllib.parse.unquote(
                html_text
            )
        )

    except Exception:
        pass


    # --------------------------------------------------------
    # 2. RAW /kampanya/slug
    # --------------------------------------------------------

    for text in text_variants:

        for slug in re.findall(
            (
                r"/kampanya/"
                r"([A-Za-z0-9çğıöşüÇĞİÖŞÜ\-]+)"
            ),
            text
        ):

            slug = clean_slug(
                slug
            )

            if slug:

                methods[
                    "raw_path"
                ].add(
                    slug
                )


    # --------------------------------------------------------
    # 3. ESCAPED \/kampanya\/slug
    # --------------------------------------------------------

    for slug in re.findall(
        (
            r"\\/"
            r"kampanya"
            r"\\/"
            r"([A-Za-z0-9çğıöşüÇĞİÖŞÜ\-]+)"
        ),
        html_text
    ):

        slug = clean_slug(
            slug
        )

        if slug:

            methods[
                "escaped_path"
            ].add(
                slug
            )


    # --------------------------------------------------------
    # 4. \u002Fkampanya\u002Fslug
    # --------------------------------------------------------

    for slug in re.findall(
        (
            r"\\u002[Ff]"
            r"kampanya"
            r"\\u002[Ff]"
            r"([A-Za-z0-9çğıöşüÇĞİÖŞÜ\-]+)"
        ),
        html_text
    ):

        slug = clean_slug(
            slug
        )

        if slug:

            methods[
                "unicode_path"
            ].add(
                slug
            )


    # --------------------------------------------------------
    # 5. %2Fkampanya%2Fslug
    # --------------------------------------------------------

    for slug in re.findall(
        (
            r"%2[Ff]"
            r"kampanya"
            r"%2[Ff]"
            r"([A-Za-z0-9çğıöşüÇĞİÖŞÜ\-]+)"
        ),
        html_text
    ):

        slug = clean_slug(
            slug
        )

        if slug:

            methods[
                "encoded_path"
            ].add(
                slug
            )


    # --------------------------------------------------------
    # 6. JSON url/href fields
    # --------------------------------------------------------

    for slug in re.findall(
        (
            r'["\'](?:url|href|link)["\']'
            r"\s*:\s*"
            r'["\']'
            r"(?:https?:\\/\\/[^\"']+)?"
            r"\\?/?kampanya\\?/"
            r"([A-Za-z0-9çğıöşüÇĞİÖŞÜ\-]+)"
        ),
        html_text,
        flags=re.I
    ):

        slug = clean_slug(
            slug
        )

        if slug:

            methods[
                "json_url"
            ].add(
                slug
            )


    # --------------------------------------------------------
    # 7. JSON slug fields
    #
    # Bunlar potansiyel adaydır.
    # Detail doğrulamasından geçmeden kabul edilmeyecek.
    # --------------------------------------------------------

    for slug in re.findall(
        (
            r'["\']slug["\']'
            r"\s*:\s*"
            r'["\']'
            r"([A-Za-z0-9çğıöşüÇĞİÖŞÜ\-]+)"
            r'["\']'
        ),
        html_text,
        flags=re.I
    ):

        slug = clean_slug(
            slug
        )

        if slug:

            methods[
                "json_slug"
            ].add(
                slug
            )


    # --------------------------------------------------------
    # UNION
    # --------------------------------------------------------

    all_slugs = set()

    for values in methods.values():

        all_slugs.update(
            values
        )


    return methods, all_slugs


# ============================================================
# DETAIL PAGE MAIN TEXT
# ============================================================

def extract_detail_text(soup):

    if soup is None:
        return ""

    clone = BeautifulSoup(
        str(soup),
        "html.parser"
    )

    for selector in [
        "script",
        "style",
        "noscript",
        "svg",
        "header",
        "footer",
        "nav",
        "form",
    ]:

        for tag in clone.select(
            selector
        ):

            tag.decompose()


    main = clone.find(
        "main"
    )

    if main is None:

        h1 = clone.find(
            "h1"
        )

        if h1:

            current = h1

            for _ in range(7):

                if current is None:
                    break

                text = clean_text(
                    current.get_text(
                        "\n",
                        strip=True
                    )
                )

                if len(text) >= 400:

                    main = current
                    break

                current = current.parent


    if main is None:

        main = clone.body


    if main is None:

        return ""


    return clean_text(
        main.get_text(
            "\n",
            strip=True
        )
    )


# ============================================================
# OFFICIAL ZIRAAT URL
# ============================================================

def extract_official_urls(
    soup,
    detail_url
):

    result = []


    for a in soup.find_all(
        "a",
        href=True
    ):

        href = urljoin(
            detail_url,
            a.get(
                "href"
            )
        )

        parsed = urlparse(
            href
        )

        host = (
            parsed.netloc
            .lower()
        )

        if host not in {
            "ziraatkatilim.com.tr",
            "www.ziraatkatilim.com.tr",
        }:

            continue


        if not parsed.path.startswith(
            "/kart-kampanyalari/"
        ):

            continue


        slug = clean_slug(
            parsed.path[
                len("/kart-kampanyalari/"):
            ]
        )

        if not slug:
            continue


        official = (
            "https://www."
            "ziraatkatilim.com.tr/"
            "kart-kampanyalari/"
            + slug
        )


        if official not in result:

            result.append(
                official
            )


    return result


# ============================================================
# CAMPAIGN PERIOD
# ============================================================

MONTHS = (
    "Ocak|Şubat|Mart|Nisan|Mayıs|Haziran|"
    "Temmuz|Ağustos|Eylül|Ekim|Kasım|Aralık"
)


def extract_dates(text):

    result = []

    patterns = [

        # 01 Ağustos 2026
        (
            rf"\b"
            rf"\d{{1,2}}\s+"
            rf"(?:{MONTHS})\s+"
            rf"\d{{4}}\b"
        ),

        # 01.08.2026
        (
            r"\b"
            r"\d{1,2}[./-]"
            r"\d{1,2}[./-]"
            r"\d{4}"
            r"\b"
        ),
    ]


    for pattern in patterns:

        for value in re.findall(
            pattern,
            text,
            flags=re.I
        ):

            # re.findall alternation yüzünden tuple
            # gelirse ilk stringi al.
            if isinstance(
                value,
                tuple
            ):

                value = next(
                    (
                        x
                        for x in value
                        if x
                    ),
                    ""
                )

            value = clean_text(
                value
            )

            if (
                value
                and
                value not in result
            ):

                result.append(
                    value
                )


    return result


def extract_period_context(text):

    text = clean_text(
        text
    )


    match = re.search(
        r"Kampanya\s+Dönemi",
        text,
        flags=re.I
    )


    if not match:

        return ""


    start = match.start()

    end = min(
        len(text),
        start + 350
    )


    return clean_text(
        text[
            start:end
        ]
    )


# ============================================================
# FETCH BANK PAGE
# ============================================================

print(
    "=" * 118
)

print(
    "ZİRAAT KATILIM - "
    "CAMPAIGN RAW SCRAPER V5"
)

print(
    "=" * 118
)


response, soup = get_page(
    BANK_URL
)


if (
    response is None
    or
    response.status_code != 200
):

    raise RuntimeError(
        "KartAvantaj banka sayfası alınamadı."
    )


print(
    "Bank HTTP:",
    response.status_code
)

print(
    "HTML len :",
    len(
        response.text
    )
)


# ============================================================
# ACTIVE COUNT
# ============================================================

page_text = clean_text(
    soup.get_text(
        "\n",
        strip=True
    )
)


active_match = re.search(
    r"(\d+)\s+aktif\s+kampanya\s+bulundu",
    page_text,
    flags=re.I
)


active_count = (
    int(
        active_match.group(1)
    )
    if active_match
    else None
)


print(
    "Aktif sayı:",
    active_count
)


# ============================================================
# DISCOVER ALL SLUGS
# ============================================================

methods, candidate_slugs = (
    discover_slugs(
        response.text,
        soup
    )
)


print()
print(
    "=" * 118
)

print(
    "SLUG DISCOVERY"
)

print(
    "=" * 118
)


for method, values in (
    methods.items()
):

    print(
        f"{method:<15}:",
        len(values)
    )


print(
    "UNION          :",
    len(
        candidate_slugs
    )
)


# ============================================================
# SAVE DISCOVERY
# ============================================================

discovery_output = {
    "aktif_kampanya_sayisi":
        active_count,

    "candidate_slug_count":
        len(
            candidate_slugs
        ),

    "methods":
        {
            key:
                sorted(
                    list(value)
                )
            for key, value
            in methods.items()
        },

    "candidate_slugs":
        sorted(
            candidate_slugs
        ),
}


with open(
    DISCOVERY_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        discovery_output,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# DETAIL VALIDATION
# ============================================================

print()
print(
    "=" * 118
)

print(
    "DETAIL VALIDATION / RAW"
)

print(
    "=" * 118
)


records = []
failed = []
not_ziraat = []


candidate_slugs = sorted(
    candidate_slugs
)


for index, slug in enumerate(
    candidate_slugs,
    start=1
):

    detail_url = (
        BASE
        + "/kampanya/"
        + slug
    )


    r, detail_soup = get_page(
        detail_url
    )


    if (
        r is None
        or
        r.status_code != 200
    ):

        failed.append({
            "slug":
                slug,

            "url":
                detail_url,

            "status":
                (
                    r.status_code
                    if r is not None
                    else None
                ),
        })

        continue


    official_urls = (
        extract_official_urls(
            detail_soup,
            detail_url
        )
    )


    # --------------------------------------------------------
    # KRİTİK FİLTRE
    #
    # Gerçek Ziraat Katılım resmi kampanya linki
    # yoksa bu kayıt kabul edilmez.
    # --------------------------------------------------------

    if not official_urls:

        not_ziraat.append(
            slug
        )

        continue


    h1 = detail_soup.find(
        "h1"
    )


    title = (
        clean_text(
            h1.get_text(
                " ",
                strip=True
            )
        )
        if h1
        else ""
    )


    raw_text = extract_detail_text(
        detail_soup
    )


    if len(raw_text) < 100:

        failed.append({
            "slug":
                slug,

            "url":
                detail_url,

            "status":
                r.status_code,

            "error":
                "ham_metin çok kısa",
        })

        continue


    period_context = (
        extract_period_context(
            raw_text
        )
    )


    dates = extract_dates(
        period_context
        if period_context
        else raw_text
    )


    record = {
        "kampanya_adi":
            title,

        # Final schema'da kullanacağımız kaynak:
        "kaynak_url":
            official_urls[0],

        # RAW provenance:
        "mirror_url":
            detail_url,

        "fetch_method":
            "kartavantaj_official_link_mirror",

        "http_status":
            r.status_code,

        "kampanya_donemi_raw":
            period_context,

        "tarih_adaylari":
            dates,

        "ham_metin":
            raw_text,
    }


    records.append(
        record
    )


    print(
        f"[{len(records):03d}] "
        f"{title}"
    )

    print(
        "      Official:",
        official_urls[0]
    )

    print(
        "      Dates   :",
        dates
    )


    time.sleep(
        0.12
    )


# ============================================================
# DEDUPE BY OFFICIAL URL
# ============================================================

by_url = {}


for record in records:

    url = record[
        "kaynak_url"
    ]


    old = by_url.get(
        url
    )


    if old is None:

        by_url[
            url
        ] = record

        continue


    # Daha uzun raw metin varsa onu tut.
    if (
        len(
            record[
                "ham_metin"
            ]
        )
        >
        len(
            old[
                "ham_metin"
            ]
        )
    ):

        by_url[
            url
        ] = record


records = list(
    by_url.values()
)


records.sort(
    key=lambda x:
        normalize(
            x[
                "kampanya_adi"
            ]
        )
)


# ============================================================
# AUDIT
# ============================================================

errors = []


if active_count is None:

    errors.append(
        (
            "Aktif kampanya sayısı "
            "sayfadan okunamadı."
        )
    )


if (
    active_count is not None
    and
    len(records) != active_count
):

    errors.append(
        (
            "Doğrulanmış kampanya "
            f"{len(records)} != "
            f"aktif sayaç {active_count}"
        )
    )


urls = [
    record[
        "kaynak_url"
    ]
    for record in records
]


duplicate_url = (
    len(urls)
    -
    len(
        set(urls)
    )
)


if duplicate_url:

    errors.append(
        (
            "Duplicate official URL: "
            f"{duplicate_url}"
        )
    )


empty_titles = sum(
    1
    for record in records
    if not record[
        "kampanya_adi"
    ]
)


if empty_titles:

    errors.append(
        (
            "Boş başlık: "
            f"{empty_titles}"
        )
    )


empty_raw = sum(
    1
    for record in records
    if not record[
        "ham_metin"
    ]
)


if empty_raw:

    errors.append(
        (
            "Boş ham_metin: "
            f"{empty_raw}"
        )
    )


# ============================================================
# OUTPUT
# ============================================================

output = {
    "banka":
        "Ziraat Katılım Bankası A.Ş.",

    "kaynak":
        (
            "KartAvantaj mirror + "
            "Ziraat Katılım resmi kaynak URL"
        ),

    "aktif_sayac":
        active_count,

    "candidate_slug":
        len(
            candidate_slugs
        ),

    "confirmed_campaign":
        len(
            records
        ),

    "not_ziraat":
        len(
            not_ziraat
        ),

    "fetch_failed":
        len(
            failed
        ),

    "duplicate_url":
        duplicate_url,

    "validation_error":
        len(
            errors
        ),

    "kampanyalar":
        records,

    "failed":
        failed,

    "not_ziraat_slugs":
        not_ziraat,

    "errors":
        errors,
}


with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# RESULT
# ============================================================

print()
print(
    "=" * 118
)

print(
    "ZİRAAT KATILIM - "
    "CAMPAIGN RAW V5 SONUCU"
)

print(
    "=" * 118
)

print(
    "Aktif sayaç       :",
    active_count
)

print(
    "Candidate slug    :",
    len(
        candidate_slugs
    )
)

print(
    "Confirmed Ziraat  :",
    len(
        records
    )
)

print(
    "Not Ziraat        :",
    len(
        not_ziraat
    )
)

print(
    "Fetch failed      :",
    len(
        failed
    )
)

print(
    "Duplicate URL     :",
    duplicate_url
)

print(
    "Validation error  :",
    len(
        errors
    )
)

print(
    "RAW JSON          :",
    OUTPUT_FILE
)

print(
    "Discovery JSON    :",
    DISCOVERY_FILE
)


# ============================================================
# DATE COVERAGE
# ============================================================

with_date = sum(
    1
    for record in records
    if record[
        "tarih_adaylari"
    ]
)


print(
    "Tarih bulunan     :",
    with_date,
    "/",
    len(records)
)


# ============================================================
# ERRORS
# ============================================================

if errors:

    print()
    print(
        "=" * 118
    )

    print(
        "HATALAR"
    )

    print(
        "=" * 118
    )

    for error in errors:

        print(
            "-",
            error
        )


# ============================================================
# FINAL
# ============================================================

print()
print(
    "=" * 118
)


if not errors:

    print(
        "SONUÇ: ZİRAAT KATILIM "
        "KAMPANYA RAW "
        f"{len(records)}/{active_count} "
        "BAŞARILI ✅"
    )

else:

    print(
        "SONUÇ: KAMPANYA RAW "
        "KONTROL GEREKİYOR ⚠️"
    )


print(
    "=" * 118
)


files.download(
    OUTPUT_FILE
)

files.download(
    DISCOVERY_FILE
)

ZİRAAT KATILIM - CAMPAIGN RAW SCRAPER V5
Bank HTTP: 200
HTML len : 1874773
Aktif sayı: 87

SLUG DISCOVERY
anchor         : 24
raw_path       : 41
escaped_path   : 0
unicode_path   : 0
encoded_path   : 0
json_url       : 0
json_slug      : 0
UNION          : 41

DETAIL VALIDATION / RAW
[001] Aile Kart'a Özel 2.000 TL'ye varan Bankkart Lira!
      Official: https://www.ziraatkatilim.com.tr/kart-kampanyalari/aile-karta-ozel-2000-tlye-varan-bankkart-lira-2
      Dates   : ['08 Ağustos 2026', '07 Eylül 2026']
[002] Akaryakıt Harcamalarınıza 400 TL Bankkart Lira!
      Official: https://www.ziraatkatilim.com.tr/kart-kampanyalari/akaryakit-harcamalariniza-400-tl-bankkart-lira-3
      Dates   : ['08 Ağustos 2026', '07 Eylül 2026']
[003] Atasun Optik'te 6 Taksit
      Official: https://www.ziraatkatilim.com.tr/kart-kampanyalari/atasun-optikte-6-taksit
      Dates   : ['11 Ağustos 2026', '31 Ağustos 2026']
[004] Bağımsız Kart'a Özel 5.000 TL'ye varan Bankkart Lira!
      Official: https://www.zi

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# ZİRAAT KATILIM
# CAMPAIGN RAW COMPLETER V6
#
# Mevcut 41 doğrulanmış RAW kaydı KORUR.
#
# Eksik kampanyaları:
#   /banka/ziraat-katilim/<sektor>
# sayfalarından keşfeder.
#
# Her yeni detay sayfası yine gerçek
# ziraatkatilim.com.tr/kart-kampanyalari/... URL'si ile
# doğrulanmadan kabul edilmez.
# ============================================================

!pip -q install requests==2.32.4 beautifulsoup4

import html
import json
import re
import time
import urllib.parse

import requests

from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

from google.colab import files


# ============================================================
# FILES / URLS
# ============================================================

INPUT_FILE = (
    "/content/"
    "ziraat_katilim_kampanyalar_raw.json"
)

OUTPUT_FILE = (
    "/content/"
    "ziraat_katilim_kampanyalar_raw_v6.json"
)

DISCOVERY_FILE = (
    "/content/"
    "ziraat_katilim_kampanya_sector_discovery_v6.json"
)


BASE = "https://kartavantaj.com"

BANK_PATH = (
    "/banka/ziraat-katilim"
)

BANK_URL = (
    BASE
    + BANK_PATH
)

EXPECTED_ACTIVE = 87


# ============================================================
# SESSION
# ============================================================

session = requests.Session()

session.headers.update(
    {
        "User-Agent": (
            "Mozilla/5.0 "
            "(Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 "
            "(KHTML, like Gecko) "
            "Chrome/151.0.0.0 "
            "Safari/537.36"
        ),

        "Accept-Language":
            "tr-TR,tr;q=0.9,en;q=0.8",
    }
)


# ============================================================
# HELPERS
# ============================================================

def clean_text(value):

    value = html.unescape(
        str(value or "")
    )

    value = (
        value
        .replace("\xa0", " ")
        .replace("’", "'")
        .replace("‘", "'")
        .replace("–", "-")
        .replace("—", "-")
    )

    value = re.sub(
        r"[ \t]+",
        " ",
        value
    )

    value = re.sub(
        r"\n[ \t]+",
        "\n",
        value
    )

    value = re.sub(
        r"\n{3,}",
        "\n\n",
        value
    )

    return value.strip()


def normalize(value):

    return (
        clean_text(value)
        .replace("İ", "i")
        .replace("I", "ı")
        .casefold()
    )


def get_page(url):

    try:

        response = session.get(
            url,
            timeout=40,
            allow_redirects=True
        )

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        return response, soup

    except Exception as error:

        print(
            "REQUEST ERROR:",
            url,
            type(error).__name__,
            error
        )

        return None, None


# ============================================================
# SLUG
# ============================================================

def clean_slug(value):

    value = urllib.parse.unquote(
        str(value or "")
    )

    value = (
        value
        .strip()
        .strip("/")
        .strip('"')
        .strip("'")
    )

    value = value.split(
        "?"
    )[0]

    value = value.split(
        "#"
    )[0]


    if not re.fullmatch(
        r"[A-Za-z0-9çğıöşüÇĞİÖŞÜ\-]+",
        value
    ):

        return None


    if len(value) < 2:

        return None


    return value


# ============================================================
# CAMPAIGN LINKS FROM ANY HTML
# ============================================================

def extract_campaign_slugs(
    html_text,
    soup
):

    slugs = set()


    # --------------------------------------------------------
    # Anchors
    # --------------------------------------------------------

    for a in soup.find_all(
        "a",
        href=True
    ):

        href = urljoin(
            BASE,
            a.get(
                "href",
                ""
            )
        )


        parsed = urlparse(
            href
        )


        if (
            parsed.netloc
            not in {
                "kartavantaj.com",
                "www.kartavantaj.com",
            }
        ):

            continue


        if not parsed.path.startswith(
            "/kampanya/"
        ):

            continue


        slug = clean_slug(
            parsed.path[
                len("/kampanya/"):
            ]
        )


        if slug:

            slugs.add(
                slug
            )


    # --------------------------------------------------------
    # Raw HTML paths
    # --------------------------------------------------------

    variants = [
        html_text,
        html.unescape(
            html_text
        ),
    ]


    try:

        variants.append(
            urllib.parse.unquote(
                html_text
            )
        )

    except Exception:

        pass


    for text in variants:

        for slug in re.findall(
            (
                r"/kampanya/"
                r"([A-Za-z0-9çğıöşüÇĞİÖŞÜ\-]+)"
            ),
            text
        ):

            slug = clean_slug(
                slug
            )

            if slug:

                slugs.add(
                    slug
                )


    return slugs


# ============================================================
# SECTOR URL DISCOVERY
# ============================================================

def extract_sector_urls(
    html_text,
    soup
):

    urls = set()


    # --------------------------------------------------------
    # Anchors
    # --------------------------------------------------------

    for a in soup.find_all(
        "a",
        href=True
    ):

        href = urljoin(
            BASE,
            a.get(
                "href",
                ""
            )
        )


        parsed = urlparse(
            href
        )


        if (
            parsed.netloc
            not in {
                "kartavantaj.com",
                "www.kartavantaj.com",
            }
        ):

            continue


        path = (
            parsed.path
            .rstrip("/")
        )


        prefix = (
            BANK_PATH
            + "/"
        )


        if not path.startswith(
            prefix
        ):

            continue


        rest = path[
            len(prefix):
        ]


        # Bir seviye sektör olsun.
        if (
            not rest
            or
            "/" in rest
        ):

            continue


        urls.add(
            BASE
            + path
        )


    # --------------------------------------------------------
    # Raw HTML:
    # /banka/ziraat-katilim/elektronik
    # --------------------------------------------------------

    variants = [
        html_text,
        html.unescape(
            html_text
        ),
    ]


    for text in variants:

        matches = re.findall(
            (
                r"/banka/"
                r"ziraat-katilim/"
                r"([A-Za-z0-9çğıöşüÇĞİÖŞÜ\-]+)"
            ),
            text
        )


        for sector in matches:

            sector = clean_slug(
                sector
            )

            if not sector:
                continue


            urls.add(
                (
                    BASE
                    + BANK_PATH
                    + "/"
                    + sector
                )
            )


    return urls


# ============================================================
# FALLBACK SECTOR URLS
#
# Bunlar yalnızca discovery amaçlı.
# 404 olursa otomatik atlanır.
# ============================================================

FALLBACK_SECTORS = [
    "akaryakit",
    "alisveris",
    "egitim",
    "elektronik",
    "e-ticaret",
    "giyim",
    "market-gida",
    "mobilya",
    "ev-yasam",
    "saglik",
    "seyahat",
    "ulasim",
    "otomotiv",
    "restoran",
    "yeme-icme",
    "kuyum",
    "sigorta",
    "dijital-platform",
    "eglence",
    "kultur-sanat",
    "kozmetik",
    "petshop",
]


# ============================================================
# OFFICIAL URL
# ============================================================

def extract_official_urls(
    soup,
    detail_url
):

    result = []


    for a in soup.find_all(
        "a",
        href=True
    ):

        href = urljoin(
            detail_url,
            a.get(
                "href",
                ""
            )
        )


        parsed = urlparse(
            href
        )


        if (
            parsed.netloc.lower()
            not in {
                "ziraatkatilim.com.tr",
                "www.ziraatkatilim.com.tr",
            }
        ):

            continue


        prefix = (
            "/kart-kampanyalari/"
        )


        if not parsed.path.startswith(
            prefix
        ):

            continue


        slug = clean_slug(
            parsed.path[
                len(prefix):
            ]
        )


        if not slug:

            continue


        url = (
            "https://www."
            "ziraatkatilim.com.tr/"
            "kart-kampanyalari/"
            + slug
        )


        if url not in result:

            result.append(
                url
            )


    return result


# ============================================================
# DETAIL TEXT
# ============================================================

def extract_detail_text(soup):

    clone = BeautifulSoup(
        str(soup),
        "html.parser"
    )


    for selector in [
        "script",
        "style",
        "noscript",
        "svg",
        "header",
        "footer",
        "nav",
        "form",
    ]:

        for tag in clone.select(
            selector
        ):

            tag.decompose()


    main = clone.find(
        "main"
    )


    if main is None:

        h1 = clone.find(
            "h1"
        )

        if h1:

            current = h1

            for _ in range(7):

                if current is None:
                    break


                text = clean_text(
                    current.get_text(
                        "\n",
                        strip=True
                    )
                )


                if len(text) >= 350:

                    main = current
                    break


                current = current.parent


    if main is None:

        main = clone.body


    if main is None:

        return ""


    return clean_text(
        main.get_text(
            "\n",
            strip=True
        )
    )


# ============================================================
# DATES
# ============================================================

MONTHS = (
    "Ocak|Şubat|Mart|Nisan|Mayıs|Haziran|"
    "Temmuz|Ağustos|Eylül|Ekim|Kasım|Aralık"
)


def extract_dates(text):

    result = []


    patterns = [
        (
            rf"\b"
            rf"\d{{1,2}}\s+"
            rf"(?:{MONTHS})\s+"
            rf"\d{{4}}\b"
        ),

        (
            r"\b"
            r"\d{1,2}[./-]"
            r"\d{1,2}[./-]"
            r"\d{4}"
            r"\b"
        ),
    ]


    for pattern in patterns:

        for match in re.finditer(
            pattern,
            text,
            flags=re.I
        ):

            value = clean_text(
                match.group(0)
            )


            if (
                value
                and
                value not in result
            ):

                result.append(
                    value
                )


    return result


def extract_period_context(text):

    match = re.search(
        r"Kampanya\s+Dönemi",
        text,
        flags=re.I
    )


    if not match:

        return ""


    start = match.start()

    return clean_text(
        text[
            start:
            start + 350
        ]
    )


# ============================================================
# LOAD EXISTING 41
# ============================================================

with open(
    INPUT_FILE,
    "r",
    encoding="utf-8"
) as f:

    old_data = json.load(
        f
    )


records = old_data.get(
    "kampanyalar",
    []
)


print(
    "=" * 118
)

print(
    "ZİRAAT KATILIM - "
    "CAMPAIGN RAW COMPLETER V6"
)

print(
    "=" * 118
)

print(
    "Mevcut confirmed:",
    len(records)
)


existing_by_official = {
    record[
        "kaynak_url"
    ]:
    record
    for record in records
}


existing_mirror_slugs = set()


for record in records:

    mirror = record.get(
        "mirror_url",
        ""
    )


    if "/kampanya/" in mirror:

        slug = clean_slug(
            mirror.split(
                "/kampanya/",
                1
            )[1]
        )


        if slug:

            existing_mirror_slugs.add(
                slug
            )


# ============================================================
# FETCH MAIN PAGE
# ============================================================

response, soup = get_page(
    BANK_URL
)


if (
    response is None
    or
    response.status_code != 200
):

    raise RuntimeError(
        "KartAvantaj banka sayfası alınamadı."
    )


page_text = clean_text(
    soup.get_text(
        "\n",
        strip=True
    )
)


count_match = re.search(
    r"(\d+)\s+aktif\s+kampanya\s+bulundu",
    page_text,
    flags=re.I
)


active_count = (
    int(
        count_match.group(1)
    )
    if count_match
    else None
)


print(
    "Aktif sayaç:",
    active_count
)


# ============================================================
# DISCOVER SECTOR URLS
# ============================================================

sector_urls = extract_sector_urls(
    response.text,
    soup
)


for sector in FALLBACK_SECTORS:

    sector_urls.add(
        (
            BASE
            + BANK_PATH
            + "/"
            + sector
        )
    )


sector_urls = sorted(
    sector_urls
)


print()
print(
    "=" * 118
)

print(
    "SECTOR DISCOVERY"
)

print(
    "=" * 118
)

print(
    "Sector URL candidate:",
    len(sector_urls)
)


# ============================================================
# CRAWL SECTORS
# ============================================================

all_candidate_slugs = set(
    existing_mirror_slugs
)

sector_logs = []


for index, url in enumerate(
    sector_urls,
    start=1
):

    r, sector_soup = get_page(
        url
    )


    if r is None:

        sector_logs.append({
            "url":
                url,

            "status":
                None,

            "campaign_slugs":
                0,
        })

        continue


    if r.status_code != 200:

        print(
            f"[{index:02d}] "
            f"{r.status_code} "
            f"{url}"
        )

        sector_logs.append({
            "url":
                url,

            "status":
                r.status_code,

            "campaign_slugs":
                0,
        })

        continue


    h1 = sector_soup.find(
        "h1"
    )


    h1_text = (
        clean_text(
            h1.get_text(
                " ",
                strip=True
            )
        )
        if h1
        else ""
    )


    slugs = extract_campaign_slugs(
        r.text,
        sector_soup
    )


    all_candidate_slugs.update(
        slugs
    )


    sector_logs.append({
        "url":
            url,

        "status":
            200,

        "h1":
            h1_text,

        "campaign_slugs":
            len(slugs),
    })


    print(
        f"[{index:02d}] "
        f"200 | "
        f"{len(slugs):02d} campaigns | "
        f"{h1_text}"
    )


    time.sleep(
        0.1
    )


# ============================================================
# UNION SUMMARY
# ============================================================

print()
print(
    "=" * 118
)

print(
    "SECTOR UNION"
)

print(
    "=" * 118
)

print(
    "Mevcut slug       :",
    len(existing_mirror_slugs)
)

print(
    "Union candidate   :",
    len(all_candidate_slugs)
)

print(
    "Yeni slug adayı   :",
    len(
        all_candidate_slugs
        -
        existing_mirror_slugs
    )
)


# ============================================================
# FETCH ONLY NEW DETAIL SLUGS
# ============================================================

new_slugs = sorted(
    all_candidate_slugs
    -
    existing_mirror_slugs
)


added = []
not_ziraat = []
failed = []


for index, slug in enumerate(
    new_slugs,
    start=1
):

    detail_url = (
        BASE
        + "/kampanya/"
        + slug
    )


    r, detail_soup = get_page(
        detail_url
    )


    if (
        r is None
        or
        r.status_code != 200
    ):

        failed.append({
            "slug":
                slug,

            "status":
                (
                    r.status_code
                    if r is not None
                    else None
                ),
        })

        continue


    official_urls = (
        extract_official_urls(
            detail_soup,
            detail_url
        )
    )


    # Ziraat official URL yoksa reddet.
    if not official_urls:

        not_ziraat.append(
            slug
        )

        continue


    raw_text = extract_detail_text(
        detail_soup
    )


    if len(raw_text) < 100:

        failed.append({
            "slug":
                slug,

            "status":
                200,

            "error":
                "raw text too short",
        })

        continue


    h1 = detail_soup.find(
        "h1"
    )


    title = (
        clean_text(
            h1.get_text(
                " ",
                strip=True
            )
        )
        if h1
        else ""
    )


    period = extract_period_context(
        raw_text
    )


    dates = extract_dates(
        period
        if period
        else raw_text
    )


    record = {
        "kampanya_adi":
            title,

        "kaynak_url":
            official_urls[0],

        "mirror_url":
            detail_url,

        "fetch_method":
            "kartavantaj_official_link_mirror",

        "http_status":
            200,

        "kampanya_donemi_raw":
            period,

        "tarih_adaylari":
            dates,

        "ham_metin":
            raw_text,
    }


    official = record[
        "kaynak_url"
    ]


    # Aynı official kampanya zaten varsa
    # duplicate olarak eklemiyoruz.
    if official in existing_by_official:

        continue


    existing_by_official[
        official
    ] = record

    added.append(
        record
    )


    print(
        f"[NEW {len(added):03d}] "
        f"{title}"
    )

    print(
        "       Official:",
        official
    )

    print(
        "       Dates   :",
        dates
    )


    time.sleep(
        0.1
    )


# ============================================================
# FINAL RECORDS
# ============================================================

records = list(
    existing_by_official.values()
)


records.sort(
    key=lambda item:
        normalize(
            item[
                "kampanya_adi"
            ]
        )
)


# ============================================================
# AUDIT
# ============================================================

errors = []


urls = [
    record[
        "kaynak_url"
    ]
    for record in records
]


duplicate_url = (
    len(urls)
    -
    len(
        set(urls)
    )
)


if duplicate_url:

    errors.append(
        (
            "Duplicate official URL: "
            f"{duplicate_url}"
        )
    )


empty_title = sum(
    1
    for record in records
    if not record[
        "kampanya_adi"
    ]
)


if empty_title:

    errors.append(
        (
            "Boş başlık: "
            f"{empty_title}"
        )
    )


empty_raw = sum(
    1
    for record in records
    if not record[
        "ham_metin"
    ]
)


if empty_raw:

    errors.append(
        (
            "Boş raw: "
            f"{empty_raw}"
        )
    )


if (
    active_count is not None
    and
    len(records) != active_count
):

    errors.append(
        (
            f"Confirmed {len(records)} "
            f"!= aktif sayaç {active_count}"
        )
    )


with_date = sum(
    1
    for record in records
    if record[
        "tarih_adaylari"
    ]
)


# ============================================================
# SAVE
# ============================================================

output = {
    "banka":
        "Ziraat Katılım Bankası A.Ş.",

    "aktif_sayac":
        active_count,

    "eski_confirmed":
        len(
            old_data.get(
                "kampanyalar",
                []
            )
        ),

    "yeni_eklenen":
        len(
            added
        ),

    "confirmed_campaign":
        len(
            records
        ),

    "sector_url_count":
        len(
            sector_urls
        ),

    "candidate_slug_union":
        len(
            all_candidate_slugs
        ),

    "not_ziraat":
        len(
            not_ziraat
        ),

    "fetch_failed":
        len(
            failed
        ),

    "duplicate_url":
        duplicate_url,

    "validation_error":
        len(
            errors
        ),

    "kampanyalar":
        records,

    "errors":
        errors,
}


with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output,
        f,
        ensure_ascii=False,
        indent=2
    )


with open(
    DISCOVERY_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "sector_logs":
                sector_logs,

            "candidate_slugs":
                sorted(
                    all_candidate_slugs
                ),

            "new_slugs":
                new_slugs,

            "not_ziraat":
                not_ziraat,

            "failed":
                failed,
        },
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# RESULT
# ============================================================

print()
print(
    "=" * 118
)

print(
    "ZİRAAT KATILIM - "
    "CAMPAIGN RAW V6 SONUCU"
)

print(
    "=" * 118
)

print(
    "Aktif sayaç       :",
    active_count
)

print(
    "Eski confirmed    :",
    len(
        old_data.get(
            "kampanyalar",
            []
        )
    )
)

print(
    "Sector URL        :",
    len(
        sector_urls
    )
)

print(
    "Candidate union   :",
    len(
        all_candidate_slugs
    )
)

print(
    "Yeni eklenen      :",
    len(
        added
    )
)

print(
    "Confirmed Ziraat  :",
    len(
        records
    )
)

print(
    "Not Ziraat        :",
    len(
        not_ziraat
    )
)

print(
    "Fetch failed      :",
    len(
        failed
    )
)

print(
    "Duplicate URL     :",
    duplicate_url
)

print(
    "Tarih bulunan     :",
    with_date,
    "/",
    len(records)
)

print(
    "Validation error  :",
    len(
        errors
    )
)

print(
    "RAW JSON          :",
    OUTPUT_FILE
)


if errors:

    print()
    print(
        "=" * 118
    )

    print(
        "HATALAR"
    )

    print(
        "=" * 118
    )


    for error in errors:

        print(
            "-",
            error
        )


print()
print(
    "=" * 118
)


if not errors:

    print(
        "SONUÇ: ZİRAAT KATILIM "
        f"KAMPANYA RAW "
        f"{len(records)}/{active_count} "
        "BAŞARILI ✅"
    )

else:

    print(
        "SONUÇ: KAMPANYA RAW "
        "HALA TAMAMLANMADI ⚠️"
    )


print(
    "=" * 118
)


files.download(
    OUTPUT_FILE
)

files.download(
    DISCOVERY_FILE
)

ZİRAAT KATILIM - CAMPAIGN RAW COMPLETER V6
Mevcut confirmed: 41
Aktif sayaç: 87

SECTOR DISCOVERY
Sector URL candidate: 22
[01] 200 | 01 campaigns | Ziraat Katılım Akaryakıt Kampanyaları
[02] 404 https://kartavantaj.com/banka/ziraat-katilim/alisveris
[03] 200 | 00 campaigns | Ziraat Katılım Dijital Platform Kampanyaları
[04] 200 | 05 campaigns | Ziraat Katılım E-Ticaret Kampanyaları
[05] 200 | 03 campaigns | Ziraat Katılım Eğitim Kampanyaları
[06] 404 https://kartavantaj.com/banka/ziraat-katilim/eglence
[07] 200 | 11 campaigns | Ziraat Katılım Elektronik Kampanyaları
[08] 404 https://kartavantaj.com/banka/ziraat-katilim/ev-yasam
[09] 404 https://kartavantaj.com/banka/ziraat-katilim/giyim
[10] 404 https://kartavantaj.com/banka/ziraat-katilim/kozmetik
[11] 200 | 00 campaigns | Ziraat Katılım Kültür, Sanat & Spor Kampanyaları
[12] 404 https://kartavantaj.com/banka/ziraat-katilim/kuyum
[13] 200 | 04 campaigns | Ziraat Katılım Market & Gıda Kampanyaları
[14] 404 https://kartavantaj.com/bank

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# ZİRAAT KATILIM
# CAMPAIGN RAW COMPLETER V7
#
# FIX:
# V6'daki tahmini sektör slug'ları kaldırıldı.
# KartAvantaj'ın gerçek kategori slug'ları kullanılıyor.
#
# Input:
#   /content/ziraat_katilim_kampanyalar_raw_v6.json
#
# Output:
#   /content/ziraat_katilim_kampanyalar_raw_v7.json
#
# Hedef:
#   87 / 87
# ============================================================

!pip -q install requests==2.32.4 beautifulsoup4

import html
import json
import re
import time
import urllib.parse
import requests

from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from google.colab import files


# ============================================================
# FILES
# ============================================================

INPUT_FILE = (
    "/content/"
    "ziraat_katilim_kampanyalar_raw_v6.json"
)

OUTPUT_FILE = (
    "/content/"
    "ziraat_katilim_kampanyalar_raw_v7.json"
)

DISCOVERY_FILE = (
    "/content/"
    "ziraat_katilim_kampanya_sector_discovery_v7.json"
)


BASE = "https://kartavantaj.com"

BANK_BASE = (
    BASE
    + "/banka/ziraat-katilim"
)

EXPECTED_ACTIVE = 87


# ============================================================
# GERÇEK KARTAVANTAJ KATEGORİ SLUG'LARI
# ============================================================

SECTORS = [
    "akaryakit",
    "anne-bebek-oyuncak",
    "diger",
    "dijital-platform",
    "egitim",
    "elektronik",
    "e-ticaret",
    "evcil-hayvan-petshop",
    "fatura-telekomunikasyon",
    "finans-yatirim",
    "giyim-aksesuar",
    "hizmet-bireysel-gelisim",
    "kitap-kirtasiye-ofis",
    "kozmetik-saglik",
    "kultur-sanat-spor",
    "market-gida",
    "mobilya-dekorasyon",
    "mucevherat-optik-saat",
    "otomotiv",
    "restoran-kafe",
    "sigorta",
    "turizm-konaklama",
    "ulasim",
    "vergi-kamu",
]


# ============================================================
# SESSION
# ============================================================

session = requests.Session()

session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/151.0.0.0 "
        "Safari/537.36"
    ),
    "Accept-Language":
        "tr-TR,tr;q=0.9,en;q=0.8",
})


# ============================================================
# HELPERS
# ============================================================

def clean_text(value):

    value = html.unescape(
        str(value or "")
    )

    value = (
        value
        .replace("\xa0", " ")
        .replace("’", "'")
        .replace("‘", "'")
        .replace("–", "-")
        .replace("—", "-")
    )

    value = re.sub(
        r"[ \t]+",
        " ",
        value
    )

    value = re.sub(
        r"\n[ \t]+",
        "\n",
        value
    )

    value = re.sub(
        r"\n{3,}",
        "\n\n",
        value
    )

    return value.strip()


def normalize(value):

    return (
        clean_text(value)
        .replace("İ", "i")
        .replace("I", "ı")
        .casefold()
    )


def get_page(url):

    try:

        response = session.get(
            url,
            timeout=40,
            allow_redirects=True
        )

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        return response, soup

    except Exception as error:

        print(
            "REQUEST ERROR:",
            url,
            type(error).__name__,
            error
        )

        return None, None


def clean_slug(value):

    value = urllib.parse.unquote(
        str(value or "")
    )

    value = (
        value
        .strip()
        .strip("/")
        .strip('"')
        .strip("'")
    )

    value = value.split("?")[0]
    value = value.split("#")[0]

    if not re.fullmatch(
        r"[A-Za-z0-9çğıöşüÇĞİÖŞÜ\-]+",
        value
    ):
        return None

    if len(value) < 2:
        return None

    return value


# ============================================================
# CAMPAIGN SLUGS
# ============================================================

def extract_campaign_slugs(
    html_text,
    soup
):

    result = set()


    # ---------------------------
    # <a href>
    # ---------------------------

    for a in soup.find_all(
        "a",
        href=True
    ):

        href = urljoin(
            BASE,
            a.get(
                "href",
                ""
            )
        )

        parsed = urlparse(
            href
        )

        if (
            parsed.netloc
            not in {
                "kartavantaj.com",
                "www.kartavantaj.com",
            }
        ):
            continue

        prefix = "/kampanya/"

        if not parsed.path.startswith(
            prefix
        ):
            continue

        slug = clean_slug(
            parsed.path[
                len(prefix):
            ]
        )

        if slug:
            result.add(slug)


    # ---------------------------
    # RAW HTML
    # ---------------------------

    variants = [
        html_text,
        html.unescape(html_text),
    ]

    try:
        variants.append(
            urllib.parse.unquote(
                html_text
            )
        )
    except Exception:
        pass


    for text in variants:

        matches = re.findall(
            (
                r"/kampanya/"
                r"([A-Za-z0-9çğıöşüÇĞİÖŞÜ\-]+)"
            ),
            text
        )

        for slug in matches:

            slug = clean_slug(slug)

            if slug:
                result.add(slug)


    return result


# ============================================================
# OFFICIAL ZIRAAT URL
# ============================================================

def extract_official_urls(
    soup,
    detail_url
):

    result = []


    for a in soup.find_all(
        "a",
        href=True
    ):

        href = urljoin(
            detail_url,
            a.get(
                "href",
                ""
            )
        )

        parsed = urlparse(
            href
        )

        if (
            parsed.netloc.lower()
            not in {
                "ziraatkatilim.com.tr",
                "www.ziraatkatilim.com.tr",
            }
        ):
            continue


        prefix = (
            "/kart-kampanyalari/"
        )

        if not parsed.path.startswith(
            prefix
        ):
            continue


        slug = clean_slug(
            parsed.path[
                len(prefix):
            ]
        )

        if not slug:
            continue


        official = (
            "https://www."
            "ziraatkatilim.com.tr/"
            "kart-kampanyalari/"
            + slug
        )


        if official not in result:
            result.append(
                official
            )


    return result


# ============================================================
# DETAIL TEXT
# ============================================================

def extract_detail_text(soup):

    clone = BeautifulSoup(
        str(soup),
        "html.parser"
    )


    for selector in [
        "script",
        "style",
        "noscript",
        "svg",
        "header",
        "footer",
        "nav",
        "form",
    ]:

        for tag in clone.select(
            selector
        ):
            tag.decompose()


    main = clone.find(
        "main"
    )


    if main is None:

        h1 = clone.find(
            "h1"
        )

        if h1:

            current = h1

            for _ in range(7):

                if current is None:
                    break

                text = clean_text(
                    current.get_text(
                        "\n",
                        strip=True
                    )
                )

                if len(text) >= 350:

                    main = current
                    break

                current = current.parent


    if main is None:
        main = clone.body


    if main is None:
        return ""


    return clean_text(
        main.get_text(
            "\n",
            strip=True
        )
    )


# ============================================================
# DATES
# ============================================================

MONTHS = (
    "Ocak|Şubat|Mart|Nisan|Mayıs|Haziran|"
    "Temmuz|Ağustos|Eylül|Ekim|Kasım|Aralık"
)


def extract_dates(text):

    result = []


    patterns = [
        (
            rf"\b"
            rf"\d{{1,2}}\s+"
            rf"(?:{MONTHS})\s+"
            rf"\d{{4}}\b"
        ),

        (
            r"\b"
            r"\d{1,2}[./-]"
            r"\d{1,2}[./-]"
            r"\d{4}"
            r"\b"
        ),
    ]


    for pattern in patterns:

        for match in re.finditer(
            pattern,
            text,
            flags=re.I
        ):

            value = clean_text(
                match.group(0)
            )

            if (
                value
                and
                value not in result
            ):
                result.append(
                    value
                )


    return result


def extract_period_context(text):

    match = re.search(
        r"Kampanya\s+Dönemi",
        text,
        flags=re.I
    )

    if not match:
        return ""


    return clean_text(
        text[
            match.start():
            match.start() + 400
        ]
    )


# ============================================================
# LOAD EXISTING 52
# ============================================================

with open(
    INPUT_FILE,
    "r",
    encoding="utf-8"
) as f:

    old_data = json.load(f)


old_records = old_data.get(
    "kampanyalar",
    []
)


print("=" * 118)
print(
    "ZİRAAT KATILIM - "
    "CAMPAIGN RAW COMPLETER V7"
)
print("=" * 118)

print(
    "Mevcut confirmed:",
    len(old_records)
)


existing_by_official = {
    record[
        "kaynak_url"
    ]:
    record
    for record in old_records
}


existing_mirror_slugs = set()


for record in old_records:

    mirror = record.get(
        "mirror_url",
        ""
    )

    if "/kampanya/" not in mirror:
        continue

    slug = clean_slug(
        mirror.split(
            "/kampanya/",
            1
        )[1]
    )

    if slug:
        existing_mirror_slugs.add(
            slug
        )


# ============================================================
# FETCH MAIN COUNT
# ============================================================

r_main, soup_main = get_page(
    BANK_BASE
)


if (
    r_main is None
    or
    r_main.status_code != 200
):
    raise RuntimeError(
        "Ana KartAvantaj sayfası alınamadı."
    )


main_text = clean_text(
    soup_main.get_text(
        "\n",
        strip=True
    )
)


count_match = re.search(
    r"(\d+)\s+aktif\s+kampanya\s+bulundu",
    main_text,
    flags=re.I
)


active_count = (
    int(
        count_match.group(1)
    )
    if count_match
    else None
)


print(
    "Aktif sayaç:",
    active_count
)


# ============================================================
# CRAWL ALL REAL SECTORS
# ============================================================

print()
print("=" * 118)
print("GERÇEK SEKTÖR SAYFALARI")
print("=" * 118)


all_candidate_slugs = set(
    existing_mirror_slugs
)

sector_logs = []


for index, sector in enumerate(
    SECTORS,
    start=1
):

    url = (
        BANK_BASE
        + "/"
        + sector
    )


    r, soup = get_page(
        url
    )


    if r is None:

        print(
            f"[{index:02d}] ERROR | "
            f"{sector}"
        )

        sector_logs.append({
            "sector":
                sector,

            "url":
                url,

            "status":
                None,

            "counter":
                None,

            "slug_count":
                0,
        })

        continue


    if r.status_code != 200:

        print(
            f"[{index:02d}] "
            f"HTTP {r.status_code} | "
            f"{sector}"
        )

        sector_logs.append({
            "sector":
                sector,

            "url":
                url,

            "status":
                r.status_code,

            "counter":
                None,

            "slug_count":
                0,
        })

        continue


    page_text = clean_text(
        soup.get_text(
            "\n",
            strip=True
        )
    )


    h1 = soup.find("h1")

    h1_text = (
        clean_text(
            h1.get_text(
                " ",
                strip=True
            )
        )
        if h1
        else ""
    )


    count_match = re.search(
        (
            r"(\d+)\s+"
            r"aktif\s+kampanya\s+bulundu"
        ),
        page_text,
        flags=re.I
    )


    counter = (
        int(
            count_match.group(1)
        )
        if count_match
        else None
    )


    slugs = extract_campaign_slugs(
        r.text,
        soup
    )


    all_candidate_slugs.update(
        slugs
    )


    sector_logs.append({
        "sector":
            sector,

        "url":
            url,

        "status":
            200,

        "h1":
            h1_text,

        "counter":
            counter,

        "slug_count":
            len(slugs),
    })


    print(
        f"[{index:02d}] "
        f"{len(slugs):02d} slug "
        f"| sayaç={counter} "
        f"| {h1_text}"
    )


    time.sleep(
        0.1
    )


# ============================================================
# UNION
# ============================================================

new_slugs = (
    all_candidate_slugs
    -
    existing_mirror_slugs
)


print()
print("=" * 118)
print("SECTOR UNION V7")
print("=" * 118)

print(
    "Mevcut slug      :",
    len(
        existing_mirror_slugs
    )
)

print(
    "Union candidate  :",
    len(
        all_candidate_slugs
    )
)

print(
    "Yeni slug adayı  :",
    len(
        new_slugs
    )
)


# ============================================================
# FETCH NEW DETAILS ONLY
# ============================================================

print()
print("=" * 118)
print("YENİ DETAIL DOĞRULAMA")
print("=" * 118)


added = []
not_ziraat = []
failed = []


for slug in sorted(
    new_slugs
):

    detail_url = (
        BASE
        + "/kampanya/"
        + slug
    )


    r, soup = get_page(
        detail_url
    )


    if (
        r is None
        or
        r.status_code != 200
    ):

        failed.append({
            "slug":
                slug,

            "status":
                (
                    r.status_code
                    if r is not None
                    else None
                ),
        })

        continue


    official_urls = (
        extract_official_urls(
            soup,
            detail_url
        )
    )


    # ----------------------------------------
    # Resmi Ziraat linki olmadan kabul YOK.
    # ----------------------------------------

    if not official_urls:

        not_ziraat.append(
            slug
        )

        continue


    h1 = soup.find(
        "h1"
    )


    title = (
        clean_text(
            h1.get_text(
                " ",
                strip=True
            )
        )
        if h1
        else ""
    )


    raw_text = extract_detail_text(
        soup
    )


    if (
        not title
        or
        len(raw_text) < 100
    ):

        failed.append({
            "slug":
                slug,

            "status":
                200,

            "error":
                "title/raw invalid",
        })

        continue


    period = extract_period_context(
        raw_text
    )


    dates = extract_dates(
        period
        if period
        else raw_text
    )


    record = {
        "kampanya_adi":
            title,

        "kaynak_url":
            official_urls[0],

        "mirror_url":
            detail_url,

        "fetch_method":
            (
                "kartavantaj_"
                "official_link_mirror"
            ),

        "http_status":
            200,

        "kampanya_donemi_raw":
            period,

        "tarih_adaylari":
            dates,

        "ham_metin":
            raw_text,
    }


    official = record[
        "kaynak_url"
    ]


    if official in existing_by_official:
        continue


    existing_by_official[
        official
    ] = record


    added.append(
        record
    )


    print(
        f"[NEW {len(added):03d}] "
        f"{title}"
    )

    print(
        "      Official:",
        official
    )

    print(
        "      Dates   :",
        dates
    )


    time.sleep(
        0.1
    )


# ============================================================
# FINAL
# ============================================================

records = list(
    existing_by_official.values()
)


records.sort(
    key=lambda item:
        normalize(
            item.get(
                "kampanya_adi",
                ""
            )
        )
)


# ============================================================
# AUDIT
# ============================================================

errors = []


urls = [
    record[
        "kaynak_url"
    ]
    for record in records
]


duplicate_url = (
    len(urls)
    -
    len(
        set(urls)
    )
)


if duplicate_url:

    errors.append(
        (
            "Duplicate URL: "
            f"{duplicate_url}"
        )
    )


names = [
    record[
        "kampanya_adi"
    ]
    for record in records
]


empty_title = sum(
    1
    for x in names
    if not x
)


if empty_title:

    errors.append(
        (
            "Boş başlık: "
            f"{empty_title}"
        )
    )


empty_raw = sum(
    1
    for record in records
    if not record[
        "ham_metin"
    ]
)


if empty_raw:

    errors.append(
        (
            "Boş ham_metin: "
            f"{empty_raw}"
        )
    )


with_date = sum(
    1
    for record in records
    if record[
        "tarih_adaylari"
    ]
)


if (
    active_count is not None
    and
    len(records) != active_count
):

    errors.append(
        (
            f"Confirmed {len(records)} "
            f"!= aktif sayaç "
            f"{active_count}"
        )
    )


# ============================================================
# SECTOR DIAGNOSTIC
#
# Eğer hâlâ eksikse hangi sektör sayfasının
# kendi sayacından az slug verdiğini direkt gösterir.
# ============================================================

sector_mismatches = []


for item in sector_logs:

    counter = item[
        "counter"
    ]

    slug_count = item[
        "slug_count"
    ]


    if (
        counter is not None
        and
        slug_count < counter
    ):

        sector_mismatches.append({
            "sector":
                item[
                    "sector"
                ],

            "counter":
                counter,

            "slug_count":
                slug_count,
        })


# ============================================================
# SAVE
# ============================================================

output = {
    "banka":
        "Ziraat Katılım Bankası A.Ş.",

    "aktif_sayac":
        active_count,

    "eski_confirmed":
        len(
            old_records
        ),

    "yeni_eklenen":
        len(
            added
        ),

    "confirmed_campaign":
        len(
            records
        ),

    "sector_count":
        len(
            SECTORS
        ),

    "candidate_slug_union":
        len(
            all_candidate_slugs
        ),

    "not_ziraat":
        len(
            not_ziraat
        ),

    "fetch_failed":
        len(
            failed
        ),

    "duplicate_url":
        duplicate_url,

    "tarih_bulunan":
        with_date,

    "validation_error":
        len(
            errors
        ),

    "sector_mismatches":
        sector_mismatches,

    "kampanyalar":
        records,

    "errors":
        errors,
}


with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output,
        f,
        ensure_ascii=False,
        indent=2
    )


with open(
    DISCOVERY_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "sector_logs":
                sector_logs,

            "sector_mismatches":
                sector_mismatches,

            "candidate_slugs":
                sorted(
                    all_candidate_slugs
                ),

            "new_slugs":
                sorted(
                    new_slugs
                ),

            "not_ziraat":
                not_ziraat,

            "failed":
                failed,
        },
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# RESULT
# ============================================================

print()
print("=" * 118)
print(
    "ZİRAAT KATILIM - "
    "CAMPAIGN RAW V7 SONUCU"
)
print("=" * 118)

print(
    "Aktif sayaç       :",
    active_count
)

print(
    "Eski confirmed    :",
    len(
        old_records
    )
)

print(
    "Gerçek sektör     :",
    len(
        SECTORS
    )
)

print(
    "Candidate union   :",
    len(
        all_candidate_slugs
    )
)

print(
    "Yeni eklenen      :",
    len(
        added
    )
)

print(
    "Confirmed Ziraat  :",
    len(
        records
    )
)

print(
    "Not Ziraat        :",
    len(
        not_ziraat
    )
)

print(
    "Fetch failed      :",
    len(
        failed
    )
)

print(
    "Duplicate URL     :",
    duplicate_url
)

print(
    "Tarih bulunan     :",
    with_date,
    "/",
    len(
        records
    )
)

print(
    "Validation error  :",
    len(
        errors
    )
)

print(
    "RAW JSON          :",
    OUTPUT_FILE
)


# ============================================================
# SECTOR MISMATCH
# ============================================================

print()
print("=" * 118)
print("SECTOR MISMATCH")
print("=" * 118)


if not sector_mismatches:

    print(
        "[]"
    )

else:

    for item in sector_mismatches:

        print(
            (
                f"- {item['sector']} "
                f"| sayaç={item['counter']} "
                f"| slug={item['slug_count']}"
            )
        )


# ============================================================
# ERRORS
# ============================================================

if errors:

    print()
    print("=" * 118)
    print("HATALAR")
    print("=" * 118)

    for error in errors:
        print(
            "-",
            error
        )


print()
print("=" * 118)


if not errors:

    print(
        "SONUÇ: ZİRAAT KATILIM "
        "KAMPANYA RAW "
        f"{len(records)}/{active_count} "
        "BAŞARILI ✅"
    )

else:

    print(
        "SONUÇ: KAMPANYA RAW "
        "KONTROL GEREKİYOR ⚠️"
    )


print("=" * 118)


files.download(
    OUTPUT_FILE
)

files.download(
    DISCOVERY_FILE
)

ZİRAAT KATILIM - CAMPAIGN RAW COMPLETER V7
Mevcut confirmed: 52
Aktif sayaç: 87

GERÇEK SEKTÖR SAYFALARI
[01] 01 slug | sayaç=1 | Ziraat Katılım Akaryakıt Kampanyaları
[02] 02 slug | sayaç=2 | Ziraat Katılım Anne, Bebek & Oyuncak Kampanyaları
[03] 02 slug | sayaç=2 | Ziraat Katılım Diğer Kampanyaları
[04] 00 slug | sayaç=0 | Ziraat Katılım Dijital Platform Kampanyaları
[05] 03 slug | sayaç=3 | Ziraat Katılım Eğitim Kampanyaları
[06] 11 slug | sayaç=11 | Ziraat Katılım Elektronik Kampanyaları
[07] 05 slug | sayaç=5 | Ziraat Katılım E-Ticaret Kampanyaları
[08] 01 slug | sayaç=1 | Ziraat Katılım Evcil Hayvan & Petshop Kampanyaları
[09] 00 slug | sayaç=0 | Ziraat Katılım Fatura & Telekomünikasyon Kampanyaları
[10] 00 slug | sayaç=0 | Ziraat Katılım Finans & Yatırım Kampanyaları
[11] 14 slug | sayaç=14 | Ziraat Katılım Giyim & Aksesuar Kampanyaları
[12] 00 slug | sayaç=0 | Ziraat Katılım Hizmet & Bireysel Gelişim Kampanyaları
[13] 00 slug | sayaç=0 | Ziraat Katılım Kitap, Kırtasiye & Ofis K

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# ZİRAAT KATILIM
# CAMPAIGN SEMANTIC INSPECTOR V1
#
# Input:
#   /content/ziraat_katilim_kampanyalar_raw_v7.json
#
# Amaç:
# 87 kampanyanın final extractor öncesi
# semantic adaylarını kontrol etmek.
# ============================================================

import json
import re
from datetime import date


INPUT_FILE = (
    "/content/"
    "ziraat_katilim_kampanyalar_raw_v7.json"
)

TODAY = date(
    2026,
    8,
    23
)


# ============================================================
# HELPERS
# ============================================================

MONTH_MAP = {
    "ocak": 1,
    "şubat": 2,
    "mart": 3,
    "nisan": 4,
    "mayıs": 5,
    "haziran": 6,
    "temmuz": 7,
    "ağustos": 8,
    "eylül": 9,
    "ekim": 10,
    "kasım": 11,
    "aralık": 12,
}


def clean_text(value):

    value = str(value or "")

    value = (
        value
        .replace("\xa0", " ")
        .replace("’", "'")
        .replace("‘", "'")
        .replace("–", "-")
        .replace("—", "-")
    )

    value = re.sub(
        r"[ \t]+",
        " ",
        value
    )

    return value.strip()


def normalize(value):

    return (
        clean_text(value)
        .replace("İ", "i")
        .replace("I", "ı")
        .casefold()
    )


def unique(values):

    result = []
    seen = set()

    for value in values:

        value = clean_text(value)

        if not value:
            continue

        key = normalize(value)

        if key in seen:
            continue

        seen.add(key)
        result.append(value)

    return result


# ============================================================
# DATE
# ============================================================

def parse_tr_date(value):

    value = clean_text(value)

    m = re.fullmatch(
        (
            r"(\d{1,2})\s+"
            r"([A-Za-zÇĞİÖŞÜçğıöşü]+)\s+"
            r"(\d{4})"
        ),
        value
    )

    if m:

        day = int(m.group(1))
        month_name = normalize(
            m.group(2)
        )
        year = int(m.group(3))

        month = MONTH_MAP.get(
            month_name
        )

        if month:

            return date(
                year,
                month,
                day
            )


    m = re.fullmatch(
        r"(\d{1,2})[./-](\d{1,2})[./-](\d{4})",
        value
    )

    if m:

        return date(
            int(m.group(3)),
            int(m.group(2)),
            int(m.group(1))
        )


    return None


# ============================================================
# TAKSIT
# ============================================================

def taksit_candidates(
    title,
    text
):

    combined = (
        title
        + "\n"
        + text
    )

    result = []


    patterns = [

        # 9 Taksit
        r"\b(\d+)\s+taksit\b",

        # 5'e Varan Taksit
        (
            r"\b(\d+)"
            r"(?:'?[ea])?\s+"
            r"varan\s+taksit\b"
        ),

        # +8 Taksit
        r"\+(\d+)\s+taksit\b",

        # 12 Aya Varan Taksit
        (
            r"\b(\d+)\s+"
            r"aya\s+varan\s+"
            r"taksit\b"
        ),
    ]


    for pattern in patterns:

        for match in re.finditer(
            pattern,
            combined,
            flags=re.I
        ):

            result.append(
                match.group(1)
            )


    return unique(result)


# ============================================================
# TL AMOUNTS
# ============================================================

def tl_candidates(text):

    values = []

    for match in re.finditer(
        (
            r"\b"
            r"(\d{1,3}"
            r"(?:[.\s]\d{3})*"
            r"(?:,\d+)?)"
            r"\s*TL\b"
        ),
        text,
        flags=re.I
    ):

        values.append(
            match.group(1)
            .replace(
                " ",
                "."
            )
            + " TL"
        )


    return unique(values)


# ============================================================
# BANKKART LIRA
# ============================================================

def bankkart_lira_candidates(text):

    result = []

    patterns = [
        (
            r"(\d{1,3}"
            r"(?:[.\s]\d{3})*"
            r"(?:,\d+)?)"
            r"\s*TL"
            r"[^.!?\n]{0,50}"
            r"Bankkart\s+Lira"
        ),

        (
            r"Bankkart\s+Lira"
            r"[^.!?\n]{0,50}"
            r"(\d{1,3}"
            r"(?:[.\s]\d{3})*"
            r"(?:,\d+)?)"
            r"\s*TL"
        ),
    ]


    for pattern in patterns:

        for match in re.finditer(
            pattern,
            text,
            flags=re.I
        ):

            value = (
                match.group(1)
                .replace(
                    " ",
                    "."
                )
                + " TL Bankkart Lira"
            )

            result.append(value)


    return unique(result)


# ============================================================
# PERCENT
# ============================================================

def percent_candidates(text):

    return unique(
        re.findall(
            r"%\s*\d+(?:[.,]\d+)?",
            text
        )
    )


# ============================================================
# SECTION CHECKS
# ============================================================

def section_flags(text):

    n = normalize(text)

    markers = {
        "donem":
            (
                "kampanya dönemi"
                in n
            ),

        "detay":
            (
                "kampanya detay"
                in n
            ),

        "kosul":
            (
                "kampanya koşul"
                in n
                or
                "katılım koşul"
                in n
            ),

        "katilim":
            (
                "katılım"
                in n
            ),

        "bankkart_lira":
            (
                "bankkart lira"
                in n
            ),
    }

    return markers


# ============================================================
# LOAD
# ============================================================

with open(
    INPUT_FILE,
    "r",
    encoding="utf-8"
) as f:

    data = json.load(f)


records = data.get(
    "kampanyalar",
    []
)


print("=" * 125)
print(
    "ZİRAAT KATILIM - "
    "CAMPAIGN SEMANTIC INSPECTOR V1"
)
print("=" * 125)

print(
    "Toplam kampanya:",
    len(records)
)

print(
    "Kontrol tarihi :",
    TODAY.isoformat()
)


# ============================================================
# AUDIT COUNTERS
# ============================================================

active_count = 0
expired_count = 0
date_problem = []
no_taksit = []
no_advantage = []

taksit_distribution = {}
end_date_distribution = {}


# ============================================================
# EACH CAMPAIGN
# ============================================================

for index, record in enumerate(
    records,
    start=1
):

    title = clean_text(
        record.get(
            "kampanya_adi",
            ""
        )
    )

    text = clean_text(
        record.get(
            "ham_metin",
            ""
        )
    )

    dates_raw = record.get(
        "tarih_adaylari",
        []
    )

    parsed_dates = [
        parse_tr_date(x)
        for x in dates_raw
    ]

    parsed_dates = [
        x
        for x in parsed_dates
        if x is not None
    ]


    start_date = (
        min(parsed_dates)
        if parsed_dates
        else None
    )

    end_date = (
        max(parsed_dates)
        if parsed_dates
        else None
    )


    if (
        start_date
        and
        end_date
        and
        start_date <= TODAY <= end_date
    ):

        status = "AKTİF"
        active_count += 1

    elif (
        end_date
        and
        TODAY > end_date
    ):

        status = "BİTMİŞ"
        expired_count += 1

    else:

        status = "TARİH?"

        date_problem.append(
            title
        )


    taksit = taksit_candidates(
        title,
        text
    )

    bankkart = (
        bankkart_lira_candidates(
            text
        )
    )

    percents = (
        percent_candidates(
            text
        )
    )

    tl_values = (
        tl_candidates(
            text
        )
    )

    flags = section_flags(
        text
    )


    if not taksit:

        no_taksit.append(
            title
        )


    if (
        not taksit
        and
        not bankkart
        and
        not percents
    ):

        no_advantage.append(
            title
        )


    for value in taksit:

        taksit_distribution[
            value
        ] = (
            taksit_distribution.get(
                value,
                0
            )
            + 1
        )


    if end_date:

        key = end_date.isoformat()

        end_date_distribution[
            key
        ] = (
            end_date_distribution.get(
                key,
                0
            )
            + 1
        )


    print()
    print(
        f"[{index:02d}/"
        f"{len(records):02d}] "
        f"{title}"
    )

    print(
        "  Status    :",
        status
    )

    print(
        "  Tarih     :",
        dates_raw
    )

    print(
        "  Taksit    :",
        taksit
    )

    print(
        "  BKL       :",
        bankkart
    )

    print(
        "  Yüzde     :",
        percents
    )

    print(
        "  TL aday   :",
        tl_values[:8]
    )

    print(
        "  Sections  :",
        flags
    )


# ============================================================
# SUMMARY
# ============================================================

print()
print("=" * 125)
print("SEMANTIC AUDIT ÖZETİ")
print("=" * 125)

print(
    "Toplam            :",
    len(records)
)

print(
    "23.08.2026 aktif  :",
    active_count
)

print(
    "Bitmiş görünen    :",
    expired_count
)

print(
    "Tarih problemi    :",
    len(date_problem)
)

print(
    "Taksit bulunan    :",
    (
        len(records)
        -
        len(no_taksit)
    )
)

print(
    "Taksit bulunmayan :",
    len(no_taksit)
)

print(
    "Avantaj belirsiz  :",
    len(no_advantage)
)


print()
print(
    "TAKSİT DAĞILIMI:"
)

for key in sorted(
    taksit_distribution,
    key=lambda x: int(x)
):

    print(
        f"- {key}: "
        f"{taksit_distribution[key]}"
    )


print()
print(
    "BİTİŞ TARİHİ DAĞILIMI:"
)

for key in sorted(
    end_date_distribution
):

    print(
        f"- {key}: "
        f"{end_date_distribution[key]}"
    )


if date_problem:

    print()
    print(
        "TARİH PROBLEMLİ:"
    )

    for title in date_problem:

        print(
            "-",
            title
        )


if no_advantage:

    print()
    print(
        "AVANTAJ ADAYI BULUNAMAYAN:"
    )

    for title in no_advantage:

        print(
            "-",
            title
        )


print()
print("=" * 125)
print("SEMANTIC INSPECTOR TAMAMLANDI ✅")
print("=" * 125)

ZİRAAT KATILIM - CAMPAIGN SEMANTIC INSPECTOR V1
Toplam kampanya: 87
Kontrol tarihi : 2026-08-23

[01/87] A101'de 6 Taksit
  Status    : AKTİF
  Tarih     : ['10 Temmuz 2025', '31 Ağustos 2026']
  Taksit    : ['6', '2']
  BKL       : ['2.000 TL Bankkart Lira', '1.000 TL Bankkart Lira']
  Yüzde     : []
  TL aday   : ['2.000 TL', '1.000 TL', '800 TL', '300 TL']
  Sections  : {'donem': True, 'detay': False, 'kosul': True, 'katilim': True, 'bankkart_lira': True}

[02/87] Abdullah Kiğılı'da 2 Taksit
  Status    : AKTİF
  Tarih     : ['01 Nisan 2025', '31 Ağustos 2026']
  Taksit    : ['2', '4', '3']
  BKL       : []
  Yüzde     : []
  TL aday   : []
  Sections  : {'donem': True, 'detay': False, 'kosul': True, 'katilim': True, 'bankkart_lira': False}

[03/87] Aile Kart'a Özel 2.000 TL'ye varan Bankkart Lira!
  Status    : AKTİF
  Tarih     : ['08 Ağustos 2026', '07 Eylül 2026']
  Taksit    : ['2', '6']
  BKL       : ['2.000 TL Bankkart Lira', '1.000 TL Bankkart Lira', '20.000 TL Bankkart Lira

In [ ]:
# ============================================================
# ZİRAAT KATILIM
# CAMPAIGN EXTRACTOR FINAL V1
#
# Input:
#   /content/ziraat_katilim_kampanyalar_raw_v7.json
#
# Output:
#   /content/ziraat_katilim_kampanya_extracted.json
#
# Final ortak 18-key schema
# ============================================================

import json
import re
from datetime import date
from collections import Counter

from google.colab import files


# ============================================================
# FILES
# ============================================================

INPUT_FILE = (
    "/content/"
    "ziraat_katilim_kampanyalar_raw_v7.json"
)

OUTPUT_FILE = (
    "/content/"
    "ziraat_katilim_kampanya_extracted.json"
)

BANK_NAME = (
    "Ziraat Katılım Bankası A.Ş."
)

TODAY = date(
    2026,
    8,
    23
)


# ============================================================
# FINAL SCHEMA
# ============================================================

SCHEMA_KEYS = [
    "banka",
    "kayit_turu",
    "urun_adi",
    "urun_kategorisi",
    "kar_payi_orani",
    "finansman_orani",
    "finansman_tutari",
    "vade",
    "taksit_sayisi",
    "masraf_bilgisi",
    "kampanya_turu",
    "kampanya_avantaji",
    "kampanya_suresi",
    "hedef_kitle",
    "para_birimi",
    "kosullar",
    "kaynak_url",
    "ham_metin",
]


LIST_FIELDS = {
    "kar_payi_orani",
    "finansman_orani",
    "finansman_tutari",
    "vade",
    "taksit_sayisi",
    "masraf_bilgisi",
    "kampanya_avantaji",
    "hedef_kitle",
    "para_birimi",
    "kosullar",
}


SCALAR_FIELDS = (
    set(SCHEMA_KEYS)
    - LIST_FIELDS
)


# ============================================================
# HELPERS
# ============================================================

def clean_text(value):

    value = str(
        value or ""
    )

    value = (
        value
        .replace("\xa0", " ")
        .replace("’", "'")
        .replace("‘", "'")
        .replace("–", "-")
        .replace("—", "-")
        .replace("\u00ad", "")
    )

    value = re.sub(
        r"[ \t]+",
        " ",
        value
    )

    value = re.sub(
        r"\n[ \t]+",
        "\n",
        value
    )

    value = re.sub(
        r"\n{3,}",
        "\n\n",
        value
    )

    return value.strip()


def normalize(value):

    value = clean_text(
        value
    )

    return (
        value
        .replace("İ", "i")
        .replace("I", "ı")
        .casefold()
    )


def unique(values):

    result = []
    seen = set()

    for value in values:

        value = clean_text(
            value
        )

        if not value:
            continue

        key = normalize(
            value
        )

        if key in seen:
            continue

        seen.add(
            key
        )

        result.append(
            value
        )

    return result


# ============================================================
# DATE
# ============================================================

MONTH_MAP = {
    "ocak": 1,
    "şubat": 2,
    "mart": 3,
    "nisan": 4,
    "mayıs": 5,
    "haziran": 6,
    "temmuz": 7,
    "ağustos": 8,
    "eylül": 9,
    "ekim": 10,
    "kasım": 11,
    "aralık": 12,
}


def parse_tr_date(value):

    value = clean_text(
        value
    )

    # 08 Ağustos 2026
    match = re.fullmatch(
        (
            r"(\d{1,2})\s+"
            r"([A-Za-zÇĞİÖŞÜçğıöşü]+)\s+"
            r"(\d{4})"
        ),
        value
    )

    if match:

        day = int(
            match.group(1)
        )

        month_name = normalize(
            match.group(2)
        )

        year = int(
            match.group(3)
        )

        month = MONTH_MAP.get(
            month_name
        )

        if month:

            return date(
                year,
                month,
                day
            )


    # 08.08.2026
    match = re.fullmatch(
        (
            r"(\d{1,2})"
            r"[./-]"
            r"(\d{1,2})"
            r"[./-]"
            r"(\d{4})"
        ),
        value
    )

    if match:

        return date(
            int(
                match.group(3)
            ),
            int(
                match.group(2)
            ),
            int(
                match.group(1)
            )
        )


    return None


def get_period(record):

    raw_dates = record.get(
        "tarih_adaylari",
        []
    )

    parsed = []

    for value in raw_dates:

        parsed_date = parse_tr_date(
            value
        )

        if parsed_date:

            parsed.append(
                (
                    parsed_date,
                    clean_text(value)
                )
            )


    parsed.sort(
        key=lambda x: x[0]
    )


    if not parsed:

        return (
            None,
            None,
            ""
        )


    start_date = parsed[0][0]
    end_date = parsed[-1][0]

    start_text = parsed[0][1]
    end_text = parsed[-1][1]


    if (
        start_date
        == end_date
    ):

        period_text = start_text

    else:

        period_text = (
            f"{start_text} - "
            f"{end_text}"
        )


    return (
        start_date,
        end_date,
        period_text
    )


# ============================================================
# NUMBER NORMALIZATION
# ============================================================

def normalize_tl_number(value):

    value = str(
        value or ""
    )

    value = re.sub(
        r"\s+",
        "",
        value
    )


    # 2000 -> 2.000
    if value.isdigit():

        number = int(
            value
        )

        return (
            f"{number:,}"
            .replace(
                ",",
                "."
            )
        )


    return value


# ============================================================
# TITLE TAKSIT
#
# KRİTİK:
# Taksit sadece BAŞLIKTAN çıkarılır.
#
# Böylece related campaign contamination alınmaz.
# ============================================================

def extract_title_taksit(title):

    title = clean_text(
        title
    )


    patterns = [

        # 12 Aya Varan Taksit
        (
            r"\b(\d+)\s+"
            r"aya\s+varan\s+taksit"
        ),

        # +8 Taksit
        (
            r"\+(\d+)\s*"
            r"taksit"
        ),

        # 9'a varan / 6'ya varan
        (
            r"\b(\d+)"
            r"'?(?:e|a|ye|ya)\s+"
            r"varan\s+taksit"
        ),

        # normal 6 Taksit
        (
            r"\b(\d+)\s+"
            r"taksit\b"
        ),
    ]


    for pattern in patterns:

        match = re.search(
            pattern,
            title,
            flags=re.I
        )

        if match:

            return [
                match.group(1)
            ]


    return []


# ============================================================
# CAMPAIGN TYPE
# ============================================================

def extract_campaign_type(title):

    n = normalize(
        title
    )


    if (
        "bankkart lira"
        in n
    ):

        return (
            "Bankkart Lira"
        )


    if (
        "indirim"
        in n
        or
        re.search(
            r"%\s*\d",
            title
        )
    ):

        return (
            "İndirim"
        )


    if extract_title_taksit(
        title
    ):

        return (
            "Taksit"
        )


    return (
        "Avantaj"
    )


# ============================================================
# TITLE ADVANTAGE
#
# Ana kampanya avantajı yalnızca başlıktan.
# ============================================================

def extract_title_advantage(title):

    title = clean_text(
        title
    )

    n = normalize(
        title
    )

    result = []


    # --------------------------------------------------------
    # BANKKART LIRA
    #
    # 2.000 TL'ye varan Bankkart Lira
    # 400 TL Bankkart Lira
    # Toplam 100 TL Bankkart Lira
    # --------------------------------------------------------

    match = re.search(
        (
            r"(\d{1,3}"
            r"(?:[.\s]\d{3})*"
            r"|\d+)"
            r"\s*TL"
            r"(?:'ye)?"
            r"\s*"
            r"(?:varan\s+)?"
            r"Bankkart\s+Lira"
        ),
        title,
        flags=re.I
    )


    if match:

        amount = normalize_tl_number(
            match.group(1)
        )

        matched_text = normalize(
            match.group(0)
        )


        if (
            "varan"
            in matched_text
        ):

            result.append(
                (
                    f"{amount} TL'ye varan "
                    f"Bankkart Lira"
                )
            )


        elif (
            "toplam"
            in normalize(
                title[
                    :match.start()
                ]
            )
        ):

            result.append(
                (
                    f"Toplam {amount} TL "
                    f"Bankkart Lira"
                )
            )


        else:

            result.append(
                (
                    f"{amount} TL "
                    f"Bankkart Lira"
                )
            )


    # --------------------------------------------------------
    # TL İNDİRİM
    #
    # 3.000 TL'ye Varan İndirim
    # --------------------------------------------------------

    match = re.search(
        (
            r"(\d{1,3}"
            r"(?:[.\s]\d{3})*"
            r"|\d+)"
            r"\s*TL"
            r"'ye\s+"
            r"varan\s+"
            r"indirim"
        ),
        title,
        flags=re.I
    )


    if match:

        amount = normalize_tl_number(
            match.group(1)
        )

        result.append(
            (
                f"{amount} TL'ye "
                f"varan indirim"
            )
        )


    # --------------------------------------------------------
    # % İNDİRİM
    # --------------------------------------------------------

    match = re.search(
        (
            r"(%\s*\d+"
            r"(?:[.,]\d+)?)"
            r"\s*indirim"
        ),
        title,
        flags=re.I
    )


    if match:

        percent = (
            match.group(1)
            .replace(
                " ",
                ""
            )
        )

        result.append(
            f"{percent} indirim"
        )


    # --------------------------------------------------------
    # TAKSIT
    # --------------------------------------------------------

    taksit = extract_title_taksit(
        title
    )


    if taksit:

        number = taksit[0]


        if re.search(
            r"peşin\s+fiyatına",
            title,
            flags=re.I
        ):

            result.append(
                (
                    f"Peşin fiyatına "
                    f"{number} taksit"
                )
            )


        elif re.search(
            (
                rf"\b{re.escape(number)}"
                r"\s+aya\s+varan\s+taksit"
            ),
            title,
            flags=re.I
        ):

            result.append(
                (
                    f"{number} aya "
                    f"varan taksit"
                )
            )


        else:

            match_varan = re.search(
                (
                    rf"\b{re.escape(number)}"
                    r"('?(?:e|a|ye|ya))"
                    r"\s+varan\s+taksit"
                ),
                title,
                flags=re.I
            )


            if match_varan:

                suffix = (
                    match_varan.group(1)
                )

                result.append(
                    (
                        f"{number}"
                        f"{suffix} "
                        f"varan taksit"
                    )
                )


            elif re.search(
                (
                    rf"\+"
                    rf"{re.escape(number)}"
                    r"\s*taksit"
                ),
                title,
                flags=re.I
            ):

                result.append(
                    f"+{number} taksit"
                )


            else:

                result.append(
                    f"{number} taksit"
                )


    # --------------------------------------------------------
    # İKİ NUMERİK OLMAYAN ANA KAMPANYA
    # --------------------------------------------------------

    if not result:

        if (
            "halalbooking"
            in n
            and
            "avantajlı tatil"
            in n
        ):

            result.append(
                (
                    "Size özel avantajlı "
                    "tatil fırsatı"
                )
            )


        elif (
            "ziraat katılım "
            "avantajlı bankkart "
            "kampanyaları"
            in n
        ):

            result.append(
                (
                    "Avantajlı Bankkart "
                    "kampanyaları"
                )
            )


        else:

            # Son güvenli fallback:
            # sayı/avantaj uydurmak yerine
            # başlığı olduğu gibi koru.
            result.append(
                title
            )


    return unique(
        result
    )


# ============================================================
# CORE TEXT ISOLATION
#
# ham_metin içinde related kampanyalar bulunduğu için
# başka kampanya başlığı başladığı anda kesiyoruz.
# ============================================================

CUT_MARKERS = [
    "Benzer Kampanyalar",
    "İlgili Kampanyalar",
    "Diğer Kampanyalar",
    "Öne Çıkan Kampanyalar",
    "İlginizi Çekebilir",
    "Bunlar da İlginizi Çekebilir",
    "Diğer Banka Kampanyaları",
    "Kampanya Önerileri",
    "Son Kampanyalar",
]


def isolate_campaign_core(
    record,
    all_titles
):

    raw = clean_text(
        record.get(
            "ham_metin",
            ""
        )
    )

    title = clean_text(
        record.get(
            "kampanya_adi",
            ""
        )
    )


    # --------------------------------------------------------
    # Current title sonrası
    # --------------------------------------------------------

    title_pos = raw.find(
        title
    )


    if title_pos >= 0:

        segment = raw[
            title_pos:
        ]

    else:

        segment = raw


    # --------------------------------------------------------
    # Koşullar başlangıcı:
    # başka kampanya başlıklarını bundan sonra ara.
    # --------------------------------------------------------

    n_segment = normalize(
        segment
    )


    condition_start = 0


    for marker in [
        "kampanya koşulları",
        "kampanya koşulları ve detayları",
        "kampanya koşulları ve katılım",
    ]:

        pos = n_segment.find(
            normalize(marker)
        )

        if pos >= 0:

            condition_start = pos
            break


    search_from = max(
        100,
        condition_start
    )


    cutoff_positions = []


    # --------------------------------------------------------
    # Known related campaign titles
    # --------------------------------------------------------

    for other_title in all_titles:

        if (
            other_title
            == title
        ):
            continue


        # Bu generic başlık bazı sayfalarda
        # navigasyon metni olarak geçebilir.
        if (
            other_title
            ==
            "Ziraat Katılım Avantajlı "
            "Bankkart Kampanyaları"
        ):
            continue


        pos = segment.find(
            other_title,
            search_from
        )


        if pos >= 0:

            cutoff_positions.append(
                pos
            )


    # --------------------------------------------------------
    # Generic related markers
    # --------------------------------------------------------

    for marker in CUT_MARKERS:

        match = re.search(
            re.escape(
                marker
            ),
            segment[
                search_from:
            ],
            flags=re.I
        )

        if match:

            cutoff_positions.append(
                (
                    search_from
                    +
                    match.start()
                )
            )


    if cutoff_positions:

        cutoff = min(
            cutoff_positions
        )

        segment = segment[
            :cutoff
        ]


    return clean_text(
        segment
    )


# ============================================================
# CONDITIONS
# ============================================================

def extract_conditions(
    core,
    title
):

    core = clean_text(
        core
    )

    n = normalize(
        core
    )


    # --------------------------------------------------------
    # Prefer explicit condition section.
    # --------------------------------------------------------

    start = None


    for marker in [
        "kampanya koşulları ve detayları",
        "kampanya koşulları ve katılım",
        "kampanya koşulları",
    ]:

        pos = n.find(
            normalize(marker)
        )

        if pos >= 0:

            start = (
                pos
                +
                len(marker)
            )

            break


    if start is not None:

        block = core[
            start:
        ]

    else:

        block = core


    # --------------------------------------------------------
    # Split lines + sentences
    # --------------------------------------------------------

    pieces = re.split(
        (
            r"\n+"
            r"|"
            r"(?<=[.!?])\s+"
        ),
        block
    )


    result = []


    skip_exact = {
        "hemen katıl",
        "hemen katil",
        "paylaş",
        "kaynak",
        "kampanya dönemi",
        "kampanya detayları",
        "kampanya koşulları",
        "ziraat katılım",
        "kartavantaj",
    }


    for piece in pieces:

        piece = clean_text(
            piece
        )

        piece = re.sub(
            r"^[•●▪◦*-]+\s*",
            "",
            piece
        )

        if not piece:
            continue


        n_piece = normalize(
            piece
        )


        if (
            n_piece
            in skip_exact
        ):
            continue


        if (
            normalize(title)
            ==
            n_piece
        ):
            continue


        # Sadece tarih satırıysa koşul değildir.
        if re.fullmatch(
            (
                r"\d{1,2}\s+"
                r"[A-Za-zÇĞİÖŞÜçğıöşü]+\s+"
                r"\d{4}"
            ),
            piece
        ):
            continue


        if re.fullmatch(
            (
                r"\d{1,2}"
                r"[./-]"
                r"\d{1,2}"
                r"[./-]"
                r"\d{4}"
            ),
            piece
        ):
            continue


        if (
            len(piece) < 15
        ):
            continue


        if (
            len(piece) > 800
        ):

            # Çok uzun paragrafları
            # tekrar cümlelere böl.
            subs = re.split(
                r"(?<=[.!?])\s+",
                piece
            )

            for sub in subs:

                sub = clean_text(
                    sub
                )

                if (
                    15
                    <= len(sub)
                    <= 800
                ):

                    result.append(
                        sub
                    )

            continue


        result.append(
            piece
        )


    return unique(
        result
    )[:40]


# ============================================================
# TARGET AUDIENCE
# ============================================================

def extract_target_audience(
    title,
    core
):

    title_n = normalize(
        title
    )

    core_n = normalize(
        core
    )

    result = []


    if (
        "aile kart"
        in title_n
    ):

        return [
            "Aile Kart sahipleri"
        ]


    if (
        "bağımsız kart"
        in title_n
    ):

        return [
            "Bağımsız Kart sahipleri"
        ]


    if (
        "ilk ek kredi kart"
        in title_n
    ):

        return [
            (
                "İlk ek kredi kartını "
                "alan müşteriler"
            )
        ]


    if (
        "troy"
        in title_n
    ):

        return [
            "TROY kart sahipleri"
        ]


    if (
        "katılım bankkart"
        in core_n
        or
        "katilim bankkart"
        in core_n
    ):

        result.append(
            "Katılım Bankkart sahipleri"
        )


    elif (
        "bankkart"
        in core_n
    ):

        result.append(
            "Bankkart sahipleri"
        )


    return unique(
        result
    )


# ============================================================
# CURRENCY
# ============================================================

def extract_currency(core):

    result = []


    if re.search(
        r"\bTL\b",
        core,
        flags=re.I
    ):

        result.append(
            "TL"
        )


    if re.search(
        r"\bUSD\b",
        core,
        flags=re.I
    ):

        result.append(
            "USD"
        )


    if re.search(
        r"\bEUR\b",
        core,
        flags=re.I
    ):

        result.append(
            "EUR"
        )


    return unique(
        result
    )


# ============================================================
# BUILD FINAL RECORD
# ============================================================

def extract_record(
    raw,
    all_titles
):

    title = clean_text(
        raw.get(
            "kampanya_adi",
            ""
        )
    )

    source_url = clean_text(
        raw.get(
            "kaynak_url",
            ""
        )
    )

    # EXACT raw preservation
    raw_text = str(
        raw.get(
            "ham_metin",
            ""
        )
    )


    core = isolate_campaign_core(
        raw,
        all_titles
    )


    start_date, end_date, period = (
        get_period(
            raw
        )
    )


    return {

        "banka":
            BANK_NAME,

        "kayit_turu":
            "kampanya",

        "urun_adi":
            title,

        "urun_kategorisi":
            "Kart Kampanyaları",

        # Kampanya indirimi / yüzde avantajı
        # finansman oranı DEĞİLDİR.
        "kar_payi_orani":
            [],

        "finansman_orani":
            [],

        "finansman_tutari":
            [],

        # Kampanyadaki taksit vade değildir.
        "vade":
            [],

        # Ana taksit yalnızca title'dan.
        "taksit_sayisi":
            extract_title_taksit(
                title
            ),

        "masraf_bilgisi":
            [],

        "kampanya_turu":
            extract_campaign_type(
                title
            ),

        "kampanya_avantaji":
            extract_title_advantage(
                title
            ),

        "kampanya_suresi":
            period,

        "hedef_kitle":
            extract_target_audience(
                title,
                core
            ),

        "para_birimi":
            extract_currency(
                core
            ),

        "kosullar":
            extract_conditions(
                core,
                title
            ),

        "kaynak_url":
            source_url,

        "ham_metin":
            raw_text,
    }


# ============================================================
# LOAD
# ============================================================

with open(
    INPUT_FILE,
    "r",
    encoding="utf-8"
) as f:

    raw_data = json.load(
        f
    )


raw_records = raw_data.get(
    "kampanyalar",
    []
)


all_titles = [
    clean_text(
        record.get(
            "kampanya_adi",
            ""
        )
    )
    for record in raw_records
]


print("=" * 120)
print(
    "ZİRAAT KATILIM - "
    "CAMPAIGN EXTRACTOR FINAL V1"
)
print("=" * 120)

print(
    "RAW kampanya:",
    len(raw_records)
)


# ============================================================
# EXTRACT
# ============================================================

records = []

errors = []


for index, raw in enumerate(
    raw_records,
    start=1
):

    try:

        record = extract_record(
            raw,
            all_titles
        )

        records.append(
            record
        )

    except Exception as error:

        errors.append(
            (
                f"[{index}] "
                f"{type(error).__name__}: "
                f"{error}"
            )
        )


# ============================================================
# SCHEMA VALIDATION
# ============================================================

for index, record in enumerate(
    records,
    start=1
):

    name = record.get(
        "urun_adi",
        "?"
    )


    if (
        list(
            record.keys()
        )
        != SCHEMA_KEYS
    ):

        errors.append(
            (
                f"[{index}] "
                f"{name} -> "
                "schema/order hatası"
            )
        )


    for field in LIST_FIELDS:

        if not isinstance(
            record.get(
                field
            ),
            list
        ):

            errors.append(
                (
                    f"{name} -> "
                    f"{field} list değil"
                )
            )


    for field in SCALAR_FIELDS:

        if not isinstance(
            record.get(
                field
            ),
            str
        ):

            errors.append(
                (
                    f"{name} -> "
                    f"{field} string değil"
                )
            )


    if (
        record[
            "banka"
        ]
        != BANK_NAME
    ):

        errors.append(
            f"{name} -> banka yanlış"
        )


    if (
        record[
            "kayit_turu"
        ]
        != "kampanya"
    ):

        errors.append(
            (
                f"{name} -> "
                "kayit_turu yanlış"
            )
        )


    for field in [
        "urun_adi",
        "kampanya_turu",
        "kampanya_suresi",
        "kaynak_url",
        "ham_metin",
    ]:

        if not record[
            field
        ]:

            errors.append(
                (
                    f"{name} -> "
                    f"{field} boş"
                )
            )


    if not record[
        "kampanya_avantaji"
    ]:

        errors.append(
            (
                f"{name} -> "
                "kampanya_avantaji boş"
            )
        )


    # Her RAW sayfada koşul bölümü vardı.
    if not record[
        "kosullar"
    ]:

        errors.append(
            (
                f"{name} -> "
                "kosullar boş"
            )
        )


    # Finansman semantiği kampanyaya sızmasın.
    for field in [
        "kar_payi_orani",
        "finansman_orani",
        "finansman_tutari",
        "vade",
    ]:

        if record[
            field
        ]:

            errors.append(
                (
                    f"{name} -> "
                    f"{field} boş olmalı"
                )
            )


# ============================================================
# COUNT / DUPLICATE
# ============================================================

if len(records) != 87:

    errors.append(
        (
            "Extracted kampanya "
            f"{len(records)} != 87"
        )
    )


urls = [
    record[
        "kaynak_url"
    ]
    for record in records
]


duplicate_url = (
    len(urls)
    -
    len(
        set(urls)
    )
)


if duplicate_url:

    errors.append(
        (
            "Duplicate URL: "
            f"{duplicate_url}"
        )
    )


names = [
    record[
        "urun_adi"
    ]
    for record in records
]


duplicate_name = (
    len(names)
    -
    len(
        set(names)
    )
)


if duplicate_name:

    errors.append(
        (
            "Duplicate ürün adı: "
            f"{duplicate_name}"
        )
    )


# ============================================================
# OFFICIAL SOURCE CHECK
# ============================================================

for record in records:

    if not record[
        "kaynak_url"
    ].startswith(
        (
            "https://www."
            "ziraatkatilim.com.tr/"
            "kart-kampanyalari/"
        )
    ):

        errors.append(
            (
                f"{record['urun_adi']} -> "
                "official kaynak_url değil"
            )
        )


# ============================================================
# TYPE DISTRIBUTION
# ============================================================

type_counts = Counter(
    record[
        "kampanya_turu"
    ]
    for record in records
)


EXPECTED_TYPES = {
    "Taksit":
        72,

    "Bankkart Lira":
        9,

    "İndirim":
        4,

    "Avantaj":
        2,
}


if dict(
    type_counts
) != EXPECTED_TYPES:

    errors.append(
        (
            "Kampanya tür dağılımı yanlış: "
            f"{dict(type_counts)}"
        )
    )


# ============================================================
# TAKSIT SEMANTIC VALIDATION
# ============================================================

for record in records:

    name = record[
        "urun_adi"
    ]

    type_ = record[
        "kampanya_turu"
    ]

    taksit = record[
        "taksit_sayisi"
    ]


    if (
        type_
        == "Taksit"
    ):

        if (
            len(taksit)
            != 1
        ):

            errors.append(
                (
                    f"{name} -> "
                    f"Taksit campaign ama "
                    f"taksit={taksit}"
                )
            )


    else:

        # Ana kampanya başka türdeyse
        # related-card taksit sayısı alınmamalı.
        if taksit:

            errors.append(
                (
                    f"{name} -> "
                    f"ana tür {type_} ama "
                    f"taksit sızmış: {taksit}"
                )
            )


# ============================================================
# PERIOD / ACTIVE VALIDATION
# ============================================================

active_count = 0

end_date_distribution = Counter()


for raw, record in zip(
    raw_records,
    records
):

    start_date, end_date, period = (
        get_period(
            raw
        )
    )


    if (
        start_date is None
        or
        end_date is None
    ):

        errors.append(
            (
                f"{record['urun_adi']} -> "
                "tarih parse edilemedi"
            )
        )

        continue


    if (
        start_date
        <= TODAY
        <= end_date
    ):

        active_count += 1

    else:

        errors.append(
            (
                f"{record['urun_adi']} -> "
                f"23.08.2026 aktif değil: "
                f"{period}"
            )
        )


    end_date_distribution[
        end_date.isoformat()
    ] += 1


if active_count != 87:

    errors.append(
        (
            "Aktif kampanya "
            f"{active_count} != 87"
        )
    )


EXPECTED_END_DATES = {
    "2026-08-31":
        69,

    "2026-09-07":
        9,

    "2026-09-30":
        2,

    "2026-12-31":
        7,
}


if (
    dict(
        sorted(
            end_date_distribution.items()
        )
    )
    != EXPECTED_END_DATES
):

    errors.append(
        (
            "Bitiş tarihi dağılımı yanlış: "
            f"{dict(end_date_distribution)}"
        )
    )


# ============================================================
# CRITICAL CAMPAIGN CHECKS
# ============================================================

by_name = {
    record[
        "urun_adi"
    ]:
    record
    for record in records
}


def require(
    name,
    field,
    expected
):

    record = by_name.get(
        name
    )


    if record is None:

        errors.append(
            f"{name} bulunamadı"
        )

        return


    actual = record[
        field
    ]


    if actual != expected:

        errors.append(
            (
                f"{name} -> "
                f"{field}: "
                f"{actual} "
                f"!= {expected}"
            )
        )


# A101 contamination kontrolü
require(
    "A101'de 6 Taksit",
    "taksit_sayisi",
    ["6"]
)

require(
    "A101'de 6 Taksit",
    "kampanya_avantaji",
    ["6 taksit"]
)


# Aile Kart
require(
    (
        "Aile Kart'a Özel "
        "2.000 TL'ye varan "
        "Bankkart Lira!"
    ),
    "kampanya_avantaji",
    [
        (
            "2.000 TL'ye varan "
            "Bankkart Lira"
        )
    ]
)


# Enterprise
require(
    (
        "Enterprise'ta Araç "
        "Kiralamalarınızda "
        "%30 İndirim"
    ),
    "kampanya_avantaji",
    [
        "%30 indirim"
    ]
)


# +8
require(
    (
        "Okul Ödemelerinizde "
        "+8 Taksit"
    ),
    "taksit_sayisi",
    ["8"]
)

require(
    (
        "Okul Ödemelerinizde "
        "+8 Taksit"
    ),
    "kampanya_avantaji",
    [
        "+8 taksit"
    ]
)


# 12 aya varan
require(
    (
        "Okul Ödemelerinizde "
        "12 Aya Varan "
        "Taksit Fırsatı"
    ),
    "taksit_sayisi",
    ["12"]
)

require(
    (
        "Okul Ödemelerinizde "
        "12 Aya Varan "
        "Taksit Fırsatı"
    ),
    "kampanya_avantaji",
    [
        "12 aya varan taksit"
    ]
)


# 6'ya varan
require(
    (
        "Trendyol'da "
        "6'ya varan Taksit"
    ),
    "taksit_sayisi",
    ["6"]
)


# TROY İdefix
require(
    (
        "TROY Kartla İdefix'te "
        "3.000 TL'ye Varan "
        "İndirim!"
    ),
    "kampanya_avantaji",
    [
        (
            "3.000 TL'ye "
            "varan indirim"
        )
    ]
)


# Veteriner 2000 -> 2.000 normalizasyonu
require(
    (
        "Veteriner ve Petshop "
        "Harcamalarınıza "
        "2000 TL Bankkart Lira!"
    ),
    "kampanya_avantaji",
    [
        "2.000 TL Bankkart Lira"
    ]
)


# ============================================================
# SAVE DIRECT LIST
# ============================================================

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        records,
        f,
        ensure_ascii=False,
        indent=4
    )


# ============================================================
# SUMMARY
# ============================================================

print()
print("=" * 120)
print(
    "ZİRAAT KATILIM - "
    "CAMPAIGN FINAL AUDIT"
)
print("=" * 120)

print(
    "RAW kampanya       :",
    len(raw_records)
)

print(
    "Extracted          :",
    len(records)
)

print(
    "Aktif 23.08.2026  :",
    active_count
)

print(
    "Duplicate URL      :",
    duplicate_url
)

print(
    "Duplicate ad       :",
    duplicate_name
)

print(
    "Validation error   :",
    len(errors)
)

print(
    "Final JSON         :",
    OUTPUT_FILE
)


print()
print("=" * 120)
print("KAMPANYA TÜR DAĞILIMI")
print("=" * 120)

for key in [
    "Taksit",
    "Bankkart Lira",
    "İndirim",
    "Avantaj",
]:

    print(
        f"{key:<15}: "
        f"{type_counts.get(key, 0)}"
    )


print()
print("=" * 120)
print("BİTİŞ TARİHİ DAĞILIMI")
print("=" * 120)

for key in sorted(
    end_date_distribution
):

    print(
        f"{key}: "
        f"{end_date_distribution[key]}"
    )


# ============================================================
# SAMPLE CRITICAL RECORDS
# ============================================================

print()
print("=" * 120)
print("KRİTİK KAYITLAR")
print("=" * 120)


CHECK_NAMES = [
    "A101'de 6 Taksit",

    (
        "Aile Kart'a Özel "
        "2.000 TL'ye varan "
        "Bankkart Lira!"
    ),

    (
        "Enterprise'ta Araç "
        "Kiralamalarınızda "
        "%30 İndirim"
    ),

    (
        "Okul Ödemelerinizde "
        "+8 Taksit"
    ),

    (
        "Okul Ödemelerinizde "
        "12 Aya Varan "
        "Taksit Fırsatı"
    ),

    (
        "Trendyol'da "
        "6'ya varan Taksit"
    ),

    (
        "TROY Kartla İdefix'te "
        "3.000 TL'ye Varan "
        "İndirim!"
    ),

    (
        "Veteriner ve Petshop "
        "Harcamalarınıza "
        "2000 TL Bankkart Lira!"
    ),
]


for name in CHECK_NAMES:

    record = by_name.get(
        name
    )

    if not record:
        continue


    print()
    print(
        name
    )

    print(
        "  Tür      :",
        record[
            "kampanya_turu"
        ]
    )

    print(
        "  Taksit   :",
        record[
            "taksit_sayisi"
        ]
    )

    print(
        "  Avantaj  :",
        record[
            "kampanya_avantaji"
        ]
    )

    print(
        "  Süre     :",
        record[
            "kampanya_suresi"
        ]
    )

    print(
        "  Hedef    :",
        record[
            "hedef_kitle"
        ]
    )

    print(
        "  Para     :",
        record[
            "para_birimi"
        ]
    )

    print(
        "  Koşul    :",
        len(
            record[
                "kosullar"
            ]
        )
    )


# ============================================================
# ERRORS
# ============================================================

if errors:

    print()
    print("=" * 120)
    print("HATALAR")
    print("=" * 120)

    for error in errors:

        print(
            "-",
            error
        )


print()
print("=" * 120)


if not errors:

    print(
        "SONUÇ: ZİRAAT KATILIM "
        "KAMPANYA EXTRACTOR "
        "87/87 TAMAMEN BAŞARILI ✅"
    )

else:

    print(
        "SONUÇ: KAMPANYA EXTRACTOR "
        "KONTROL GEREKİYOR ❌"
    )


print("=" * 120)


files.download(
    OUTPUT_FILE
)

ZİRAAT KATILIM - CAMPAIGN EXTRACTOR FINAL V1
RAW kampanya: 87

ZİRAAT KATILIM - CAMPAIGN FINAL AUDIT
RAW kampanya       : 87
Extracted          : 87
Aktif 23.08.2026  : 87
Duplicate URL      : 0
Duplicate ad       : 0
Validation error   : 0
Final JSON         : /content/ziraat_katilim_kampanya_extracted.json

KAMPANYA TÜR DAĞILIMI
Taksit         : 72
Bankkart Lira  : 9
İndirim        : 4
Avantaj        : 2

BİTİŞ TARİHİ DAĞILIMI
2026-08-31: 69
2026-09-07: 9
2026-09-30: 2
2026-12-31: 7

KRİTİK KAYITLAR

A101'de 6 Taksit
  Tür      : Taksit
  Taksit   : ['6']
  Avantaj  : ['6 taksit']
  Süre     : 10 Temmuz 2025 - 31 Ağustos 2026
  Hedef    : ['Katılım Bankkart sahipleri']
  Para     : ['TL']
  Koşul    : 8

Aile Kart'a Özel 2.000 TL'ye varan Bankkart Lira!
  Tür      : Bankkart Lira
  Taksit   : []
  Avantaj  : ["2.000 TL'ye varan Bankkart Lira"]
  Süre     : 08 Ağustos 2026 - 07 Eylül 2026
  Hedef    : ['Aile Kart sahipleri']
  Para     : ['TL']
  Koşul    : 10

Enterprise'ta Araç Kira

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# ZİRAAT KATILIM
# FINAL MERGER V1
#
# Input:
#   /content/ziraat_katilim_finansman_extracted.json
#   /content/ziraat_katilim_kampanya_extracted.json
#
# Output:
#   /content/ziraat_katilim_final.json
#
# Beklenen:
#   20 finansman + 87 kampanya = 107 kayıt
# ============================================================

import json

from collections import Counter
from google.colab import files


# ============================================================
# FILES
# ============================================================

FINANCE_FILE = (
    "/content/"
    "ziraat_katilim_finansman_extracted.json"
)

CAMPAIGN_FILE = (
    "/content/"
    "ziraat_katilim_kampanya_extracted.json"
)

OUTPUT_FILE = (
    "/content/"
    "ziraat_katilim_final.json"
)


BANK_NAME = (
    "Ziraat Katılım Bankası A.Ş."
)


# ============================================================
# SCHEMA
# ============================================================

SCHEMA_KEYS = [
    "banka",
    "kayit_turu",
    "urun_adi",
    "urun_kategorisi",
    "kar_payi_orani",
    "finansman_orani",
    "finansman_tutari",
    "vade",
    "taksit_sayisi",
    "masraf_bilgisi",
    "kampanya_turu",
    "kampanya_avantaji",
    "kampanya_suresi",
    "hedef_kitle",
    "para_birimi",
    "kosullar",
    "kaynak_url",
    "ham_metin",
]


LIST_FIELDS = {
    "kar_payi_orani",
    "finansman_orani",
    "finansman_tutari",
    "vade",
    "taksit_sayisi",
    "masraf_bilgisi",
    "kampanya_avantaji",
    "hedef_kitle",
    "para_birimi",
    "kosullar",
}


SCALAR_FIELDS = (
    set(SCHEMA_KEYS)
    - LIST_FIELDS
)


# ============================================================
# LOAD
# ============================================================

with open(
    FINANCE_FILE,
    "r",
    encoding="utf-8"
) as f:

    finance = json.load(f)


with open(
    CAMPAIGN_FILE,
    "r",
    encoding="utf-8"
) as f:

    campaigns = json.load(f)


print("=" * 118)
print("ZİRAAT KATILIM - FINAL MERGER V1")
print("=" * 118)

print(
    "Finansman:",
    len(finance)
)

print(
    "Kampanya :",
    len(campaigns)
)


# ============================================================
# MERGE
# ============================================================

records = (
    finance
    +
    campaigns
)


# ============================================================
# VALIDATION
# ============================================================

errors = []


# ------------------------------------------------------------
# EXPECTED COUNTS
# ------------------------------------------------------------

if len(finance) != 20:

    errors.append(
        (
            "Finansman sayısı "
            f"{len(finance)} != 20"
        )
    )


if len(campaigns) != 87:

    errors.append(
        (
            "Kampanya sayısı "
            f"{len(campaigns)} != 87"
        )
    )


if len(records) != 107:

    errors.append(
        (
            "Toplam kayıt "
            f"{len(records)} != 107"
        )
    )


# ------------------------------------------------------------
# EACH RECORD
# ------------------------------------------------------------

for index, record in enumerate(
    records,
    start=1
):

    name = record.get(
        "urun_adi",
        "?"
    )


    # Exact schema + order
    if (
        list(record.keys())
        != SCHEMA_KEYS
    ):

        errors.append(
            (
                f"[{index}] "
                f"{name} -> "
                "schema/order hatası"
            )
        )


    # List fields
    for field in LIST_FIELDS:

        if not isinstance(
            record.get(field),
            list
        ):

            errors.append(
                (
                    f"[{index}] "
                    f"{name} -> "
                    f"{field} list değil"
                )
            )


    # Scalar fields
    for field in SCALAR_FIELDS:

        if not isinstance(
            record.get(field),
            str
        ):

            errors.append(
                (
                    f"[{index}] "
                    f"{name} -> "
                    f"{field} string değil"
                )
            )


    # Bank
    if (
        record.get("banka")
        != BANK_NAME
    ):

        errors.append(
            (
                f"[{index}] "
                f"{name} -> "
                "banka adı yanlış"
            )
        )


    # Record type
    if (
        record.get("kayit_turu")
        not in {
            "finansman",
            "kampanya",
        }
    ):

        errors.append(
            (
                f"[{index}] "
                f"{name} -> "
                "kayit_turu geçersiz"
            )
        )


    # Required
    for field in (
        "urun_adi",
        "kaynak_url",
        "ham_metin",
    ):

        if not record.get(field):

            errors.append(
                (
                    f"[{index}] "
                    f"{name} -> "
                    f"{field} boş"
                )
            )


    # Currency convention
    if (
        "TRY"
        in record.get(
            "para_birimi",
            []
        )
    ):

        errors.append(
            (
                f"[{index}] "
                f"{name} -> "
                "TRY kullanılmamalı, TL olmalı"
            )
        )


# ============================================================
# TYPE COUNTS
# ============================================================

type_counts = Counter(
    record["kayit_turu"]
    for record in records
)


if (
    type_counts.get(
        "finansman",
        0
    )
    != 20
):

    errors.append(
        (
            "Final finansman sayısı: "
            f"{type_counts.get('finansman', 0)}"
        )
    )


if (
    type_counts.get(
        "kampanya",
        0
    )
    != 87
):

    errors.append(
        (
            "Final kampanya sayısı: "
            f"{type_counts.get('kampanya', 0)}"
        )
    )


# ============================================================
# DUPLICATES
# ============================================================

urls = [
    record["kaynak_url"]
    for record in records
]


duplicate_url = (
    len(urls)
    -
    len(set(urls))
)


if duplicate_url:

    errors.append(
        (
            "Duplicate URL: "
            f"{duplicate_url}"
        )
    )


# Aynı isim farklı kayit_turu altında teorik olarak
# mümkün olabileceği için type+name ile kontrol.
name_keys = [
    (
        record["kayit_turu"],
        record["urun_adi"],
    )
    for record in records
]


duplicate_name = (
    len(name_keys)
    -
    len(set(name_keys))
)


if duplicate_name:

    errors.append(
        (
            "Duplicate type+ürün adı: "
            f"{duplicate_name}"
        )
    )


# ============================================================
# FINANCE SEMANTIC
# ============================================================

for record in finance:

    name = record["urun_adi"]


    if (
        record["kayit_turu"]
        != "finansman"
    ):

        errors.append(
            (
                f"{name} -> "
                "finance dosyasında "
                "kayit_turu yanlış"
            )
        )


    if record["kampanya_turu"]:

        errors.append(
            (
                f"{name} -> "
                "finansmanda kampanya_turu "
                "boş olmalı"
            )
        )


    if record["kampanya_avantaji"]:

        errors.append(
            (
                f"{name} -> "
                "finansmanda "
                "kampanya_avantaji "
                "boş olmalı"
            )
        )


    if record["kampanya_suresi"]:

        errors.append(
            (
                f"{name} -> "
                "finansmanda "
                "kampanya_suresi "
                "boş olmalı"
            )
        )


# ============================================================
# CAMPAIGN SEMANTIC
# ============================================================

for record in campaigns:

    name = record["urun_adi"]


    if (
        record["kayit_turu"]
        != "kampanya"
    ):

        errors.append(
            (
                f"{name} -> "
                "campaign dosyasında "
                "kayit_turu yanlış"
            )
        )


    if not record[
        "kampanya_turu"
    ]:

        errors.append(
            (
                f"{name} -> "
                "kampanya_turu boş"
            )
        )


    if not record[
        "kampanya_avantaji"
    ]:

        errors.append(
            (
                f"{name} -> "
                "kampanya_avantaji boş"
            )
        )


    if not record[
        "kampanya_suresi"
    ]:

        errors.append(
            (
                f"{name} -> "
                "kampanya_suresi boş"
            )
        )


    # Kampanyalarda finansman semantiği olmamalı
    for field in (
        "kar_payi_orani",
        "finansman_orani",
        "finansman_tutari",
        "vade",
    ):

        if record[field]:

            errors.append(
                (
                    f"{name} -> "
                    f"kampanyada {field} "
                    "boş olmalı"
                )
            )


# ============================================================
# CATEGORY DISTRIBUTION
# ============================================================

finance_categories = Counter(
    record["urun_kategorisi"]
    for record in finance
)


campaign_types = Counter(
    record["kampanya_turu"]
    for record in campaigns
)


EXPECTED_FINANCE_CATEGORIES = {
    "Konut-Gayrimenkul Finansmanı": 5,
    "Taşıt Finansmanı": 3,
    "İhtiyaç Finansmanı": 8,
    (
        "Sürdürülebilirlik Temalı "
        "Bireysel Ürünler"
    ): 4,
}


if (
    dict(finance_categories)
    != EXPECTED_FINANCE_CATEGORIES
):

    errors.append(
        (
            "Finansman kategori "
            "dağılımı yanlış: "
            f"{dict(finance_categories)}"
        )
    )


EXPECTED_CAMPAIGN_TYPES = {
    "Taksit": 72,
    "Bankkart Lira": 9,
    "İndirim": 4,
    "Avantaj": 2,
}


if (
    dict(campaign_types)
    != EXPECTED_CAMPAIGN_TYPES
):

    errors.append(
        (
            "Kampanya tür "
            "dağılımı yanlış: "
            f"{dict(campaign_types)}"
        )
    )


# ============================================================
# SAVE
#
# Wrapper YOK.
# Direkt 107 kayıtlık JSON listesi.
# ============================================================

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        records,
        f,
        ensure_ascii=False,
        indent=4
    )


# ============================================================
# FINAL AUDIT
# ============================================================

print()
print("=" * 118)
print("ZİRAAT KATILIM - FINAL AUDIT")
print("=" * 118)

print(
    "Toplam kayıt      :",
    len(records)
)

print(
    "Finansman         :",
    type_counts.get(
        "finansman",
        0
    )
)

print(
    "Kampanya          :",
    type_counts.get(
        "kampanya",
        0
    )
)

print(
    "Duplicate URL     :",
    duplicate_url
)

print(
    "Duplicate kayıt   :",
    duplicate_name
)

print(
    "Validation error  :",
    len(errors)
)

print(
    "Final JSON        :",
    OUTPUT_FILE
)


print()
print("=" * 118)
print("FİNANSMAN KATEGORİ DAĞILIMI")
print("=" * 118)

for key, value in (
    finance_categories.items()
):

    print(
        f"{key}: {value}"
    )


print()
print("=" * 118)
print("KAMPANYA TÜR DAĞILIMI")
print("=" * 118)

for key in [
    "Taksit",
    "Bankkart Lira",
    "İndirim",
    "Avantaj",
]:

    print(
        f"{key:<15}: "
        f"{campaign_types.get(key, 0)}"
    )


# ============================================================
# ERRORS
# ============================================================

if errors:

    print()
    print("=" * 118)
    print("HATALAR")
    print("=" * 118)

    for error in errors:

        print(
            "-",
            error
        )


print()
print("=" * 118)


if not errors:

    print(
        "SONUÇ: ZİRAAT KATILIM "
        "FINAL 107/107 "
        "TAMAMEN BAŞARILI ✅"
    )

else:

    print(
        "SONUÇ: ZİRAAT KATILIM "
        "FINAL KONTROL GEREKİYOR ❌"
    )


print("=" * 118)


files.download(
    OUTPUT_FILE
)

ZİRAAT KATILIM - FINAL MERGER V1
Finansman: 20
Kampanya : 87

ZİRAAT KATILIM - FINAL AUDIT
Toplam kayıt      : 107
Finansman         : 20
Kampanya          : 87
Duplicate URL     : 0
Duplicate kayıt   : 0
Validation error  : 0
Final JSON        : /content/ziraat_katilim_final.json

FİNANSMAN KATEGORİ DAĞILIMI
Konut-Gayrimenkul Finansmanı: 5
Taşıt Finansmanı: 3
İhtiyaç Finansmanı: 8
Sürdürülebilirlik Temalı Bireysel Ürünler: 4

KAMPANYA TÜR DAĞILIMI
Taksit         : 72
Bankkart Lira  : 9
İndirim        : 4
Avantaj        : 2

SONUÇ: ZİRAAT KATILIM FINAL 107/107 TAMAMEN BAŞARILI ✅


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>